In [1]:
!pip -q uninstall -y transformers accelerate datasets tokenizers huggingface_hub safetensors sentencepiece pyarrow
!pip -q install -U "numpy<2"
!pip -q install -U "huggingface_hub>=0.26.0" "tokenizers>=0.20.0" "transformers>=4.54.0" "accelerate>=1.0.0" "datasets>=2.20.0" safetensors sentencepiece

In [1]:
import numpy as np, pandas as pd
import pyarrow as pa, datasets
import transformers, accelerate, torch

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("pyarrow:", pa.__version__)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

/Users/jeongseong-gyeong/Documents/aiary/aiary/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jeongseong-gyeong/Documents/aiary/aiary/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


numpy: 1.26.4
pandas: 2.3.3
pyarrow: 21.0.0
datasets: 4.5.0
transformers: 4.57.6
accelerate: 1.10.1
torch: 2.8.0
cuda: False


In [ ]:
# from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

In [ ]:
import os
print(os.environ.get("OPENAI_API_KEY"))

In [4]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [4]:
# ============================================================
# [CELL 1] Aiary v3 Hybrid Pipeline - 공통 준비/유틸 (Drive + Pool + one_lines 생성)
# ------------------------------------------------------------
# 이 셀에서 하는 일
# - Google Drive(Shared Drive) 마운트, 실행 폴더 생성
# - Situation Pool / State Surface Dict 로드
# - one_lines(사진 캡션) 생성: 템플릿/동의어/조사(종성)/노이즈/필터/trace
# - 하루 샘플링(1~10장, 보통 3~5장) + 다양성 필터
#
# 다음 셀에서 이 셀의 함수/변수들을 그대로 사용
# - (CELL 2) Teacher(GPT)로 anchors 생성 + KoBART round1 학습
# - (CELL 3) Student 생성 + 하이브리드 교정(EXAONE->GPT) + 학습데이터는 "교정 중심"으로 구성 + round2 학습
# ============================================================

import os, re, json, time, random, glob
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional
from datetime import datetime

import pandas as pd

# from google.colab import drive
# drive.mount("/content/drive", force_remount=True)

# ----------------------------
# 사용자 설정(여기만 수정)
# ----------------------------
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

SHARED_ROOT = "."

SITUATION_CSV_PATH  = f"data/situation_pool_v3_semantic_5000.csv"
STATE_DICT_CSV_PATH = f"data/state_surface_dictionary_v1.csv"

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"{SHARED_ROOT}/generated/aiary_v3_hybrid_{RUN_TS}"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# 전역 설정
# ----------------------------
SEED = 777
random.seed(SEED)

PHOTO_COUNT_DIST  = {1:0.08,2:0.12,3:0.24,4:0.22,5:0.18,6:0.08,7:0.04,8:0.02,9:0.01,10:0.01}
DAILY_LENGTH_DIST = {"SHORT":0.25, "MEDIUM":0.55, "LONG":0.20}
PERSONA_RATIO = 0.20

TIME_ORDER = ["아침","오전","점심","오후","저녁","밤"]
TIME_IDX = {t:i for i,t in enumerate(TIME_ORDER)}
TURNING_EVENTS = {"minor_conflict","refusal","accident","achievement","separation","reunion"}

ONE_LINE_MIN_CHARS = 12
ONE_LINE_MAX_CHARS = 60

BANNED_PHRASES = [
    "영원", "미래", "추억", "고요", "에너지가 느껴", "위로", "괜찮아", "힘내",
    "항상", "언제나", "평생"
]

NOISE_PROB = {
    "drop_one": 0.35,
    "shuffle": 0.30,
    "omit_object": 0.50,
    "omit_time": 0.60,
    "insert_adverb": 0.25,
    "tighten": 0.20
}
ADVERBS = ["조용히", "천천히", "잠깐", "한참", "살짝", "계속"]

SHARD_SIZE = 500
FLUSH_EVERY = 20
MAX_ATTEMPTS = 800000

# ----------------------------
# 유틸
# ----------------------------
def normalize_space(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()

def contains_banned(s: str) -> bool:
    s = str(s)
    return any(p in s for p in BANNED_PHRASES)

def weighted_choice(dist: Dict[Any, float]) -> Any:
    keys = list(dist.keys())
    weights = list(dist.values())
    return random.choices(keys, weights=weights, k=1)[0]

def clamp_text_len(s: str, max_chars: int) -> str:
    s = s.strip()
    return s[:max_chars].rstrip() if len(s) > max_chars else s

def sentence_count_heuristic(text: str) -> int:
    parts = re.split(r"[.!?]\s*|[。！？]\s*|\n", text)
    parts = [p.strip() for p in parts if p.strip()]
    return len(parts)

def safe_join_one_lines(one_lines: List[str]) -> str:
    return "\n".join(one_lines)

# ----------------------------
# 조사(종성) 처리
# ----------------------------
def has_final_consonant(korean_char: str) -> bool:
    code = ord(korean_char)
    if code < 0xAC00 or code > 0xD7A3:
        return False
    return (code - 0xAC00) % 28 != 0

def choose_particle(word: str, pair: Tuple[str, str]) -> str:
    word = word.strip()
    if not word:
        return pair[1]
    last = word[-1]
    return pair[0] if has_final_consonant(last) else pair[1]

def attach_particle(word: str, pair: Tuple[str, str]) -> str:
    return word + choose_particle(word, pair)

# ----------------------------
# 표면형 변형(장소/사물/활동)
# ----------------------------
PLACE_SYNONYMS = {
    "집": ["집", "집 안", "집에서"],
    "거실": ["거실", "거실 바닥", "거실 한쪽"],
    "주방": ["주방", "식탁 옆", "주방 쪽"],
    "침실": ["침실", "침대 위", "방 안"],
    "욕실": ["욕실", "욕실 앞", "욕조 옆"],
    "현관": ["현관", "문 앞", "신발장 앞"],
    "놀이터": ["놀이터", "동네 놀이터", "그네 옆"],
    "공원": ["공원", "잔디밭", "산책로"],
    "어린이집": ["어린이집", "교실", "등원길"],
    "키즈카페": ["키즈카페", "볼풀장", "미끄럼틀 앞"],
    "마트": ["마트", "매대 앞", "카트 옆"],
    "카페": ["카페", "테이블 옆", "창가 자리"],
    "병원": ["병원", "대기실", "진료실 앞"],
    "도서관": ["도서관", "책장 앞", "읽는 자리"],
    "집 앞": ["집 앞", "현관 밖", "집 앞 길"],
    "조부모 집": ["할머니 댁", "할아버지 댁", "조부모님 댁"]
}

OBJECT_SYNONYMS = {
    "젖병": ["젖병", "우유병"],
    "빨대컵": ["빨대컵", "물컵", "컵"],
    "숟가락": ["숟가락", "스푼", "작은 스푼"],
    "이불": ["이불", "담요"],
    "인형": ["인형", "애착인형"],
    "가방": ["가방", "등원가방"],
    "블록": ["블록", "블럭"],
    "크레파스": ["크레파스", "색연필", "그림도구"],
    "자전거": ["자전거", "작은 자전거"]
}

ACTIVITY_PATTERNS = [
    "{activity} 중이다",
    "{activity}하고 있다",
    "{activity}하는 모습이다",
    "{activity}에 집중한다",
    "{activity}을(를) 해본다",
    "{activity}하다가 멈칫한다",
]

def render_place(place: str) -> str:
    place = normalize_space(place)
    return random.choice(PLACE_SYNONYMS[place]) if place in PLACE_SYNONYMS else place

def render_object(obj: str) -> str:
    obj = normalize_space(obj)
    return random.choice(OBJECT_SYNONYMS[obj]) if obj in OBJECT_SYNONYMS else obj

def render_activity(activity: str) -> str:
    activity = normalize_space(activity)
    return random.choice(ACTIVITY_PATTERNS).format(activity=activity)

# ----------------------------
# 상태 표면표현 로드
# ----------------------------
def load_state_surface_dict(path: str) -> Dict[str, List[str]]:
    dfd = pd.read_csv(path)
    out = {}
    for _, r in dfd.iterrows():
        key = normalize_space(r["상태키"])
        variants = str(r["표면표현들"]).split("|")
        variants = [normalize_space(v) for v in variants if normalize_space(v)]
        out[key] = variants if variants else [key]
    return out

def render_state_phrase(state_key: str, state_surface: Dict[str, List[str]]) -> str:
    return random.choice(state_surface.get(state_key, [state_key]))

# ----------------------------
# Situation Pool 로드
# ----------------------------
@dataclass
class Situation:
    situation_id: str
    age_group: str
    time_slot: str
    place: str
    activity: str
    target: str
    obj: str
    state_type: str
    state_key: str
    event_type: str
    raw_state: str

def load_situation_pool(path: str) -> List[Situation]:
    df = pd.read_csv(path)
    required = ["situation_id","연령대","시간대","장소","활동","대상","사물","상태유형","상태키","event_type","상태"]
    for c in required:
        if c not in df.columns:
            raise ValueError(f"CSV missing column: {c}")
    pool = []
    for _, r in df.iterrows():
        pool.append(Situation(
            situation_id=normalize_space(r["situation_id"]),
            age_group=normalize_space(r["연령대"]),
            time_slot=normalize_space(r["시간대"]),
            place=normalize_space(r["장소"]),
            activity=normalize_space(r["활동"]),
            target=normalize_space(r["대상"]),
            obj=normalize_space(r["사물"]),
            state_type=normalize_space(r["상태유형"]),
            state_key=normalize_space(r["상태키"]),
            event_type=normalize_space(r["event_type"]),
            raw_state=normalize_space(r["상태"]),
        ))
    return pool

def build_index_by_age(pool: List[Situation]) -> Dict[str, List[Situation]]:
    by_age = {}
    for s in pool:
        by_age.setdefault(s.age_group, []).append(s)
    return by_age

# ----------------------------
# one_line 템플릿(어순 변형 포함)
# ----------------------------
TEMPLATE_GROUPS = {
    "A_place_act": [
        "{place}에서 {act} {target_subj} {state}.",
        "{time}{place}에서 {act} {target_subj} {state}.",
        "{place}에서 {target_subj} {act} {state}.",
    ],
    "B_target_obj": [
        "{obj_obj} 만지며 {place}에서 {act} {target_subj} {state}.",
        "{obj_obj} 들고 {act} {target_subj} {state}.",
        "{obj_obj} 챙긴 {target_subj} {place}에서 {act} {state}.",
    ],
    "C_trace_first": [
        "{state} {target_subj} {place}에서 {act}.",
        "{state} 채로 {act} {target_subj}.",
        "{state} {place}에서 {act} {target_subj}.",
    ],
    "D_caption_short": [
        "{place}, {act} {target_subj}. {state}.",
        "{act} {target_subj}, {state}.",
        "{place}에서 {act}… {state}.",
    ],
    "E_contrast": [
        "막 {act}하더니, {state} {target_subj}.",
        "{act}하다가 {state} {target_subj} 멈칫한다.",
    ],
    "F_order_variant": [
        "{act} {target_subj} {place}에서 {state}.",
        "{state} {target_subj} {act} {place}에서.",
        "{target_subj} {act}하다가 {state} (장소: {place}).",
    ],
}
GROUP_WEIGHTS = {
    "A_place_act": 0.26,
    "B_target_obj": 0.20,
    "C_trace_first": 0.18,
    "D_caption_short": 0.12,
    "E_contrast": 0.10,
    "F_order_variant": 0.14,
}

def sample_template() -> str:
    group = weighted_choice(GROUP_WEIGHTS)
    return random.choice(TEMPLATE_GROUPS[group])

def maybe_time_prefix(time_slot: str) -> str:
    if random.random() < NOISE_PROB["omit_time"]:
        return ""
    variants = {
        "아침": ["아침에 ", "아침부터 "],
        "오전": ["오전에 ", "오전 내내 "],
        "점심": ["점심때 ", "점심 무렵 "],
        "오후": ["오후에 ", "오후 내내 "],
        "저녁": ["저녁에 ", "저녁 무렵 "],
        "밤": ["밤에 ", "밤늦게 "]
    }
    return random.choice(variants.get(time_slot, [time_slot + "에 "]))

def tighten_variant(text: str) -> str:
    t = text
    t = t.replace("하는 모습이다", "하는 중이다")
    t = t.replace("하고 있다", "하는 중이다")
    t = t.replace("…", "")
    return normalize_space(t)

def render_one_line_with_trace(s: Situation, state_surface: Dict[str, List[str]]) -> Tuple[str, Dict[str, Any]]:
    place_surface = render_place(s.place)
    obj_surface = render_object(s.obj)
    act_surface = render_activity(s.activity)
    state_surface_phrase = render_state_phrase(s.state_key, state_surface)
    time_prefix = maybe_time_prefix(s.time_slot)

    object_omitted = False
    if random.random() < NOISE_PROB["omit_object"]:
        obj_surface = random.choice(["손", "옷", "장난감", "가방", "컵", "인형", "책"])
        object_omitted = True

    target_subj = attach_particle(s.target, ("이", "가"))
    obj_obj = attach_particle(obj_surface, ("을", "를"))

    text = normalize_space(sample_template().format(
        time=time_prefix,
        place=place_surface,
        act=act_surface,
        target_subj=target_subj,
        obj_obj=obj_obj,
        state=state_surface_phrase
    ))

    if random.random() < NOISE_PROB["insert_adverb"]:
        adv = random.choice(ADVERBS)
        if "에서" in text and random.random() < 0.6:
            text = text.replace("에서", f"에서 {adv}", 1)
        else:
            text = f"{adv} {text}"
        text = normalize_space(text)

    if random.random() < NOISE_PROB["tighten"]:
        text = tighten_variant(text)

    text = clamp_text_len(text, ONE_LINE_MAX_CHARS)

    if contains_banned(text) or len(text) < ONE_LINE_MIN_CHARS:
        state_surface_phrase = render_state_phrase(s.state_key, state_surface)
        text = normalize_space(sample_template().format(
            time=time_prefix,
            place=place_surface,
            act=act_surface,
            target_subj=target_subj,
            obj_obj=obj_obj,
            state=state_surface_phrase
        ))
        text = clamp_text_len(text, ONE_LINE_MAX_CHARS)

    trace = {
        "situation_id": s.situation_id,
        "time_slot": s.time_slot,
        "time_prefix_used": bool(time_prefix),
        "place_raw": s.place,
        "place_surface": place_surface,
        "activity_raw": s.activity,
        "activity_surface": act_surface,
        "target_raw": s.target,
        "target_surface_subject": target_subj,
        "object_raw": s.obj,
        "object_surface": obj_surface,
        "object_omitted": object_omitted,
        "state_type": s.state_type,
        "state_key": s.state_key,
        "state_surface": state_surface_phrase,
        "event_type": s.event_type,
    }
    return text, trace

# ----------------------------
# Day sampling + 다양성 필터
# ----------------------------
def sample_day_situations(by_age: Dict[str, List[Situation]]) -> List[Situation]:
    k = weighted_choice(PHOTO_COUNT_DIST)
    age = random.choice(list(by_age.keys()))
    candidates = by_age[age]

    picked = []
    used = set()
    tries = 0
    while len(picked) < k and tries < k * 200:
        s = random.choice(candidates)
        if s.situation_id in used:
            tries += 1
            continue
        used.add(s.situation_id)
        picked.append(s)
        tries += 1

    if picked and not any(p.event_type in TURNING_EVENTS for p in picked) and random.random() < 0.35:
        turning_candidates = [x for x in candidates if x.event_type in TURNING_EVENTS]
        if turning_candidates:
            picked[random.randrange(len(picked))] = random.choice(turning_candidates)

    if random.random() < 0.80:
        picked.sort(key=lambda x: TIME_IDX.get(x.time_slot, 999))
    else:
        random.shuffle(picked)

    return picked

def validate_day_diversity(day_situations: List[Situation]) -> bool:
    if not day_situations:
        return False
    acts = [s.activity for s in day_situations]
    places = [s.place for s in day_situations]
    states = [s.state_key for s in day_situations]
    if len(day_situations) >= 4 and len(set(acts)) <= 1:
        return False
    if len(day_situations) >= 5 and len(set(places)) <= 1 and len(set(acts)) <= 2:
        return False
    if len(day_situations) >= 4 and len(set(states)) <= 1:
        return False
    return True

def perturb_one_lines(one_lines: List[str], traces: List[Dict[str, Any]]) -> Tuple[List[str], List[Dict[str, Any]]]:
    idxs = list(range(len(one_lines)))
    if len(idxs) >= 3 and random.random() < NOISE_PROB["drop_one"]:
        drop_i = random.choice(idxs)
        idxs = [i for i in idxs if i != drop_i]
    if len(idxs) >= 3 and random.random() < NOISE_PROB["shuffle"]:
        random.shuffle(idxs)
    return [one_lines[i] for i in idxs], [traces[i] for i in idxs]

def validate_one_lines(one_lines: List[str]) -> bool:
    if not (1 <= len(one_lines) <= 10):
        return False
    if len(set(one_lines)) != len(one_lines):
        return False
    for x in one_lines:
        if not x or "\n" in x:
            return False
        if len(x) < ONE_LINE_MIN_CHARS:
            return False
        if contains_banned(x):
            return False
    joined = " ".join(one_lines)
    if joined.count("에서") > max(5, len(one_lines) + 2) and len(one_lines) <= 5:
        return False
    return True

# ----------------------------
# 저장 유틸(JSONL shard)
# ----------------------------
def shard_path(prefix: str, shard_idx: int) -> str:
    return os.path.join(OUT_DIR, f"{prefix}_{shard_idx:04d}.jsonl")

def write_jsonl_shards(prefix: str, records: List[Dict[str, Any]]):
    os.makedirs(OUT_DIR, exist_ok=True)
    shard_idx = 0
    for i in range(0, len(records), SHARD_SIZE):
        path = shard_path(prefix, shard_idx)
        with open(path, "w", encoding="utf-8") as f:
            for r in records[i:i+SHARD_SIZE]:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("[SAVED]", path, "n=", min(SHARD_SIZE, len(records) - i))
        shard_idx += 1

def load_jsonl_glob(pattern: str) -> List[Dict[str, Any]]:
    paths = sorted(glob.glob(pattern))
    out = []
    for p in paths:
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    out.append(json.loads(line))
    return out

def save_run_config(extra: Optional[Dict[str, Any]]=None):
    cfg = {
        "seed": SEED,
        "photo_count_dist": PHOTO_COUNT_DIST,
        "daily_length_dist": DAILY_LENGTH_DIST,
        "persona_ratio": PERSONA_RATIO,
        "noise_prob": NOISE_PROB,
        "paths": {
            "situation_csv": SITUATION_CSV_PATH,
            "state_dict_csv": STATE_DICT_CSV_PATH,
            "out_dir": OUT_DIR
        }
    }
    if extra:
        cfg.update(extra)
    with open(os.path.join(OUT_DIR, "run_config.json"), "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)

# ----------------------------
# 풀 로드
# ----------------------------
pool = load_situation_pool(SITUATION_CSV_PATH)
state_surface = load_state_surface_dict(STATE_DICT_CSV_PATH)
by_age = build_index_by_age(pool)

save_run_config()

print("[READY]")
print("OUT_DIR:", OUT_DIR)
print("pool size:", len(pool))
print("state keys:", len(state_surface))

[READY]
OUT_DIR: ./generated/aiary_v3_hybrid_20260304_115749
pool size: 5000
state keys: 24


In [ ]:
# anchor 미생성시, 실행
# ============================================================
# [CELL 2] Round 0~1: Teacher(GPT) anchors 생성 + KoBART round1 학습
# ------------------------------------------------------------
# 이 셀에서 하는 일
# - Teacher(gpt-4.1-mini)로 anchors 생성: (one_lines 여러 줄 -> 하루일기)
# - anchors로 KoBART 1차 fine-tune (student_round1 저장)
#
# 실전 팁
# - 처음엔 ANCHOR_TARGET=200으로 테스트 후 2000으로 확장 추천
# ============================================================


from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
TEACHER_MODEL = "gpt-4.1-mini"

ANCHOR_TARGET = 2000  # 테스트는 200 권장
MAX_OUTPUT_TOKENS_CAP = 900

def call_teacher(system: str, user: str, temperature: float, max_output_tokens: int) -> str:
    backoff = 1.5
    last_err = None
    for attempt in range(1, 9):
        try:
            resp = client.responses.create(
                model=TEACHER_MODEL,
                input=[{"role":"system","content":system},{"role":"user","content":user}],
                temperature=temperature,
                max_output_tokens=min(max_output_tokens, MAX_OUTPUT_TOKENS_CAP),
            )
            text = getattr(resp, "output_text", None)
            return str(text if text is not None else resp).strip()
        except Exception as e:
            last_err = e
            time.sleep(min(20, backoff ** attempt) + random.random())
    raise RuntimeError(f"Teacher call failed after retries: {last_err}")

def build_daily_diary_prompt(one_lines: List[str], length_tag: str, persona_mode: bool=False) -> Tuple[str, str, float, int]:
    if length_tag == "SHORT":
        guideline, max_tokens, temp = "3~5문장", 320, 0.55
    elif length_tag == "MEDIUM":
        guideline, max_tokens, temp = "6~8문장", 520, 0.65
    else:
        guideline, max_tokens, temp = "9~11문장", 720, 0.70

    system = (
        "너는 0~6세 아이를 키우는 보호자의 하루 기록을 쓴다. "
        "입력으로 주어진 한 줄 기록(사진 캡션들)을 모두 반영하여 하루일기를 작성한다. "
        "금지: 영원/미래/추억/위로/과한 감탄/시적 분위기. "
        "절대 추가하지 말 것: 입력에 없는 새로운 장소/사물/행동/사건. "
        "허용: 입력 장면에서 자연스럽게 드러나는 보호자의 감정/해석을 1~2문장 정도만 덧붙일 수 있다 "
        "(예: 마음이 놓였다, 살짝 웃음이 났다, 조금 안도했다). "
        "각 한 줄의 구체 요소(행동/표정/상태)를 빠짐없이 녹여라."
    )
    if persona_mode:
        system += (
            "추가 옵션: 전체 문장 중 1문장만 보호자의 따뜻한 시선을 담되, "
            "추상어(사랑, 감동, 행복) 대신 관측 기반 감정 표현으로만 쓴다."
        )

    user = (
        f"아래 한 줄 기록들을 모두 반영해서 하루일기를 써라.\n"
        f"- 문장 수: {guideline}\n"
        f"- 조건: 각 한 줄 내용을 최소 1회 이상 구체적으로 반영\n\n"
        "[한 줄 기록]\n" + "\n".join([f"- {x}" for x in one_lines])
    )
    return system, user, temp, max_tokens

def validate_daily_diary(text: str, length_tag: str) -> bool:
    if not text or contains_banned(text):
        return False
    sc = sentence_count_heuristic(text)
    if length_tag == "SHORT":
        return 2 <= sc <= 7
    if length_tag == "MEDIUM":
        return 5 <= sc <= 11
    return 8 <= sc <= 15

def generate_anchor_records(target_n: int) -> List[Dict[str, Any]]:
    created = 0
    attempts = 0
    records = []
    while created < target_n and attempts < MAX_ATTEMPTS:
        attempts += 1

        day_situations = sample_day_situations(by_age)
        if not validate_day_diversity(day_situations):
            continue

        one_lines, one_lines_trace = [], []
        for s in day_situations:
            ol, tr = render_one_line_with_trace(s, state_surface)
            one_lines.append(ol)
            one_lines_trace.append(tr)

        one_lines, one_lines_trace = perturb_one_lines(one_lines, one_lines_trace)
        if not validate_one_lines(one_lines):
            continue

        length_tag = weighted_choice(DAILY_LENGTH_DIST)
        persona_mode = (random.random() < PERSONA_RATIO)

        system, user, temp, max_tokens = build_daily_diary_prompt(one_lines, length_tag, persona_mode)
        daily_diary = normalize_space(call_teacher(system, user, temp, max_tokens))

        if not validate_daily_diary(daily_diary, length_tag):
            continue

        age_group = day_situations[0].age_group if day_situations else ""
        record = {
            "id": f"anchor_{created:06d}",
            "stage": "anchor_teacher",
            "meta": {
                "age_group": age_group,
                "k_photos": len(one_lines),
                "length_tag": length_tag,
                "teacher_model": TEACHER_MODEL,
                "persona_mode": persona_mode,
                "seed": SEED,
            },
            "input": {
                "one_lines": one_lines,
                "joined": safe_join_one_lines(one_lines),
                "one_lines_trace": one_lines_trace
            },
            "output": {"daily_diary": daily_diary},
            "trace": {"attempt_idx": attempts}
        }
        records.append(record)
        created += 1

        if created % 50 == 0:
            print(f"[ANCHOR] created={created}/{target_n} attempts={attempts}")

    if created < target_n:
        print("[WARN] anchor generation ended early:", created, "created")
    return records

anchors = generate_anchor_records(ANCHOR_TARGET)
write_jsonl_shards("anchors", anchors)

# ----------------------------
# KoBART round1 학습
# ----------------------------
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

STUDENT_BASE_MODEL = "gogamza/kobart-base-v2"
STUDENT_SAVE_DIR = f"{OUT_DIR}/models/kobart_student_round1"
os.makedirs(STUDENT_SAVE_DIR, exist_ok=True)

def to_hf_dataset(records: List[Dict[str, Any]]) -> Dataset:
    rows = [{"src": r["input"]["joined"], "tgt": r["output"]["daily_diary"]} for r in records]
    return Dataset.from_list(rows)

ds = to_hf_dataset(anchors)
split = ds.train_test_split(test_size=0.02, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

tok = AutoTokenizer.from_pretrained(STUDENT_BASE_MODEL, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(STUDENT_BASE_MODEL)

MAX_SRC_LEN = 512
MAX_TGT_LEN = 512

def preprocess(batch):
    model_inputs = tok(batch["src"], max_length=MAX_SRC_LEN, truncation=True)
    labels = tok(text_target=batch["tgt"], max_length=MAX_TGT_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tok, model=model)

training_args = TrainingArguments(
    output_dir=f"{OUT_DIR}/runs/kobart_round1",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    num_train_epochs=2,
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    tokenizer=tok,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(STUDENT_SAVE_DIR)
tok.save_pretrained(STUDENT_SAVE_DIR)

print("[DONE] anchors saved: anchors_****.jsonl")
print("[DONE] student saved:", STUDENT_SAVE_DIR)
print("[OUT_DIR]", OUT_DIR)

[ANCHOR] created=50/2000 attempts=50
[ANCHOR] created=100/2000 attempts=102
[ANCHOR] created=150/2000 attempts=152
[ANCHOR] created=200/2000 attempts=202
[ANCHOR] created=250/2000 attempts=252
[ANCHOR] created=300/2000 attempts=303
[ANCHOR] created=350/2000 attempts=354
[ANCHOR] created=400/2000 attempts=404
[ANCHOR] created=450/2000 attempts=454
[ANCHOR] created=500/2000 attempts=505
[ANCHOR] created=550/2000 attempts=557
[ANCHOR] created=600/2000 attempts=607
[ANCHOR] created=650/2000 attempts=658
[ANCHOR] created=700/2000 attempts=708
[ANCHOR] created=750/2000 attempts=758
[ANCHOR] created=800/2000 attempts=808
[ANCHOR] created=850/2000 attempts=858
[ANCHOR] created=900/2000 attempts=908
[ANCHOR] created=950/2000 attempts=958
[ANCHOR] created=1000/2000 attempts=1008
[ANCHOR] created=1050/2000 attempts=1058
[ANCHOR] created=1100/2000 attempts=1108
[ANCHOR] created=1150/2000 attempts=1159
[ANCHOR] created=1200/2000 attempts=1209
[ANCHOR] created=1250/2000 attempts=1259
[ANCHOR] create

config.json: 0.00B [00:00, ?B/s]

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Map:   0%|          | 0/1960 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
import accelerate, transformers
print("accelerate:", accelerate.__version__)
print("transformers:", transformers.__version__)

accelerate: 0.34.2
transformers: 4.54.1


In [ ]:
# ============================================================
# [CELL 2-REPAIR] Round 0~1: (이미 완료된 anchors는 "로드") + (미완료된 KoBART round1 학습만 수행)
# ------------------------------------------------------------
# 로그상 완료된 것:
# - anchors 2000개 생성 완료
# - anchors_0000~0003.jsonl로 저장 완료 (각 500개)
#
# 로그상 미완료된 것:
# - KoBART round1 학습 단계에서 Trainer(tokenizer=...) 인자 에러로 중단
#
# 이 셀은:
# - anchors 파일이 있으면 "생성 스킵"하고 로드
# - KoBART round1 학습만 실행 (Trainer에서 tokenizer 인자 제거)
# ============================================================


import os, glob, random, re
from typing import List, Dict, Any

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

# ----------------------------
# 경로/상수
# ----------------------------
OUT_DIR = "/content/drive/MyDrive/aiary_공유문서함/generated/aiary_v3_hybrid_20260220_014739/"
ANCHOR_GLOB = os.path.join(OUT_DIR, "anchors_*.jsonl")  # write_jsonl_shards("anchors", ...) 규칙
STUDENT_BASE_MODEL = "gogamza/kobart-base-v2"
STUDENT_SAVE_DIR = f"{OUT_DIR}/models/kobart_student_round1"
os.makedirs(STUDENT_SAVE_DIR, exist_ok=True)

SEED = globals().get("SEED", 777)

# ----------------------------
# 유틸(이 셀 단독 실행 대비)
# - 너 노트북에 이미 같은 함수가 있으면 이 정의는 무시해도 됨
# ----------------------------
def _normalize_space(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def _read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(__import__("json").loads(line))
    return rows

def load_jsonl_glob(pattern: str) -> List[Dict[str, Any]]:
    paths = sorted(glob.glob(pattern))
    if not paths:
        raise FileNotFoundError(f"[ERROR] anchors not found: {pattern}")
    all_rows = []
    for p in paths:
        all_rows.extend(_read_jsonl(p))
    return all_rows

def to_hf_dataset(records: List[Dict[str, Any]]) -> Dataset:
    rows = [{"src": r["input"]["joined"], "tgt": r["output"]["daily_diary"]} for r in records]
    return Dataset.from_list(rows)

# ----------------------------
# 1) ✅ 완료된 작업: anchors 로드 (생성 스킵)
# ----------------------------
anchor_files = sorted(glob.glob(ANCHOR_GLOB))
print("[CHECK] anchor shards:", len(anchor_files))
for p in anchor_files[:8]:
    print(" -", p)

anchors = load_jsonl_glob(ANCHOR_GLOB)
print("[LOADED] anchors:", len(anchors))

# 안전 체크(대략 2000개 기대)
if len(anchors) < 100:
    raise RuntimeError(f"[ERROR] anchors too small: {len(anchors)}. pattern={ANCHOR_GLOB}")

# ----------------------------
# 2) ❌ 미완료 작업: KoBART round1 학습만 수행
# ----------------------------
ds = to_hf_dataset(anchors)
split = ds.train_test_split(test_size=0.02, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

tok = AutoTokenizer.from_pretrained(STUDENT_BASE_MODEL, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(STUDENT_BASE_MODEL)

MAX_SRC_LEN = 512
MAX_TGT_LEN = 512

def preprocess(batch):
    model_inputs = tok(batch["src"], max_length=MAX_SRC_LEN, truncation=True)
    labels = tok(text_target=batch["tgt"], max_length=MAX_TGT_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tok, model=model)

training_args = TrainingArguments(
    output_dir=f"{OUT_DIR}/runs/kobart_round1",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    num_train_epochs=2,
    warmup_ratio=0.05,          # 경고 뜨면 무시 가능(학습은 됨)
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

# ✅ 핵심 수정: tokenizer 인자 제거 (너 환경에서 TypeError 방지)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(STUDENT_SAVE_DIR)
tok.save_pretrained(STUDENT_SAVE_DIR)

print("[DONE] anchors were already saved:", ANCHOR_GLOB)
print("[DONE] student saved:", STUDENT_SAVE_DIR)
print("[OUT_DIR]", OUT_DIR)

[CHECK] anchor shards: 4
 - /content/drive/MyDrive/aiary_공유문서함/generated/aiary_v3_hybrid_20260220_014739/anchors_0000.jsonl
 - /content/drive/MyDrive/aiary_공유문서함/generated/aiary_v3_hybrid_20260220_014739/anchors_0001.jsonl
 - /content/drive/MyDrive/aiary_공유문서함/generated/aiary_v3_hybrid_20260220_014739/anchors_0002.jsonl
 - /content/drive/MyDrive/aiary_공유문서함/generated/aiary_v3_hybrid_20260220_014739/anchors_0003.jsonl
[LOADED] anchors: 2000


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Map:   0%|          | 0/1960 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
200,1.567000,1.478103


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3852: UserWarning: Moving the following attributes in the config to the generation config: {'forced_eos_token_id': 1}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


[DONE] anchors were already saved: /content/drive/MyDrive/aiary_공유문서함/generated/aiary_v3_hybrid_20260220_014739/anchors_*.jsonl
[DONE] student saved: /content/drive/MyDrive/aiary_공유문서함/generated/aiary_v3_hybrid_20260220_014739//models/kobart_student_round1
[OUT_DIR] /content/drive/MyDrive/aiary_공유문서함/generated/aiary_v3_hybrid_20260220_014739/


In [7]:
# ============================================================
# [CELL 3] Round 2+ 운영(교정 중심 학습셋)
# Student 생성 + (지표 기반) 하위/최하위 선별 + EXAONE 로컬 교정 + GPT 최하위 교정 + 다음 student 학습
# ------------------------------------------------------------
# 변경점(직관 지표 추가)
# - coverage: 한 줄 기록(각 줄) 반영률(0~1)
# - novelty: 입력에 없던 단어 비율(0~1)  -> 추가 사실(환각) 위험
# - repetition: 반복/접속사 과다(0~1 근사)
# - 최종 split: score + (novelty/coverage/repetition) 기반으로 섞어서 타깃 선정
#
# EXAONE 동작 방식 변경점
# - chat_template 기반 프롬프트 생성 + 생성분만 디코딩(prompt 섞임 제거)
# ============================================================

!pip -q install -U openai "transformers>=4.54.0" accelerate datasets sentencepiece

import os, time, random, re, math
from typing import List, Dict, Any, Tuple

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

# ----------------------------
# 라운드/비율 설정
# ----------------------------
ROUND_IDX = 2
STUDENT_GEN_TARGET = 10000

EXAONE_REWRITE_RATIO = 0.20
GPT_REWRITE_RATIO = 0.10

KEEP_TRAIN_TOP_RATIO = 0.35  # keep 중 상위만 학습(추천 0.25~0.45)

OUT_DIR = "generated/aiary_v3_hybrid_20260220_014739"

# 현재 student 경로(첫 실행은 round1)
CURRENT_STUDENT_PATH = f"{OUT_DIR}/models/kobart_student_round1"
NEXT_STUDENT_SAVE_DIR = f"{OUT_DIR}/models/kobart_student_round{ROUND_IDX}"
os.makedirs(NEXT_STUDENT_SAVE_DIR, exist_ok=True)

# EXAONE 로컬 모델
EXAONE_MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"

# Teacher(GPT) 준비
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)
TEACHER_MODEL = "gpt-4.1-mini"

def call_teacher(system: str, user: str, temperature: float, max_output_tokens: int) -> str:
    backoff = 1.5
    last_err = None
    for attempt in range(1, 9):
        try:
            resp = client.responses.create(
                model=TEACHER_MODEL,
                input=[{"role":"system","content":system},{"role":"user","content":user}],
                temperature=temperature,
                max_output_tokens=max_output_tokens,
            )
            text = getattr(resp, "output_text", None)
            return str(text if text is not None else resp).strip()
        except Exception as e:
            last_err = e
            time.sleep(min(20, backoff ** attempt) + random.random())
    raise RuntimeError(f"Teacher call failed after retries: {last_err}")

# ============================================================
# (A) 직관 지표: coverage / novelty / repetition
# ============================================================

_STOPWORDS = set(["오늘","아침","오전","점심","오후","저녁","밤","아이","아기","엄마","아빠","보호자"])
_CONJ = ["그리고", "그래서", "하지만", "그러다", "그러고", "또", "또는", "또한"]

def _ko_words(s: str) -> List[str]:
    # 2글자 이상 한글 덩어리 (아주 가벼운 토큰)
    return re.findall(r"[가-힣]{2,}", s)

def _keywords_from_line(line: str, max_k: int = 6) -> List[str]:
    words = [w for w in _ko_words(line) if w not in _STOPWORDS]
    # 너무 흔한 동사/형용사성 단어도 걸릴 수 있지만, 가벼운 휴리스틱으로 충분히 유의미
    # 중복 제거 + 앞에서 max_k
    out = []
    seen = set()
    for w in words:
        if w in seen:
            continue
        seen.add(w)
        out.append(w)
        if len(out) >= max_k:
            break
    return out

def coverage_score(one_lines: List[str], diary: str) -> float:
    """각 줄에서 뽑은 키워드 중 1개 이상 diary에 포함되면 그 줄은 '반영됨'."""
    if not one_lines:
        return 0.0
    hit = 0
    for line in one_lines:
        kws = _keywords_from_line(line, max_k=6)
        if not kws:
            # 키워드가 없으면 line 자체 일부(한글2+)가 있으면 hit 처리
            hit += 1 if any(tok in diary for tok in _ko_words(line)[:3]) else 0
            continue
        if any(k in diary for k in kws):
            hit += 1
    return hit / max(1, len(one_lines))

def novelty_score(one_lines: List[str], diary: str) -> float:
    """입력에 없던 단어 비율(0~1). 높을수록 '추가 사실' 위험."""
    in_words = set()
    for l in one_lines:
        in_words.update(_keywords_from_line(l, max_k=999))
    out_words = [w for w in _ko_words(diary) if w not in _STOPWORDS]
    if not out_words:
        return 0.0
    out_set = set(out_words)
    novel = [w for w in out_set if w not in in_words]
    return len(novel) / max(1, len(out_set))

def repetition_score(diary: str) -> float:
    """
    반복/접속사 과다를 0~1 근사로.
    - 접속사 count
    - 문장 시작 반복
    - 단어 반복 비율(유니크 비율 낮으면 반복 높음)
    """
    if not diary:
        return 1.0

    conj_cnt = sum(diary.count(c) for c in _CONJ)
    # 문장 분리(대충)
    sents = [s.strip() for s in re.split(r"[.!?\n]+", diary) if s.strip()]
    starts = [s[:4] for s in sents if len(s) >= 4]
    start_rep = 0.0
    if starts:
        counts = {}
        for st in starts:
            counts[st] = counts.get(st, 0) + 1
        maxc = max(counts.values())
        # 1이면 반복 없음, 2 이상이면 반복 존재
        start_rep = max(0, (maxc - 1) / max(1, len(starts) - 1))

    words = [w for w in _ko_words(diary) if w not in _STOPWORDS]
    uniq_ratio = (len(set(words)) / max(1, len(words))) if words else 1.0
    word_rep = 1.0 - uniq_ratio  # 낮을수록 좋음

    # 접속사: 0~1로 정규화(대충 0~10 범위 가정)
    conj_norm = min(1.0, conj_cnt / 10.0)

    # 가중합(직관적 비중)
    return float(min(1.0, 0.45 * word_rep + 0.35 * start_rep + 0.20 * conj_norm))

# ============================================================
# (B) quick_score 개선: 기존 + coverage/novelty/repetition 반영
# ============================================================

def quick_score(one_lines: List[str], diary: str, length_tag: str) -> float:
    score = 100.0

    # 1) 금지어
    if contains_banned(diary):
        score -= 30

    # 2) 길이(문장수)
    sc = sentence_count_heuristic(diary)
    if length_tag == "SHORT" and not (2 <= sc <= 7):
        score -= 15
    if length_tag == "MEDIUM" and not (5 <= sc <= 11):
        score -= 15
    if length_tag == "LONG" and not (8 <= sc <= 15):
        score -= 15

    # 3) 커버리지(줄별 반영률)
    cov = coverage_score(one_lines, diary)  # 0~1
    score -= (1.0 - cov) * 35  # 커버리지 부족은 강하게 감점

    # 4) 추가 사실 위험(입력에 없는 단어 비율)
    nov = novelty_score(one_lines, diary)  # 0~1
    score -= nov * 25

    # 5) 반복
    rep = repetition_score(diary)          # 0~1
    score -= rep * 20

    # 6) 너무 짧음
    if len(diary) < 50:
        score -= 10

    # 메타로도 남기고 싶으면 호출부에서 meta에 저장해도 됨
    return float(score)

# ============================================================
# (C) GPT 프롬프트
# ============================================================

def build_gpt_rewrite_prompt(one_lines: List[str], length_tag: str) -> Tuple[str, str, float, int]:
    if length_tag == "SHORT":
        guideline, max_tokens, temp = "3~5문장", 260, 0.5
    elif length_tag == "MEDIUM":
        guideline, max_tokens, temp = "6~8문장", 420, 0.55
    else:
        guideline, max_tokens, temp = "9~11문장", 560, 0.6

    system = (
        "너는 한국어 하루일기 최종 편집장이다. "
        "입력 한 줄 기록을 모두 반영해 자연스럽게 재작성한다. "
        "추가 사실(새 장소/사물/행동/사건) 금지. "
        "문장 반복/접속사 나열을 줄이고, 관측 기반 감정 표현 1~2문장만 허용. "
        "금지어(영원/미래/추억/위로/과한 감탄/시적 분위기) 사용 금지."
    )
    user = (
        f"아래 한 줄 기록을 바탕으로 하루일기를 재작성하라.\n"
        f"- 문장 수: {guideline}\n"
        f"- 조건: 각 한 줄 내용을 최소 1회 이상 구체적으로 반영\n\n"
        "[한 줄 기록]\n" + "\n".join([f"- {x}" for x in one_lines])
    )
    return system, user, temp, max_tokens

# ============================================================
# (D) EXAONE 로컬 리라이트 (chat_template + 생성분만 디코드)
# ============================================================

exa_tok = AutoTokenizer.from_pretrained(EXAONE_MODEL_ID, use_fast=True)
exa_model = AutoModelForCausalLM.from_pretrained(
    EXAONE_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
exa_model.eval()
if exa_tok.pad_token_id is None:
    exa_tok.pad_token_id = exa_tok.eos_token_id

def _length_guideline(length_tag: str) -> str:
    if length_tag == "SHORT":
        return "3~5문장"
    elif length_tag == "MEDIUM":
        return "6~8문장"
    else:
        return "9~11문장"

def _build_exa_messages(one_lines: List[str], length_tag: str) -> List[Dict[str, str]]:
    guideline = _length_guideline(length_tag)
    system = (
        "역할: 한국어 하루일기 편집자(로컬 교정자)\n"
        "최우선 원칙: 입력에 없는 사실(새 장소/사물/행동/사건) 절대 추가 금지.\n"
        "목표: 한 줄 기록을 모두 반영해 자연스럽고 반복이 적은 하루일기로 재작성."
    )
    user = (
        "아래 한 줄 기록(사진 캡션)을 모두 반영해서 하루일기를 자연스럽게 재작성하라.\n"
        "제약:\n"
        "- 입력에 없는 새로운 장소/사물/행동/사건을 절대 추가하지 마라.\n"
        "- 각 한 줄의 핵심 요소(행동/표정/상태)를 최소 1회 이상 구체적으로 반영하라.\n"
        "- 문장 반복/접속사 나열을 줄여라.\n"
        "- 감정은 관측 기반으로 1~2문장만 허용한다.\n"
        "- 금지어: 영원/미래/추억/위로/과한 감탄/시적 분위기\n"
        f"- 문장 수: {guideline}\n\n"
        "[한 줄 기록]\n" + "\n".join([f"- {x}" for x in one_lines]) + "\n\n"
        "출력은 ‘하루일기 본문만’ 단독으로 출력하라."
    )
    return [{"role":"system","content":system},{"role":"user","content":user}]

def _postprocess_exa(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    if "</think>" in text:
        text = text.split("</think>")[-1].strip()
    if "출력:" in text:
        text = text.split("출력:")[-1].strip()
    if "하루일기만 출력하라" in text:
        text = text.replace("하루일기만 출력하라.", "").strip()
    return normalize_space(text)

def exaone_rewrite(one_lines: List[str], length_tag: str) -> str:
    messages = _build_exa_messages(one_lines, length_tag)

    # chat template -> string
    try:
        prompt_text = exa_tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    except Exception:
        # fallback plain prompt
        guideline = _length_guideline(length_tag)
        prompt_text = (
            "역할: 한국어 하루일기 편집자\n"
            "목표: 아래 한 줄 기록(사진 캡션)을 모두 반영해서 하루일기를 자연스럽게 재작성한다.\n"
            "제약:\n"
            "- 입력에 없는 새로운 장소/사물/행동/사건을 절대 추가하지 마라.\n"
            "- 각 한 줄의 핵심 요소(행동/표정/상태)를 최소 1회 이상 구체적으로 반영하라.\n"
            "- 문장 반복/접속사 나열을 줄여라.\n"
            "- 감정은 관측 기반으로 1~2문장만 허용한다.\n"
            "- 금지어: 영원/미래/추억/위로/과한 감탄/시적 분위기\n"
            f"- 문장 수: {guideline}\n\n"
            "[한 줄 기록]\n" + "\n".join([f"- {x}" for x in one_lines]) + "\n\n"
            "출력: 하루일기만 출력하라.\n"
        )

    inputs = exa_tok(prompt_text, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(exa_model.device) for k, v in inputs.items()}

    max_new = 220 if length_tag == "SHORT" else (320 if length_tag == "MEDIUM" else 420)

    with torch.no_grad():
        out = exa_model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=False,
            num_beams=3,
            repetition_penalty=1.10,
            eos_token_id=exa_tok.eos_token_id,
            pad_token_id=exa_tok.pad_token_id,
        )

    # 생성된 부분만 디코드
    prompt_len = inputs["input_ids"].shape[-1]
    gen_ids = out[0, prompt_len:]
    text = exa_tok.decode(gen_ids, skip_special_tokens=True)
    return _postprocess_exa(text)

# ============================================================
# (E) student(KoBART) 로드 + 생성
# ============================================================

student_tok = AutoTokenizer.from_pretrained(CURRENT_STUDENT_PATH, use_fast=True)
student_model = AutoModelForSeq2SeqLM.from_pretrained(CURRENT_STUDENT_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"
student_model = student_model.to(device)
student_model.eval()

def gen_student_daily(one_lines: List[str], length_tag: str) -> str:
    src = safe_join_one_lines(one_lines)
    max_new = 160 if length_tag == "SHORT" else (260 if length_tag == "MEDIUM" else 360)
    enc = student_tok(src, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(device) for k, v in enc.items()}
    enc.pop("token_type_ids", None)
    with torch.no_grad():
        out = student_model.generate(
            **enc,
            max_new_tokens=max_new,
            num_beams=4,
            do_sample=False,
            repetition_penalty=1.08,
        )
    return normalize_space(student_tok.decode(out[0], skip_special_tokens=True))

# ============================================================
# (F) student 초안 생성
# ============================================================

def generate_student_batch(target_n: int) -> List[Dict[str, Any]]:
    created = 0
    attempts = 0
    records = []
    while created < target_n and attempts < MAX_ATTEMPTS:
        attempts += 1

        day_situations = sample_day_situations(by_age)
        if not validate_day_diversity(day_situations):
            continue

        one_lines, one_lines_trace = [], []
        for s in day_situations:
            ol, tr = render_one_line_with_trace(s, state_surface)
            one_lines.append(ol)
            one_lines_trace.append(tr)

        one_lines, one_lines_trace = perturb_one_lines(one_lines, one_lines_trace)
        if not validate_one_lines(one_lines):
            continue

        length_tag = weighted_choice(DAILY_LENGTH_DIST)
        diary = gen_student_daily(one_lines, length_tag)

        cov = coverage_score(one_lines, diary)
        nov = novelty_score(one_lines, diary)
        rep = repetition_score(diary)
        score = quick_score(one_lines, diary, length_tag)

        age_group = day_situations[0].age_group if day_situations else ""
        rec = {
            "id": f"r{ROUND_IDX}_student_{created:06d}",
            "stage": f"round{ROUND_IDX}_student_gen",
            "meta": {
                "round_idx": ROUND_IDX,
                "age_group": age_group,
                "k_photos": len(one_lines),
                "length_tag": length_tag,
                "student_model": CURRENT_STUDENT_PATH,
                "seed": SEED,
                "score": score,
                "coverage": cov,
                "novelty": nov,
                "repetition": rep,
            },
            "input": {
                "one_lines": one_lines,
                "joined": safe_join_one_lines(one_lines),
                "one_lines_trace": one_lines_trace,
            },
            "output": {"daily_diary": diary},
        }
        records.append(rec)
        created += 1

        if created % 200 == 0:
            print(f"[STUDENT GEN] {created}/{target_n} attempts={attempts}")

    if created < target_n:
        print("[WARN] student generation ended early:", created, "created")
    return records

student_recs = generate_student_batch(STUDENT_GEN_TARGET)

# ============================================================
# (G) 하이브리드 교정 대상 split (단일 점수 정렬 X, 지표 섞어서 뽑기)
# ============================================================

def _unique_take(items: List[Dict[str, Any]], k: int) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for r in items:
        rid = r["id"]
        if rid in seen:
            continue
        seen.add(rid)
        out.append(r)
        if len(out) >= k:
            break
    return out

k_gpt = int(len(student_recs) * GPT_REWRITE_RATIO)
k_exa = int(len(student_recs) * EXAONE_REWRITE_RATIO)

# 정렬 리스트
by_score_bad = sorted(student_recs, key=lambda r: r["meta"]["score"])                  # 낮을수록 나쁨
by_novelty   = sorted(student_recs, key=lambda r: r["meta"]["novelty"], reverse=True) # 높을수록 위험
by_cov_bad   = sorted(student_recs, key=lambda r: r["meta"]["coverage"])              # 낮을수록 나쁨
by_rep_bad   = sorted(student_recs, key=lambda r: r["meta"]["repetition"], reverse=True)

# GPT: 환각 위험 + 커버리지 최악 중심(비용 비쌈)
gpt_mix = (
    by_novelty[: max(1, k_gpt // 2)] +
    by_cov_bad[: max(1, k_gpt - (k_gpt // 2))]
)
gpt_targets = _unique_take(gpt_mix, k_gpt)

# EXAONE: 반복 + 점수 하위 중심(로컬 교정)
exa_mix = (
    by_rep_bad[: max(1, k_exa // 2)] +
    by_score_bad[: max(1, k_exa - (k_exa // 2))]
)
exa_targets = _unique_take([r for r in exa_mix if r["id"] not in {x["id"] for x in gpt_targets}], k_exa)

# keep: 나머지
gpt_ids = {r["id"] for r in gpt_targets}
exa_ids = {r["id"] for r in exa_targets}
keep_targets = [r for r in student_recs if (r["id"] not in gpt_ids and r["id"] not in exa_ids)]

print("[SPLIT] keep:", len(keep_targets), "exaone:", len(exa_targets), "gpt:", len(gpt_targets))

# ============================================================
# (H) EXAONE 교정(전부 학습에 포함)
# ============================================================

exa_rewritten = []
for i, r in enumerate(exa_targets, 1):
    one_lines = r["input"]["one_lines"]
    length_tag = r["meta"]["length_tag"]

    rew = exaone_rewrite(one_lines, length_tag)
    if (not rew) or contains_banned(rew):
        rew = r["output"]["daily_diary"]

    rr = dict(r)
    rr["stage"] = f"round{ROUND_IDX}_exaone_rewrite"
    rr["meta"] = dict(r["meta"])
    rr["meta"]["rewriter"] = "exaone_local"
    rr["meta"]["exaone_model"] = EXAONE_MODEL_ID
    rr["meta"]["rewritten"] = True
    rr["output"] = {"daily_diary": rew}
    exa_rewritten.append(rr)

    if i % 200 == 0:
        print(f"[EXAONE REWRITE] {i}/{len(exa_targets)}")

# ============================================================
# (I) GPT 최하위 교정(전부 학습에 포함)
# ============================================================

gpt_rewritten = []
for i, r in enumerate(gpt_targets, 1):
    one_lines = r["input"]["one_lines"]
    length_tag = r["meta"]["length_tag"]

    system, user, temp, max_tokens = build_gpt_rewrite_prompt(one_lines, length_tag)
    rew = normalize_space(call_teacher(system, user, temp, max_tokens))

    if (not rew) or contains_banned(rew):
        rew = r["output"]["daily_diary"]

    rr = dict(r)
    rr["stage"] = f"round{ROUND_IDX}_gpt_rewrite"
    rr["meta"] = dict(r["meta"])
    rr["meta"]["rewriter"] = "gpt_final"
    rr["meta"]["teacher_model"] = TEACHER_MODEL
    rr["meta"]["rewritten"] = True
    rr["output"] = {"daily_diary": rew}
    gpt_rewritten.append(rr)

    if i % 100 == 0:
        print(f"[GPT REWRITE] {i}/{len(gpt_targets)}")

# ============================================================
# (J) 저장용 merged_full(전체 보관)
# ============================================================

merged_full = keep_targets + exa_rewritten + gpt_rewritten
random.shuffle(merged_full)

write_jsonl_shards(f"round{ROUND_IDX}_student_gen", student_recs)
write_jsonl_shards(f"round{ROUND_IDX}_exaone_rewritten", exa_rewritten)
write_jsonl_shards(f"round{ROUND_IDX}_gpt_rewritten", gpt_rewritten)
write_jsonl_shards(f"merged_full_round{ROUND_IDX}", merged_full)

print("[DONE] merged_full saved:", f"merged_full_round{ROUND_IDX}_****.jsonl")

# ============================================================
# (K) 학습용 데이터 구성(교정 중심)
# - exaone 전부 + gpt 전부 + keep 상위 일부만
# ============================================================

keep_sorted_good = sorted(keep_targets, key=lambda r: r["meta"]["score"], reverse=True)  # 높을수록 좋음
k_keep_train = max(0, int(len(keep_sorted_good) * KEEP_TRAIN_TOP_RATIO))
keep_train = keep_sorted_good[:k_keep_train]

train_for_finetune = keep_train + exa_rewritten + gpt_rewritten
random.shuffle(train_for_finetune)

write_jsonl_shards(f"train_round{ROUND_IDX}_for_finetune", train_for_finetune)

print("[TRAIN SPLIT]")
print("keep_train:", len(keep_train), "exa:", len(exa_rewritten), "gpt:", len(gpt_rewritten), "total:", len(train_for_finetune))

# ============================================================
# (L) anchors + (교정 중심 train_for_finetune)로 다음 student 학습
# ============================================================

anchors_all = load_jsonl_glob(os.path.join(OUT_DIR, "anchors_*.jsonl"))
train_records = anchors_all + train_for_finetune
random.shuffle(train_records)

def to_hf_dataset(records: List[Dict[str, Any]]) -> Dataset:
    rows = [{"src": r["input"]["joined"], "tgt": r["output"]["daily_diary"]} for r in records]
    return Dataset.from_list(rows)

ds = to_hf_dataset(train_records)
split = ds.train_test_split(test_size=0.02, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

tok = AutoTokenizer.from_pretrained("gogamza/kobart-base-v2", use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(CURRENT_STUDENT_PATH)

MAX_SRC_LEN = 512
MAX_TGT_LEN = 256

def preprocess(batch):
    model_inputs = tok(batch["src"], max_length=MAX_SRC_LEN, truncation=True)
    labels = tok(text_target=batch["tgt"], max_length=MAX_TGT_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tok, model=model)

training_args = TrainingArguments(
    output_dir=f"{OUT_DIR}/runs/kobart_round{ROUND_IDX}",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=4e-5,
    num_train_epochs=2,
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=300,
    save_steps=300,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

# NOTE: transformers 버전에 따라 Trainer(tokenizer=...)가 에러날 수 있음.
# 안전하게 tokenizer 인자는 빼고, 저장은 tok.save_pretrained로 처리.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(NEXT_STUDENT_SAVE_DIR)
tok.save_pretrained(NEXT_STUDENT_SAVE_DIR)

extra = {
    "round_idx": ROUND_IDX,
    "student_gen_target": STUDENT_GEN_TARGET,
    "exaone_rewrite_ratio": EXAONE_REWRITE_RATIO,
    "gpt_rewrite_ratio": GPT_REWRITE_RATIO,
    "keep_train_top_ratio": KEEP_TRAIN_TOP_RATIO,
    "current_student_path": CURRENT_STUDENT_PATH,
    "next_student_path": NEXT_STUDENT_SAVE_DIR,
    "exaone_model_id": EXAONE_MODEL_ID,
    "teacher_model": TEACHER_MODEL,
}
save_run_config(extra=extra)

print("[DONE] next student saved:", NEXT_STUDENT_SAVE_DIR)
print("[NEXT] 다음 라운드 실행 방법")
print("       ROUND_IDX += 1")
print("       CURRENT_STUDENT_PATH = NEXT_STUDENT_SAVE_DIR")
print("       필요하면 EXAONE/GPT 비율을 줄이기")
print("[OUT_DIR]", OUT_DIR)

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


[STUDENT GEN] 200/10000 attempts=200
[STUDENT GEN] 400/10000 attempts=400
[STUDENT GEN] 600/10000 attempts=601
[STUDENT GEN] 800/10000 attempts=801
[STUDENT GEN] 1000/10000 attempts=1001
[STUDENT GEN] 1200/10000 attempts=1201
[STUDENT GEN] 1400/10000 attempts=1401
[STUDENT GEN] 1600/10000 attempts=1601
[STUDENT GEN] 1800/10000 attempts=1801
[STUDENT GEN] 2000/10000 attempts=2001
[STUDENT GEN] 2200/10000 attempts=2201
[STUDENT GEN] 2400/10000 attempts=2401
[STUDENT GEN] 2600/10000 attempts=2602
[STUDENT GEN] 2800/10000 attempts=2802
[STUDENT GEN] 3000/10000 attempts=3003
[STUDENT GEN] 3200/10000 attempts=3204
[STUDENT GEN] 3400/10000 attempts=3404
[STUDENT GEN] 3600/10000 attempts=3604
[STUDENT GEN] 3800/10000 attempts=3804
[STUDENT GEN] 4000/10000 attempts=4004
[STUDENT GEN] 4200/10000 attempts=4204
[STUDENT GEN] 4400/10000 attempts=4404
[STUDENT GEN] 4600/10000 attempts=4604
[STUDENT GEN] 4800/10000 attempts=4804
[STUDENT GEN] 5000/10000 attempts=5004
[STUDENT GEN] 5200/10000 attempts

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Map: 100%|██████████| 134/134 [00:00<00:00, 11173.25 examples/s]
/Users/jeongseong-gyeong/Documents/aiary/aiary/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss
300,0.893300,0.864997
600,0.768500,0.812908


/Users/jeongseong-gyeong/Documents/aiary/aiary/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[DONE] next student saved: generated/aiary_v3_hybrid_20260220_014739/models/kobart_student_round2
[NEXT] 다음 라운드 실행 방법
       ROUND_IDX += 1
       CURRENT_STUDENT_PATH = NEXT_STUDENT_SAVE_DIR
       필요하면 EXAONE/GPT 비율을 줄이기
[OUT_DIR] generated/aiary_v3_hybrid_20260220_014739


In [6]:
# ============================================================
# [CELL 3] Round 3+ 운영(GPT 교정 중심 학습셋)  ✅ EXAONE 제거 버전
# Student 생성 + (지표 기반) GPT 교정 대상 선별 + GPT 교정 + 다음 student 학습
# ------------------------------------------------------------
# - coverage: 한 줄 기록(각 줄) 반영률(0~1)
# - novelty: 입력에 없던 단어 비율(0~1)  -> 추가 사실(환각) 위험
# - repetition: 반복/접속사 과다(0~1 근사)
# - split: keep + gpt_targets
# ============================================================

import os, time, random, re, math
from typing import List, Dict, Any, Tuple

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

# ----------------------------
# 라운드/비율 설정
# ----------------------------
ROUND_IDX = 3
STUDENT_GEN_TARGET = 10000

# ✅ GPT만 사용
GPT_REWRITE_RATIO = 0.20     # 필요하면 0.10~0.30 사이로 조절
KEEP_TRAIN_TOP_RATIO = 0.35  # keep 중 상위만 학습(추천 0.25~0.45)

OUT_DIR = "generated/aiary_v3_hybrid_20260220_014739"

# ✅ Round3 시작 시: CURRENT는 round2 모델이어야 함
CURRENT_STUDENT_PATH = f"{OUT_DIR}/models/kobart_student_round2"
NEXT_STUDENT_SAVE_DIR = f"{OUT_DIR}/models/kobart_student_round{ROUND_IDX}"
os.makedirs(NEXT_STUDENT_SAVE_DIR, exist_ok=True)

# Teacher(GPT) 준비
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)
TEACHER_MODEL = "gpt-4.1-mini"

def call_teacher(system: str, user: str, temperature: float, max_output_tokens: int) -> str:
    backoff = 1.5
    last_err = None
    for attempt in range(1, 9):
        try:
            resp = client.responses.create(
                model=TEACHER_MODEL,
                input=[{"role":"system","content":system},{"role":"user","content":user}],
                temperature=temperature,
                max_output_tokens=max_output_tokens,
            )
            text = getattr(resp, "output_text", None)
            return str(text if text is not None else resp).strip()
        except Exception as e:
            last_err = e
            time.sleep(min(20, backoff ** attempt) + random.random())
    raise RuntimeError(f"Teacher call failed after retries: {last_err}")

# ============================================================
# (A) 직관 지표: coverage / novelty / repetition
# ============================================================

_STOPWORDS = set(["오늘","아침","오전","점심","오후","저녁","밤","아이","아기","엄마","아빠","보호자"])
_CONJ = ["그리고", "그래서", "하지만", "그러다", "그러고", "또", "또는", "또한"]

def _ko_words(s: str) -> List[str]:
    return re.findall(r"[가-힣]{2,}", s)

def _keywords_from_line(line: str, max_k: int = 6) -> List[str]:
    words = [w for w in _ko_words(line) if w not in _STOPWORDS]
    out, seen = [], set()
    for w in words:
        if w in seen:
            continue
        seen.add(w)
        out.append(w)
        if len(out) >= max_k:
            break
    return out

def coverage_score(one_lines: List[str], diary: str) -> float:
    if not one_lines:
        return 0.0
    hit = 0
    for line in one_lines:
        kws = _keywords_from_line(line, max_k=6)
        if not kws:
            hit += 1 if any(tok in diary for tok in _ko_words(line)[:3]) else 0
            continue
        if any(k in diary for k in kws):
            hit += 1
    return hit / max(1, len(one_lines))

def novelty_score(one_lines: List[str], diary: str) -> float:
    in_words = set()
    for l in one_lines:
        in_words.update(_keywords_from_line(l, max_k=999))
    out_words = [w for w in _ko_words(diary) if w not in _STOPWORDS]
    if not out_words:
        return 0.0
    out_set = set(out_words)
    novel = [w for w in out_set if w not in in_words]
    return len(novel) / max(1, len(out_set))

def repetition_score(diary: str) -> float:
    if not diary:
        return 1.0

    conj_cnt = sum(diary.count(c) for c in _CONJ)
    sents = [s.strip() for s in re.split(r"[.!?\n]+", diary) if s.strip()]
    starts = [s[:4] for s in sents if len(s) >= 4]

    start_rep = 0.0
    if starts:
        counts = {}
        for st in starts:
            counts[st] = counts.get(st, 0) + 1
        maxc = max(counts.values())
        start_rep = max(0, (maxc - 1) / max(1, len(starts) - 1))

    words = [w for w in _ko_words(diary) if w not in _STOPWORDS]
    uniq_ratio = (len(set(words)) / max(1, len(words))) if words else 1.0
    word_rep = 1.0 - uniq_ratio

    conj_norm = min(1.0, conj_cnt / 10.0)
    return float(min(1.0, 0.45 * word_rep + 0.35 * start_rep + 0.20 * conj_norm))

# ============================================================
# (B) quick_score: 기존 + coverage/novelty/repetition 반영
# ============================================================

def quick_score(one_lines: List[str], diary: str, length_tag: str) -> float:
    score = 100.0

    if contains_banned(diary):
        score -= 30

    sc = sentence_count_heuristic(diary)
    if length_tag == "SHORT" and not (2 <= sc <= 7):
        score -= 15
    if length_tag == "MEDIUM" and not (5 <= sc <= 11):
        score -= 15
    if length_tag == "LONG" and not (8 <= sc <= 15):
        score -= 15

    cov = coverage_score(one_lines, diary)
    score -= (1.0 - cov) * 35

    nov = novelty_score(one_lines, diary)
    score -= nov * 25

    rep = repetition_score(diary)
    score -= rep * 20

    if len(diary) < 50:
        score -= 10

    return float(score)

# ============================================================
# (C) GPT 프롬프트
# ============================================================

def build_gpt_rewrite_prompt(one_lines: List[str], length_tag: str) -> Tuple[str, str, float, int]:
    if length_tag == "SHORT":
        guideline, max_tokens, temp = "3~5문장", 260, 0.5
    elif length_tag == "MEDIUM":
        guideline, max_tokens, temp = "6~8문장", 420, 0.55
    else:
        guideline, max_tokens, temp = "9~11문장", 560, 0.6

    system = (
        "너는 한국어 하루일기 최종 편집장이다. "
        "입력 한 줄 기록을 모두 반영해 자연스럽게 재작성한다. "
        "추가 사실(새 장소/사물/행동/사건) 금지. "
        "문장 반복/접속사 나열을 줄이고, 관측 기반 감정 표현 1~2문장만 허용. "
        "금지어(영원/미래/추억/위로/과한 감탄/시적 분위기) 사용 금지."
    )
    user = (
        f"아래 한 줄 기록을 바탕으로 하루일기를 재작성하라.\n"
        f"- 문장 수: {guideline}\n"
        f"- 조건: 각 한 줄 내용을 최소 1회 이상 구체적으로 반영\n\n"
        "[한 줄 기록]\n" + "\n".join([f"- {x}" for x in one_lines])
    )
    return system, user, temp, max_tokens

# ============================================================
# (D) student(KoBART) 로드 + 생성
# ============================================================

student_tok = AutoTokenizer.from_pretrained(CURRENT_STUDENT_PATH, use_fast=True)
student_model = AutoModelForSeq2SeqLM.from_pretrained(CURRENT_STUDENT_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"
student_model = student_model.to(device)
student_model.eval()

def gen_student_daily(one_lines: List[str], length_tag: str) -> str:
    src = safe_join_one_lines(one_lines)
    max_new = 160 if length_tag == "SHORT" else (260 if length_tag == "MEDIUM" else 360)
    enc = student_tok(src, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(device) for k, v in enc.items()}
    enc.pop("token_type_ids", None)
    with torch.no_grad():
        out = student_model.generate(
            **enc,
            max_new_tokens=max_new,
            num_beams=4,
            do_sample=False,
            repetition_penalty=1.08,
        )
    return normalize_space(student_tok.decode(out[0], skip_special_tokens=True))

# ============================================================
# (E) student 초안 생성
# ============================================================

def generate_student_batch(target_n: int) -> List[Dict[str, Any]]:
    created = 0
    attempts = 0
    records = []
    while created < target_n and attempts < MAX_ATTEMPTS:
        attempts += 1

        day_situations = sample_day_situations(by_age)
        if not validate_day_diversity(day_situations):
            continue

        one_lines, one_lines_trace = [], []
        for s in day_situations:
            ol, tr = render_one_line_with_trace(s, state_surface)
            one_lines.append(ol)
            one_lines_trace.append(tr)

        one_lines, one_lines_trace = perturb_one_lines(one_lines, one_lines_trace)
        if not validate_one_lines(one_lines):
            continue

        length_tag = weighted_choice(DAILY_LENGTH_DIST)
        diary = gen_student_daily(one_lines, length_tag)

        cov = coverage_score(one_lines, diary)
        nov = novelty_score(one_lines, diary)
        rep = repetition_score(diary)
        score = quick_score(one_lines, diary, length_tag)

        age_group = day_situations[0].age_group if day_situations else ""
        rec = {
            "id": f"r{ROUND_IDX}_student_{created:06d}",
            "stage": f"round{ROUND_IDX}_student_gen",
            "meta": {
                "round_idx": ROUND_IDX,
                "age_group": age_group,
                "k_photos": len(one_lines),
                "length_tag": length_tag,
                "student_model": CURRENT_STUDENT_PATH,
                "seed": SEED,
                "score": score,
                "coverage": cov,
                "novelty": nov,
                "repetition": rep,
            },
            "input": {
                "one_lines": one_lines,
                "joined": safe_join_one_lines(one_lines),
                "one_lines_trace": one_lines_trace,
            },
            "output": {"daily_diary": diary},
        }
        records.append(rec)
        created += 1

        if created % 200 == 0:
            print(f"[STUDENT GEN] {created}/{target_n} attempts={attempts}")

    if created < target_n:
        print("[WARN] student generation ended early:", created, "created")
    return records

student_recs = generate_student_batch(STUDENT_GEN_TARGET)

# ============================================================
# (F) GPT 교정 대상 split (keep vs gpt_targets)
# ============================================================

def _unique_take(items: List[Dict[str, Any]], k: int) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for r in items:
        rid = r["id"]
        if rid in seen:
            continue
        seen.add(rid)
        out.append(r)
        if len(out) >= k:
            break
    return out

k_gpt = int(len(student_recs) * GPT_REWRITE_RATIO)

by_novelty   = sorted(student_recs, key=lambda r: r["meta"]["novelty"], reverse=True)
by_cov_bad   = sorted(student_recs, key=lambda r: r["meta"]["coverage"])
by_score_bad = sorted(student_recs, key=lambda r: r["meta"]["score"])

# 비용을 GPT에 쓰는 목적: "환각 위험 + 커버리지 부족 + 점수 바닥"을 정리
# 믹스 비율은 필요하면 조절 가능
gpt_mix = (
    by_novelty[: max(1, int(k_gpt * 0.45))] +
    by_cov_bad[: max(1, int(k_gpt * 0.35))] +
    by_score_bad[: max(1, k_gpt - int(k_gpt * 0.45) - int(k_gpt * 0.35))]
)
gpt_targets = _unique_take(gpt_mix, k_gpt)

gpt_ids = {r["id"] for r in gpt_targets}
keep_targets = [r for r in student_recs if r["id"] not in gpt_ids]

print("[SPLIT] keep:", len(keep_targets), "gpt:", len(gpt_targets))

# ============================================================
# (G) GPT 교정(전부 학습에 포함)
# ============================================================

gpt_rewritten = []
for i, r in enumerate(gpt_targets, 1):
    one_lines = r["input"]["one_lines"]
    length_tag = r["meta"]["length_tag"]

    system, user, temp, max_tokens = build_gpt_rewrite_prompt(one_lines, length_tag)
    rew = normalize_space(call_teacher(system, user, temp, max_tokens))

    # 안전장치
    if (not rew) or contains_banned(rew):
        rew = r["output"]["daily_diary"]

    rr = dict(r)
    rr["stage"] = f"round{ROUND_IDX}_gpt_rewrite"
    rr["meta"] = dict(r["meta"])
    rr["meta"]["rewriter"] = "gpt_final"
    rr["meta"]["teacher_model"] = TEACHER_MODEL
    rr["meta"]["rewritten"] = True
    rr["output"] = {"daily_diary": rew}
    gpt_rewritten.append(rr)

    if i % 100 == 0:
        print(f"[GPT REWRITE] {i}/{len(gpt_targets)}")

# ============================================================
# (H) 저장용 merged_full(전체 보관)
# ============================================================

merged_full = keep_targets + gpt_rewritten
random.shuffle(merged_full)

write_jsonl_shards(f"round{ROUND_IDX}_student_gen", student_recs)
write_jsonl_shards(f"round{ROUND_IDX}_gpt_rewritten", gpt_rewritten)
write_jsonl_shards(f"merged_full_round{ROUND_IDX}", merged_full)

print("[DONE] merged_full saved:", f"merged_full_round{ROUND_IDX}_****.jsonl")

# ============================================================
# (I) 학습용 데이터 구성(GPT 교정 중심)
# - gpt 전부 + keep 상위 일부만
# ============================================================

keep_sorted_good = sorted(keep_targets, key=lambda r: r["meta"]["score"], reverse=True)
k_keep_train = max(0, int(len(keep_sorted_good) * KEEP_TRAIN_TOP_RATIO))
keep_train = keep_sorted_good[:k_keep_train]

train_for_finetune = keep_train + gpt_rewritten
random.shuffle(train_for_finetune)

write_jsonl_shards(f"train_round{ROUND_IDX}_for_finetune", train_for_finetune)

print("[TRAIN SPLIT]")
print("keep_train:", len(keep_train), "gpt:", len(gpt_rewritten), "total:", len(train_for_finetune))

# ============================================================
# (J) anchors + (교정 중심 train_for_finetune)로 다음 student 학습
# ============================================================

anchors_all = load_jsonl_glob(os.path.join(OUT_DIR, "anchors_*.jsonl"))
train_records = anchors_all + train_for_finetune
random.shuffle(train_records)

def to_hf_dataset(records: List[Dict[str, Any]]) -> Dataset:
    rows = [{"src": r["input"]["joined"], "tgt": r["output"]["daily_diary"]} for r in records]
    return Dataset.from_list(rows)

ds = to_hf_dataset(train_records)
split = ds.train_test_split(test_size=0.02, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

tok = AutoTokenizer.from_pretrained("gogamza/kobart-base-v2", use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(CURRENT_STUDENT_PATH)

MAX_SRC_LEN = 512
MAX_TGT_LEN = 256

def preprocess(batch):
    model_inputs = tok(batch["src"], max_length=MAX_SRC_LEN, truncation=True)
    labels = tok(text_target=batch["tgt"], max_length=MAX_TGT_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tok, model=model)

training_args = TrainingArguments(
    output_dir=f"{OUT_DIR}/runs/kobart_round{ROUND_IDX}",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=4e-5,
    num_train_epochs=2,
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=300,
    save_steps=300,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(NEXT_STUDENT_SAVE_DIR)
tok.save_pretrained(NEXT_STUDENT_SAVE_DIR)

extra = {
    "round_idx": ROUND_IDX,
    "student_gen_target": STUDENT_GEN_TARGET,
    "gpt_rewrite_ratio": GPT_REWRITE_RATIO,
    "keep_train_top_ratio": KEEP_TRAIN_TOP_RATIO,
    "current_student_path": CURRENT_STUDENT_PATH,
    "next_student_path": NEXT_STUDENT_SAVE_DIR,
    "teacher_model": TEACHER_MODEL,
}
save_run_config(extra=extra)

print("[DONE] next student saved:", NEXT_STUDENT_SAVE_DIR)
print("[NEXT] 다음 라운드 실행 방법")
print("       ROUND_IDX += 1")
print("       CURRENT_STUDENT_PATH = NEXT_STUDENT_SAVE_DIR")
print("       필요하면 GPT 비율을 줄이기/늘리기")
print("[OUT_DIR]", OUT_DIR)

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


[STUDENT GEN] 200/10000 attempts=200
[STUDENT GEN] 400/10000 attempts=400
[STUDENT GEN] 600/10000 attempts=601
[STUDENT GEN] 800/10000 attempts=801
[STUDENT GEN] 1000/10000 attempts=1001
[STUDENT GEN] 1200/10000 attempts=1201
[STUDENT GEN] 1400/10000 attempts=1401
[STUDENT GEN] 1600/10000 attempts=1601
[STUDENT GEN] 1800/10000 attempts=1801
[STUDENT GEN] 2000/10000 attempts=2001
[STUDENT GEN] 2200/10000 attempts=2201
[STUDENT GEN] 2400/10000 attempts=2401
[STUDENT GEN] 2600/10000 attempts=2602
[STUDENT GEN] 2800/10000 attempts=2802
[STUDENT GEN] 3000/10000 attempts=3003
[STUDENT GEN] 3200/10000 attempts=3204
[STUDENT GEN] 3400/10000 attempts=3404
[STUDENT GEN] 3600/10000 attempts=3604
[STUDENT GEN] 3800/10000 attempts=3804
[STUDENT GEN] 4000/10000 attempts=4004
[STUDENT GEN] 4200/10000 attempts=4204
[STUDENT GEN] 4400/10000 attempts=4404
[STUDENT GEN] 4600/10000 attempts=4604
[STUDENT GEN] 4800/10000 attempts=4804
[STUDENT GEN] 5000/10000 attempts=5004
[STUDENT GEN] 5200/10000 attempts

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Map: 100%|██████████| 131/131 [00:00<00:00, 10706.01 examples/s]
/Users/jeongseong-gyeong/Documents/aiary/aiary/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss
300,0.700600,0.606313
600,0.605800,0.576814


/Users/jeongseong-gyeong/Documents/aiary/aiary/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[DONE] next student saved: generated/aiary_v3_hybrid_20260220_014739/models/kobart_student_round3
[NEXT] 다음 라운드 실행 방법
       ROUND_IDX += 1
       CURRENT_STUDENT_PATH = NEXT_STUDENT_SAVE_DIR
       필요하면 GPT 비율을 줄이기/늘리기
[OUT_DIR] generated/aiary_v3_hybrid_20260220_014739


In [5]:
# ============================================================
# [CELL 3] Round 4+ 운영(GPT 교정 중심 학습셋)  ✅ EXAONE 제거 버전
# Student 생성 + (지표 기반) GPT 교정 대상 선별 + GPT 교정 + 다음 student 학습
# ============================================================

import os, time, random, re, math
from typing import List, Dict, Any, Tuple

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

# ----------------------------
# 라운드/비율 설정
# ----------------------------
ROUND_IDX = 4
STUDENT_GEN_TARGET = 10000

# ✅ GPT만 사용
GPT_REWRITE_RATIO = 0.20     # 필요하면 0.10~0.30 사이로 조절
KEEP_TRAIN_TOP_RATIO = 0.35  # keep 중 상위만 학습(추천 0.25~0.45)

OUT_DIR = "generated/aiary_v3_hybrid_20260220_014739"

# ✅ Round4 시작 시: CURRENT는 round3 모델이어야 함
CURRENT_STUDENT_PATH = f"{OUT_DIR}/models/kobart_student_round3"
NEXT_STUDENT_SAVE_DIR = f"{OUT_DIR}/models/kobart_student_round{ROUND_IDX}"
os.makedirs(NEXT_STUDENT_SAVE_DIR, exist_ok=True)

# Teacher(GPT) 준비
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)
TEACHER_MODEL = "gpt-4.1-mini"

def call_teacher(system: str, user: str, temperature: float, max_output_tokens: int) -> str:
    backoff = 1.5
    last_err = None
    for attempt in range(1, 9):
        try:
            resp = client.responses.create(
                model=TEACHER_MODEL,
                input=[{"role":"system","content":system},{"role":"user","content":user}],
                temperature=temperature,
                max_output_tokens=max_output_tokens,
            )
            text = getattr(resp, "output_text", None)
            return str(text if text is not None else resp).strip()
        except Exception as e:
            last_err = e
            time.sleep(min(20, backoff ** attempt) + random.random())
    raise RuntimeError(f"Teacher call failed after retries: {last_err}")

# ============================================================
# (A) 직관 지표: coverage / novelty / repetition
# ============================================================

_STOPWORDS = set(["오늘","아침","오전","점심","오후","저녁","밤","아이","아기","엄마","아빠","보호자"])
_CONJ = ["그리고", "그래서", "하지만", "그러다", "그러고", "또", "또는", "또한"]

def _ko_words(s: str) -> List[str]:
    return re.findall(r"[가-힣]{2,}", s)

def _keywords_from_line(line: str, max_k: int = 6) -> List[str]:
    words = [w for w in _ko_words(line) if w not in _STOPWORDS]
    out, seen = [], set()
    for w in words:
        if w in seen:
            continue
        seen.add(w)
        out.append(w)
        if len(out) >= max_k:
            break
    return out

def coverage_score(one_lines: List[str], diary: str) -> float:
    if not one_lines:
        return 0.0
    hit = 0
    for line in one_lines:
        kws = _keywords_from_line(line, max_k=6)
        if not kws:
            hit += 1 if any(tok in diary for tok in _ko_words(line)[:3]) else 0
            continue
        if any(k in diary for k in kws):
            hit += 1
    return hit / max(1, len(one_lines))

def novelty_score(one_lines: List[str], diary: str) -> float:
    in_words = set()
    for l in one_lines:
        in_words.update(_keywords_from_line(l, max_k=999))
    out_words = [w for w in _ko_words(diary) if w not in _STOPWORDS]
    if not out_words:
        return 0.0
    out_set = set(out_words)
    novel = [w for w in out_set if w not in in_words]
    return len(novel) / max(1, len(out_set))

def repetition_score(diary: str) -> float:
    if not diary:
        return 1.0

    conj_cnt = sum(diary.count(c) for c in _CONJ)
    sents = [s.strip() for s in re.split(r"[.!?\n]+", diary) if s.strip()]
    starts = [s[:4] for s in sents if len(s) >= 4]

    start_rep = 0.0
    if starts:
        counts = {}
        for st in starts:
            counts[st] = counts.get(st, 0) + 1
        maxc = max(counts.values())
        start_rep = max(0, (maxc - 1) / max(1, len(starts) - 1))

    words = [w for w in _ko_words(diary) if w not in _STOPWORDS]
    uniq_ratio = (len(set(words)) / max(1, len(words))) if words else 1.0
    word_rep = 1.0 - uniq_ratio

    conj_norm = min(1.0, conj_cnt / 10.0)
    return float(min(1.0, 0.45 * word_rep + 0.35 * start_rep + 0.20 * conj_norm))

# ============================================================
# (B) quick_score: 기존 + coverage/novelty/repetition 반영
# ============================================================

def quick_score(one_lines: List[str], diary: str, length_tag: str) -> float:
    score = 100.0

    if contains_banned(diary):
        score -= 30

    sc = sentence_count_heuristic(diary)
    if length_tag == "SHORT" and not (2 <= sc <= 7):
        score -= 15
    if length_tag == "MEDIUM" and not (5 <= sc <= 11):
        score -= 15
    if length_tag == "LONG" and not (8 <= sc <= 15):
        score -= 15

    cov = coverage_score(one_lines, diary)
    score -= (1.0 - cov) * 35

    nov = novelty_score(one_lines, diary)
    score -= nov * 25

    rep = repetition_score(diary)
    score -= rep * 20

    if len(diary) < 50:
        score -= 10

    return float(score)

# ============================================================
# (C) GPT 프롬프트
# ============================================================

def build_gpt_rewrite_prompt(one_lines: List[str], length_tag: str) -> Tuple[str, str, float, int]:
    if length_tag == "SHORT":
        guideline, max_tokens, temp = "3~5문장", 260, 0.5
    elif length_tag == "MEDIUM":
        guideline, max_tokens, temp = "6~8문장", 420, 0.55
    else:
        guideline, max_tokens, temp = "9~11문장", 560, 0.6

    system = (
        "너는 한국어 하루일기 최종 편집장이다. "
        "입력 한 줄 기록을 모두 반영해 자연스럽게 재작성한다. "
        "추가 사실(새 장소/사물/행동/사건) 금지. "
        "문장 반복/접속사 나열을 줄이고, 관측 기반 감정 표현 1~2문장만 허용. "
        "금지어(영원/미래/추억/위로/과한 감탄/시적 분위기) 사용 금지."
    )
    user = (
        f"아래 한 줄 기록을 바탕으로 하루일기를 재작성하라.\n"
        f"- 문장 수: {guideline}\n"
        f"- 조건: 각 한 줄 내용을 최소 1회 이상 구체적으로 반영\n\n"
        "[한 줄 기록]\n" + "\n".join([f"- {x}" for x in one_lines])
    )
    return system, user, temp, max_tokens

# ============================================================
# (D) student(KoBART) 로드 + 생성
# ============================================================

student_tok = AutoTokenizer.from_pretrained(CURRENT_STUDENT_PATH, use_fast=True)
student_model = AutoModelForSeq2SeqLM.from_pretrained(CURRENT_STUDENT_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"
student_model = student_model.to(device)
student_model.eval()

def gen_student_daily(one_lines: List[str], length_tag: str) -> str:
    src = safe_join_one_lines(one_lines)
    max_new = 160 if length_tag == "SHORT" else (260 if length_tag == "MEDIUM" else 360)
    enc = student_tok(src, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(device) for k, v in enc.items()}
    enc.pop("token_type_ids", None)
    with torch.no_grad():
        out = student_model.generate(
            **enc,
            max_new_tokens=max_new,
            num_beams=4,
            do_sample=False,
            repetition_penalty=1.08,
        )
    return normalize_space(student_tok.decode(out[0], skip_special_tokens=True))

# ============================================================
# (E) student 초안 생성
# ============================================================

def generate_student_batch(target_n: int) -> List[Dict[str, Any]]:
    created = 0
    attempts = 0
    records = []
    while created < target_n and attempts < MAX_ATTEMPTS:
        attempts += 1

        day_situations = sample_day_situations(by_age)
        if not validate_day_diversity(day_situations):
            continue

        one_lines, one_lines_trace = [], []
        for s in day_situations:
            ol, tr = render_one_line_with_trace(s, state_surface)
            one_lines.append(ol)
            one_lines_trace.append(tr)

        one_lines, one_lines_trace = perturb_one_lines(one_lines, one_lines_trace)
        if not validate_one_lines(one_lines):
            continue

        length_tag = weighted_choice(DAILY_LENGTH_DIST)
        diary = gen_student_daily(one_lines, length_tag)

        cov = coverage_score(one_lines, diary)
        nov = novelty_score(one_lines, diary)
        rep = repetition_score(diary)
        score = quick_score(one_lines, diary, length_tag)

        age_group = day_situations[0].age_group if day_situations else ""
        rec = {
            "id": f"r{ROUND_IDX}_student_{created:06d}",
            "stage": f"round{ROUND_IDX}_student_gen",
            "meta": {
                "round_idx": ROUND_IDX,
                "age_group": age_group,
                "k_photos": len(one_lines),
                "length_tag": length_tag,
                "student_model": CURRENT_STUDENT_PATH,
                "seed": SEED,
                "score": score,
                "coverage": cov,
                "novelty": nov,
                "repetition": rep,
            },
            "input": {
                "one_lines": one_lines,
                "joined": safe_join_one_lines(one_lines),
                "one_lines_trace": one_lines_trace,
            },
            "output": {"daily_diary": diary},
        }
        records.append(rec)
        created += 1

        if created % 200 == 0:
            print(f"[STUDENT GEN] {created}/{target_n} attempts={attempts}")

    if created < target_n:
        print("[WARN] student generation ended early:", created, "created")
    return records

student_recs = generate_student_batch(STUDENT_GEN_TARGET)

# ============================================================
# (F) GPT 교정 대상 split (keep vs gpt_targets)
# ============================================================

def _unique_take(items: List[Dict[str, Any]], k: int) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for r in items:
        rid = r["id"]
        if rid in seen:
            continue
        seen.add(rid)
        out.append(r)
        if len(out) >= k:
            break
    return out

k_gpt = int(len(student_recs) * GPT_REWRITE_RATIO)

by_novelty   = sorted(student_recs, key=lambda r: r["meta"]["novelty"], reverse=True)
by_cov_bad   = sorted(student_recs, key=lambda r: r["meta"]["coverage"])
by_score_bad = sorted(student_recs, key=lambda r: r["meta"]["score"])

gpt_mix = (
    by_novelty[: max(1, int(k_gpt * 0.45))] +
    by_cov_bad[: max(1, int(k_gpt * 0.35))] +
    by_score_bad[: max(1, k_gpt - int(k_gpt * 0.45) - int(k_gpt * 0.35))]
)
gpt_targets = _unique_take(gpt_mix, k_gpt)

gpt_ids = {r["id"] for r in gpt_targets}
keep_targets = [r for r in student_recs if r["id"] not in gpt_ids]

print("[SPLIT] keep:", len(keep_targets), "gpt:", len(gpt_targets))

# ============================================================
# (G) GPT 교정(전부 학습에 포함)
# ============================================================

gpt_rewritten = []
for i, r in enumerate(gpt_targets, 1):
    one_lines = r["input"]["one_lines"]
    length_tag = r["meta"]["length_tag"]

    system, user, temp, max_tokens = build_gpt_rewrite_prompt(one_lines, length_tag)
    rew = normalize_space(call_teacher(system, user, temp, max_tokens))

    if (not rew) or contains_banned(rew):
        rew = r["output"]["daily_diary"]

    rr = dict(r)
    rr["stage"] = f"round{ROUND_IDX}_gpt_rewrite"
    rr["meta"] = dict(r["meta"])
    rr["meta"]["rewriter"] = "gpt_final"
    rr["meta"]["teacher_model"] = TEACHER_MODEL
    rr["meta"]["rewritten"] = True
    rr["output"] = {"daily_diary": rew}
    gpt_rewritten.append(rr)

    if i % 100 == 0:
        print(f"[GPT REWRITE] {i}/{len(gpt_targets)}")

# ============================================================
# (H) 저장용 merged_full(전체 보관)
# ============================================================

merged_full = keep_targets + gpt_rewritten
random.shuffle(merged_full)

write_jsonl_shards(f"round{ROUND_IDX}_student_gen", student_recs)
write_jsonl_shards(f"round{ROUND_IDX}_gpt_rewritten", gpt_rewritten)
write_jsonl_shards(f"merged_full_round{ROUND_IDX}", merged_full)

print("[DONE] merged_full saved:", f"merged_full_round{ROUND_IDX}_****.jsonl")

# ============================================================
# (I) 학습용 데이터 구성(GPT 교정 중심)
# - gpt 전부 + keep 상위 일부만
# ============================================================

keep_sorted_good = sorted(keep_targets, key=lambda r: r["meta"]["score"], reverse=True)
k_keep_train = max(0, int(len(keep_sorted_good) * KEEP_TRAIN_TOP_RATIO))
keep_train = keep_sorted_good[:k_keep_train]

train_for_finetune = keep_train + gpt_rewritten
random.shuffle(train_for_finetune)

write_jsonl_shards(f"train_round{ROUND_IDX}_for_finetune", train_for_finetune)

print("[TRAIN SPLIT]")
print("keep_train:", len(keep_train), "gpt:", len(gpt_rewritten), "total:", len(train_for_finetune))

# ============================================================
# (J) anchors + (교정 중심 train_for_finetune)로 다음 student 학습
# ============================================================

anchors_all = load_jsonl_glob(os.path.join(OUT_DIR, "anchors_*.jsonl"))
train_records = anchors_all + train_for_finetune
random.shuffle(train_records)

def to_hf_dataset(records: List[Dict[str, Any]]) -> Dataset:
    rows = [{"src": r["input"]["joined"], "tgt": r["output"]["daily_diary"]} for r in records]
    return Dataset.from_list(rows)

ds = to_hf_dataset(train_records)
split = ds.train_test_split(test_size=0.02, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

tok = AutoTokenizer.from_pretrained("gogamza/kobart-base-v2", use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(CURRENT_STUDENT_PATH)

MAX_SRC_LEN = 512
MAX_TGT_LEN = 256

def preprocess(batch):
    model_inputs = tok(batch["src"], max_length=MAX_SRC_LEN, truncation=True)
    labels = tok(text_target=batch["tgt"], max_length=MAX_TGT_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tok, model=model)

training_args = TrainingArguments(
    output_dir=f"{OUT_DIR}/runs/kobart_round{ROUND_IDX}",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=4e-5,
    num_train_epochs=2,
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=300,
    save_steps=300,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(NEXT_STUDENT_SAVE_DIR)
tok.save_pretrained(NEXT_STUDENT_SAVE_DIR)

extra = {
    "round_idx": ROUND_IDX,
    "student_gen_target": STUDENT_GEN_TARGET,
    "gpt_rewrite_ratio": GPT_REWRITE_RATIO,
    "keep_train_top_ratio": KEEP_TRAIN_TOP_RATIO,
    "current_student_path": CURRENT_STUDENT_PATH,
    "next_student_path": NEXT_STUDENT_SAVE_DIR,
    "teacher_model": TEACHER_MODEL,
}
save_run_config(extra=extra)

print("[DONE] next student saved:", NEXT_STUDENT_SAVE_DIR)
print("[NEXT] 다음 라운드 실행 방법")
print("       ROUND_IDX += 1")
print("       CURRENT_STUDENT_PATH = NEXT_STUDENT_SAVE_DIR")
print("       필요하면 GPT 비율을 줄이기/늘리기")
print("[OUT_DIR]", OUT_DIR)

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


[STUDENT GEN] 200/10000 attempts=200
[STUDENT GEN] 400/10000 attempts=400
[STUDENT GEN] 600/10000 attempts=601
[STUDENT GEN] 800/10000 attempts=801
[STUDENT GEN] 1000/10000 attempts=1001
[STUDENT GEN] 1200/10000 attempts=1201
[STUDENT GEN] 1400/10000 attempts=1401
[STUDENT GEN] 1600/10000 attempts=1601
[STUDENT GEN] 1800/10000 attempts=1801
[STUDENT GEN] 2000/10000 attempts=2001
[STUDENT GEN] 2200/10000 attempts=2201
[STUDENT GEN] 2400/10000 attempts=2401
[STUDENT GEN] 2600/10000 attempts=2602
[STUDENT GEN] 2800/10000 attempts=2802
[STUDENT GEN] 3000/10000 attempts=3003
[STUDENT GEN] 3200/10000 attempts=3204
[STUDENT GEN] 3400/10000 attempts=3404
[STUDENT GEN] 3600/10000 attempts=3604
[STUDENT GEN] 3800/10000 attempts=3804
[STUDENT GEN] 4000/10000 attempts=4004
[STUDENT GEN] 4200/10000 attempts=4204
[STUDENT GEN] 4400/10000 attempts=4404
[STUDENT GEN] 4600/10000 attempts=4604
[STUDENT GEN] 4800/10000 attempts=4804
[STUDENT GEN] 5000/10000 attempts=5004
[STUDENT GEN] 5200/10000 attempts

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Map: 100%|██████████| 131/131 [00:00<00:00, 12366.73 examples/s]
/Users/jeongseong-gyeong/Documents/aiary/aiary/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss
300,0.627200,0.602033
600,0.543500,0.577387


/Users/jeongseong-gyeong/Documents/aiary/aiary/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[DONE] next student saved: generated/aiary_v3_hybrid_20260220_014739/models/kobart_student_round4
[NEXT] 다음 라운드 실행 방법
       ROUND_IDX += 1
       CURRENT_STUDENT_PATH = NEXT_STUDENT_SAVE_DIR
       필요하면 GPT 비율을 줄이기/늘리기
[OUT_DIR] generated/aiary_v3_hybrid_20260220_014739


In [ ]:
import os, glob

CURRENT_STUDENT_PATH = "/content/drive/MyDrive/aiary_공유문서함/generated/aiary_v3_hybrid_20260220_014739/models/kobart_student_round1"

print("exists:", os.path.exists(CURRENT_STUDENT_PATH))
print("files:")
for p in sorted(glob.glob(CURRENT_STUDENT_PATH + "/*")):
    print(" -", os.path.basename(p))

exists: True
files:
 - config.json
 - generation_config.json
 - special_tokens_map.json
 - tokenizer.json
 - tokenizer_config.json
 - training_args.bin


In [ ]:
import os, glob, shutil

OUT_DIR = "/content/drive/MyDrive/aiary_공유문서함/generated/aiary_v3_hybrid_20260220_014739"
run_dir = os.path.join(OUT_DIR, "runs", "kobart_round1")

ckpts = sorted(glob.glob(os.path.join(run_dir, "checkpoint-*")), key=lambda p: int(p.split("-")[-1]) if p.split("-")[-1].isdigit() else -1)
print("found checkpoints:", len(ckpts))
for p in ckpts[-10:]:
    files = os.listdir(p)
    has_weight = ("model.safetensors" in files) or ("pytorch_model.bin" in files)
    print(" -", p, "weight=", has_weight)

# 가장 마지막(스텝 큰) 체크포인트에서 weight 찾기
best = None
for p in reversed(ckpts):
    files = os.listdir(p)
    if ("model.safetensors" in files) or ("pytorch_model.bin" in files):
        best = p
        break

print("best checkpoint:", best)

# (옵션A) 그냥 이 경로를 student 경로로 쓰기
CURRENT_STUDENT_PATH = best

# (옵션B) best checkpoint의 파일을 models/kobart_student_round1로 복사해서 '정식 student 폴더' 만들기
dst = os.path.join(OUT_DIR, "models", "kobart_student_round1")
os.makedirs(dst, exist_ok=True)

for fn in ["model.safetensors","pytorch_model.bin","config.json","generation_config.json","tokenizer.json","tokenizer_config.json","special_tokens_map.json"]:
    src = os.path.join(best, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(dst, fn))

print("dst files:", sorted(os.listdir(dst)))

found checkpoints: 2
 - /content/drive/MyDrive/aiary_공유문서함/generated/aiary_v3_hybrid_20260220_014739/runs/kobart_round1/checkpoint-200 weight= True
 - /content/drive/MyDrive/aiary_공유문서함/generated/aiary_v3_hybrid_20260220_014739/runs/kobart_round1/checkpoint-246 weight= False
best checkpoint: /content/drive/MyDrive/aiary_공유문서함/generated/aiary_v3_hybrid_20260220_014739/runs/kobart_round1/checkpoint-200
dst files: ['config.json', 'generation_config.json', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


In [1]:
%pip install -q emoji regex pandas


Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 11.5 MB/s  0:00:00
  Attempting uninstall: emoji
    Found existing installation: emoji 1.2.0
    Uninstalling emoji-1.2.0:
      Successfully uninstalled emoji-1.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kss 6.0.6 requires emoji==1.2.0, but you have emoji 2.15.0 which is incompatible.
pecab 1.0.8 requires emoji==1.2.0, but you have emoji 2.15.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import json
import torch
import regex as re
import emoji
from typing import List
from transformers import BartForConditionalGeneration, PreTrainedTokenizerFast

# ---------------------------------------
# 1) 경로 설정
# ---------------------------------------
BASE_DIR = "."

MODEL_DIR = os.path.join(BASE_DIR, "generated/aiary_v3_hybrid_20260220_014739/models/kobart_student_round4")
ONE_LINE_FILE = os.path.join(BASE_DIR, "data/one_line_texts.txt")

OUTPUT_JSON = os.path.join(BASE_DIR, "outputs/generated_diary_from_one_line.json")
OUTPUT_TXT  = os.path.join(BASE_DIR, "outputs/generated_diary_from_one_line.txt")

MAX_INPUT_LEN = 256

# "max_new_tokens"는 생성 길이 상한(가장 중요)
MAX_NEW_TOKENS = 180

# beam search 사용 시 권장
NUM_BEAMS = 4

SEED = 777


# ---------------------------------------
# 2) 이모지 제거 + 텍스트 정제 함수
# ---------------------------------------
def remove_emoji(text: str) -> str:
    text = emoji.replace_emoji(text, "")
    emoji_pattern = re.compile(r"[\p{Emoji}\p{Emoji_Presentation}\p{Extended_Pictographic}]", flags=re.UNICODE)
    return emoji_pattern.sub("", text)

def clean_sentence(text: str) -> str:
    text = str(text)
    text = remove_emoji(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def split_sentences_ko(text: str) -> List[str]:
    # 간단한 문장 분리(종결부호 기준)
    # 필요하면 kss 같은 걸로 더 고급화 가능하지만, 추론 코드엔 이 정도가 안전함
    parts = re.split(r"(?<=[.!?。！？])\s+|(?<=[다요죠]\.)\s+|(?<=다)\s+(?=[가-힣])", text.strip())
    parts = [p.strip() for p in parts if p and p.strip()]
    return parts

def dedup_sentences(text: str, max_sentences: int = 10) -> str:
    sents = split_sentences_ko(text)
    seen = set()
    out = []
    for s in sents:
        key = re.sub(r"\s+", "", s)
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
        if len(out) >= max_sentences:
            break
    return " ".join(out).strip()

def trim_tail(text: str, max_chars: int = 900) -> str:
    text = text.strip()
    if len(text) <= max_chars:
        return text
    cut = text[:max_chars]
    # 마지막 문장 끝에서 자르기
    m = re.search(r"^(.+?[.!?。！？])[^.!?。！？]*$", cut)
    return m.group(1).strip() if m else cut.strip()


# ---------------------------------------
# 3) 모델 / 토크나이저 로드
# ---------------------------------------
print("[INFO] 모델 / 토크나이저 로드 중...")
if not os.path.exists(MODEL_DIR):
    raise FileNotFoundError(f"모델 디렉토리를 찾을 수 없습니다: {MODEL_DIR}")

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_DIR)
model = BartForConditionalGeneration.from_pretrained(MODEL_DIR)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(f"[INFO] device = {device}")
print("[INFO] 모델 로드 완료")


# ---------------------------------------
# 4) 요약(여러 줄) → 하루일기 생성 함수
# ---------------------------------------
def generate_diary_from_summary(summary_text: str) -> str:
    input_text = f"[SUMMARY]\n{summary_text}\n[DIARY]"

    enc = tokenizer(
        input_text,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=True,              # padding="max_length"보다 안정적(불필요 pad 줄임)
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    # (선택) 자주 나오는 군더더기 문구를 금지하고 싶으면 여기 추가
    banned_phrases = [
        "오늘 하루 아이와 함께한 소소한 순간들이",
        "조금 안도",
        "세심하게 살펴야겠다는 생각",
        "앞으로도 이런 순간",
    ]
    bad_words_ids = []
    for p in banned_phrases:
        ids = tokenizer.encode(p, add_special_tokens=False)
        if len(ids) > 0:
            bad_words_ids.append(ids)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,

            max_new_tokens=140,      # 180 -> 140으로 더 강하게 컷
            min_new_tokens=60,

            do_sample=False,
            num_beams=5,             # 4 -> 5
            early_stopping=True,

            no_repeat_ngram_size=3,  # 4 -> 3
            repetition_penalty=1.25, # 1.15 -> 1.25
            length_penalty=0.8,      # 짧게 유도(중요)

            bad_words_ids=bad_words_ids if bad_words_ids else None,

            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred = pred.replace("[DIARY]", "").strip()

    def drop_cliche_sentences(text: str) -> str:
        sents = split_sentences_ko(text)
        banned_patterns = [
            r"마음이 놓였다",
            r"소중하게 다가왔다",
            r"세심하게 살펴야겠다는 생각",
            r"안심이 되었다",
            r"생각이 들었다",
        ]
        out = []
        for s in sents:
            if any(re.search(p, s) for p in banned_patterns):
                continue
            out.append(s)
        return " ".join(out).strip()

    # 후처리: 중복 문장 제거 + 꼬리 자르기
    pred = drop_cliche_sentences(pred)
    pred = dedup_sentences(pred, max_sentences=8)
    pred = trim_tail(pred, max_chars=700)
    return pred


# ---------------------------------------
# 5) one_line_texts.txt 로드 (전체가 하루 인풋)
# ---------------------------------------
if not os.path.exists(ONE_LINE_FILE):
    raise FileNotFoundError(f"한 줄 일기 파일을 찾을 수 없습니다: {ONE_LINE_FILE}")

with open(ONE_LINE_FILE, "r", encoding="utf-8") as f:
    raw_lines = [line.strip() for line in f.readlines() if line.strip()]

if not raw_lines:
    raise ValueError("one_line_texts.txt 안에 내용이 없습니다.")

print(f"[INFO] 파일에서 읽은 문장 개수: {len(raw_lines)}")
print("[LOG] 원본 한 줄 일기들:")
for ex in raw_lines:
    print("-", ex)

cleaned_sentences = [clean_sentence(s) for s in raw_lines]

bullet_lines = [f"{i}. {sent}" for i, sent in enumerate(cleaned_sentences, start=1)]
combined_summary = "\n".join(bullet_lines)

print("\n[LOG] 모델에 들어갈 SUMMARY 포맷:")
print(combined_summary)


# ---------------------------------------
# 6) 하루일기 생성
# ---------------------------------------
print("\n[INFO] 하루 일기 생성 중...")
generated_diary = generate_diary_from_summary(combined_summary)
print("\n[GENERATED DIARY]")
print(generated_diary)


# ---------------------------------------
# 7) 결과 저장 (JSON + TXT)
# ---------------------------------------
result_obj = {
    "raw_lines": raw_lines,
    "cleaned_summary_bullets": bullet_lines,
    "combined_summary": combined_summary,
    "generated_diary": generated_diary,
}

os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(result_obj, f, ensure_ascii=False, indent=2)

with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    f.write(generated_diary + "\n")

print(f"\n[INFO] JSON 저장 완료: {OUTPUT_JSON}")
print(f"[INFO] TXT 저장 완료:  {OUTPUT_TXT}")
print("\n[INFO] 인퍼런스 전체 완료")

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


[INFO] 모델 / 토크나이저 로드 중...
[INFO] device = cpu
[INFO] 모델 로드 완료
[INFO] 파일에서 읽은 문장 개수: 5
[LOG] 원본 한 줄 일기들:
- 따뜻한 차 안에서 과자 먹방 시작🍪🚗💕
- 고기 냄새 가득한 따뜻한 가족 외식시간🍖🥰
- 장난감 총과 친구, 오늘도 신나게 놀았어요🔫🤠✨
- 따끈한 고기 냄새에 기대 가득한 밤🍖✨
- 불꽃 속에서 아빠 손 잡고 따뜻함을 느꼈어요🔥🤗

[LOG] 모델에 들어갈 SUMMARY 포맷:
1. 따뜻한 차 안에서 과자 먹방 시작
2. 고기 냄새 가득한 따뜻한 가족 외식시간
3. 장난감 총과 친구, 오늘도 신나게 놀았어요
4. 따끈한 고기 냄새에 기대 가득한 밤
5. 불꽃 속에서 아빠 손 잡고 따뜻함을 느꼈어요

[INFO] 하루 일기 생성 중...

[GENERATED DIARY]
오늘 아침, 아이가 따뜻한 차 안에서 과자 먹방을 하며 즐거워하는 모습을 보았다. 한편 장난감 총과 친구는 오늘도 신나게 놀며 즐거운 시간을 보냈다. 오후에는 아빠가 옆에서 지켜보는 가운데 아이가 따끈한 고기 냄새에 기대어 편안해하는 모습이 인상적이었다. 저녁에는 아빠와 함께 불꽃 놀이를 하며 아빠 손 잡고 따뜻함을 느꼈다. 아이가 다양한 감정을 느끼는 모습을 보며 조금은 안도하는 마음이 들었다. 오후에는 텐감 정과 친구와 함께 장난

[INFO] JSON 저장 완료: ./outputs/generated_diary_from_one_line.json
[INFO] TXT 저장 완료:  ./outputs/generated_diary_from_one_line.txt

[INFO] 인퍼런스 전체 완료


In [3]:
# ============================================================
# Replay(과거 train shard) 정제 통과율 확인 + (선택) replay 샘플링까지
# - 너가 저장해둔 train_round*_for_finetune_****.jsonl 기준
# - 각 라운드별: 총개수 / struct_ok 통과 / 통과율 / 주요 실패 원인 카운트 출력
# ============================================================

import os, glob, json, random, re
from collections import Counter, defaultdict
from typing import List, Dict, Any, Tuple

SEED = 777
random.seed(SEED)

OUT_DIR = "generated/aiary_v3_hybrid_20260220_014739"
# 예: train_round4_for_finetune_0000.jsonl 이런 식이면 아래 glob이 잡힘
TRAIN_GLOB = os.path.join(OUT_DIR, "train_round*_for_finetune_*.jsonl")

# ----------------------------
# (1) JSONL 로더
# ----------------------------
def load_jsonl(path: str) -> List[Dict[str, Any]]:
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            out.append(json.loads(line))
    return out

def load_jsonl_glob(pattern: str) -> List[Dict[str, Any]]:
    paths = sorted(glob.glob(pattern))
    all_rows = []
    for p in paths:
        all_rows.extend(load_jsonl(p))
    return all_rows

# ----------------------------
# (2) 구조 필터(라운드5에 쓴 것과 동일 컨셉)
# ----------------------------
_CLICHE_PATTERNS = [
    r"오늘 하루.*소중",
    r"소중하게 다가왔",
    r"마음(이 )?놓였",
    r"안도(하|한|하는)",
    r"세심하게 살펴야겠",
    r"생각(이 )?들었",
    r"평범한 하루",
    r"마음(이 )?따뜻",
    r"작은 변화",
    r"눈에 들어온 날",
]

def sentence_count_heuristic(text: str) -> int:
    if not text:
        return 0
    parts = [p.strip() for p in re.split(r"[.!?\n]+", text) if p.strip()]
    return len(parts)

def has_unfinished_tail(text: str) -> bool:
    if not text:
        return True
    t = text.strip()
    if re.search(r"[.!?。！？]\s*$", t):
        return False
    if re.search(r"(다|요|죠|음)\s*$", t):
        return False
    return True

def count_cliches(text: str) -> int:
    if not text:
        return 0
    c = 0
    for p in _CLICHE_PATTERNS:
        if re.search(p, text):
            c += 1
    return c

def exact_duplicate_sentence_ratio(text: str) -> float:
    if not text:
        return 0.0
    sents = [s.strip() for s in re.split(r"[.!?\n]+", text) if s.strip()]
    if len(sents) <= 1:
        return 0.0
    norm = [re.sub(r"\s+", "", s) for s in sents]
    uniq = len(set(norm))
    return float(1.0 - (uniq / max(1, len(norm))))

def ngram_repeat_ratio(text: str, n: int = 4) -> float:
    if not text:
        return 0.0
    toks = re.findall(r"[가-힣]{1,}|[0-9]+|[A-Za-z]+", text)
    if len(toks) < n * 2:
        return 0.0
    ngrams = [" ".join(toks[i:i+n]) for i in range(len(toks)-n+1)]
    total = len(ngrams)
    uniq = len(set(ngrams))
    return float(1.0 - (uniq / max(1, total)))

def has_weird_korean(text: str) -> bool:
    if not text:
        return True
    if re.search(r"[ㄱ-ㅎㅏ-ㅣ]{2,}", text):
        return True
    if re.search(r"(.)\1\1\1", text):
        return True
    return False

def structural_ok(text: str) -> bool:
    if not text:
        return False
    if has_weird_korean(text):
        return False
    if has_unfinished_tail(text):
        return False
    if exact_duplicate_sentence_ratio(text) >= 0.25:
        return False
    if ngram_repeat_ratio(text, n=4) >= 0.16:
        return False
    if count_cliches(text) >= 2:
        return False
    return True

def failure_reasons(text: str) -> List[str]:
    reasons = []
    if not text:
        return ["empty"]
    if has_weird_korean(text):
        reasons.append("weird_korean")
    if has_unfinished_tail(text):
        reasons.append("unfinished_tail")
    if exact_duplicate_sentence_ratio(text) >= 0.25:
        reasons.append("dup_sentence")
    if ngram_repeat_ratio(text, n=4) >= 0.16:
        reasons.append("ngram4_repeat")
    if count_cliches(text) >= 2:
        reasons.append("cliche>=2")
    if not reasons:
        reasons.append("ok")
    return reasons

# ----------------------------
# (3) 라운드 파싱 + 통계
# ----------------------------
def parse_round_from_path(path: str) -> int:
    # .../train_round4_for_finetune_0000.jsonl -> 4
    m = re.search(r"train_round(\d+)_for_finetune_", os.path.basename(path))
    return int(m.group(1)) if m else -1

paths = sorted(glob.glob(TRAIN_GLOB))
if not paths:
    raise FileNotFoundError(f"No train shards found: {TRAIN_GLOB}")

round_to_paths = defaultdict(list)
for p in paths:
    r = parse_round_from_path(p)
    round_to_paths[r].append(p)

print("[FOUND ROUNDS]", sorted(round_to_paths.keys()))

round_stats = {}
for r in sorted(round_to_paths.keys()):
    total = 0
    ok = 0
    reason_cnt = Counter()
    sent_cnts = []
    cliche_cnts = []
    dup_ratios = []
    ngram4s = []

    for p in round_to_paths[r]:
        rows = load_jsonl(p)
        for row in rows:
            # row 형태: {"src":..., "tgt":...} 로 저장했을 가능성이 큼
            tgt = row.get("tgt") or row.get("output", {}).get("daily_diary") or ""
            total += 1

            for rsn in failure_reasons(tgt):
                reason_cnt[rsn] += 1

            if structural_ok(tgt):
                ok += 1

            sent_cnts.append(sentence_count_heuristic(tgt))
            cliche_cnts.append(count_cliches(tgt))
            dup_ratios.append(exact_duplicate_sentence_ratio(tgt))
            ngram4s.append(ngram_repeat_ratio(tgt, n=4))

    pass_rate = ok / max(1, total)
    round_stats[r] = (total, ok, pass_rate, reason_cnt)

    print(f"\n[ROUND {r}] total={total} ok={ok} pass_rate={pass_rate:.3f}")
    print("  top_fail_reasons:", reason_cnt.most_common(8))
    # 분포를 대충 보기 위한 요약
    def pct_ge(vals, thr):
        return sum(1 for v in vals if v >= thr) / max(1, len(vals))
    print(f"  sent_cnt: avg={sum(sent_cnts)/max(1,len(sent_cnts)):.2f}  "
          f">=12={pct_ge(sent_cnts,12):.2%}  <=4={sum(1 for v in sent_cnts if v<=4)/max(1,len(sent_cnts)):.2%}")
    print(f"  cliche_cnt>=2: {sum(1 for v in cliche_cnts if v>=2)/max(1,len(cliche_cnts)):.2%}")
    print(f"  dup_sent_ratio>=0.25: {sum(1 for v in dup_ratios if v>=0.25)/max(1,len(dup_ratios)):.2%}")
    print(f"  ngram4_rep>=0.16: {sum(1 for v in ngram4s if v>=0.16)/max(1,len(ngram4s)):.2%}")

# ----------------------------
# (4) (선택) replay 버퍼 만들기: ok만 모아서 라운드별 가중 샘플링
# ----------------------------
def build_replay_buffer(
    round_to_paths: Dict[int, List[str]],
    max_per_round: int = 5000,
    round_weights: Dict[int, float] = None,
) -> List[Dict[str, Any]]:
    """
    - ok 샘플만 모아서 라운드별로 max_per_round 제한
    - round_weights가 있으면 그 비율대로 최종 mix
    """
    per_round_ok = {}
    for r, ps in round_to_paths.items():
        ok_rows = []
        for p in ps:
            for row in load_jsonl(p):
                tgt = row.get("tgt") or row.get("output", {}).get("daily_diary") or ""
                if structural_ok(tgt):
                    ok_rows.append(row)
        random.shuffle(ok_rows)
        per_round_ok[r] = ok_rows[:max_per_round]

    # weights 없으면 최근 라운드 우선(단순)
    if round_weights is None:
        # 예: round4:0.5, round3:0.3, round2:0.2 같은 느낌
        rounds = sorted(per_round_ok.keys())
        if rounds:
            recent = max(rounds)
            round_weights = {r: 0.0 for r in rounds}
            for r in rounds:
                if r == recent:
                    round_weights[r] = 0.5
                elif r == recent - 1:
                    round_weights[r] = 0.3
                else:
                    round_weights[r] += 0.2 / max(1, (len(rounds)-2))

    # normalize weights
    s = sum(round_weights.values()) if round_weights else 1.0
    round_weights = {k: v/s for k, v in round_weights.items()}

    # 최종 섞기
    total_ok = sum(len(v) for v in per_round_ok.values())
    if total_ok == 0:
        print("[REPLAY] No ok samples found.")
        return []

    # 기본적으로 전체 ok를 다 쓰지 말고, 원하는 크기만 뽑는 게 안전
    # 여기서는 예시로 total_ok의 20%만 replay로 만든다.
    target_size = int(total_ok * 0.20)

    replay = []
    for r, rows in per_round_ok.items():
        take = int(target_size * round_weights.get(r, 0.0))
        if take <= 0:
            continue
        replay.extend(rows[:take])

    random.shuffle(replay)
    print(f"\n[REPLAY] built: {len(replay)} samples (target={target_size})")
    for r in sorted(per_round_ok.keys()):
        print(f"  round{r}: ok_pool={len(per_round_ok[r])} weight={round_weights.get(r,0):.3f}")
    return replay

# 사용 예시:
replay = build_replay_buffer(round_to_paths, max_per_round=6000)

# replay를 라운드5 학습에 섞을 때는 보통:
# train_records = anchors_all + train_for_finetune + replay
# 단, replay 비중은 전체 train_records의 10~20% 정도로 유지 추천

[FOUND ROUNDS] [2, 3, 4]

[ROUND 2] total=4692 ok=1813 pass_rate=0.386
  top_fail_reasons: [('cliche>=2', 2668), ('unfinished_tail', 2551), ('ok', 1813), ('ngram4_repeat', 1078), ('dup_sentence', 86)]
  sent_cnt: avg=9.54  >=12=34.04%  <=4=17.67%
  cliche_cnt>=2: 56.86%
  dup_sent_ratio>=0.25: 1.83%
  ngram4_rep>=0.16: 22.98%

[ROUND 3] total=4532 ok=1557 pass_rate=0.344
  top_fail_reasons: [('cliche>=2', 2827), ('unfinished_tail', 2714), ('ngram4_repeat', 2168), ('ok', 1557), ('dup_sentence', 598)]
  sent_cnt: avg=10.33  >=12=36.47%  <=4=6.71%
  cliche_cnt>=2: 62.38%
  dup_sent_ratio>=0.25: 13.20%
  ngram4_rep>=0.16: 47.84%

[ROUND 4] total=4529 ok=1566 pass_rate=0.346
  top_fail_reasons: [('cliche>=2', 2833), ('unfinished_tail', 2752), ('ngram4_repeat', 2433), ('ok', 1566), ('dup_sentence', 1058)]
  sent_cnt: avg=10.34  >=12=36.12%  <=4=7.13%
  cliche_cnt>=2: 62.55%
  dup_sent_ratio>=0.25: 23.36%
  ngram4_rep>=0.16: 53.72%

[REPLAY] built: 986 samples (target=987)
  round2: ok_pool=1

In [5]:
# ============================================================
# 체크 필요한 것들 한 번에 보는 분석 코드 (전체 수정본)
# - TRAIN / MERGED_FULL / STUDENT_GEN / GPT_REWRITE 포맷이 섞여도 자동으로 tgt/src 추출
# - stage/meta/input/output 구조(네 샘플) 지원
# - src/tgt 구조 지원
# - 반복 문장 TOP-N, 입력 노이즈, novelty 상위 샘플 출력 포함
# ============================================================

import os, glob, json, random, re
from collections import Counter, defaultdict
from typing import Dict, Any, List, Tuple

SEED = 777
random.seed(SEED)

OUT_DIR = "generated/aiary_v3_hybrid_20260220_014739"

# (A) 학습 shard (보통 train_round*_for_finetune_*.jsonl)
TRAIN_GLOB = os.path.join(OUT_DIR, "train_round*_for_finetune_*.jsonl")

# (B) merged_full shard(있으면 stage/meta/input/output 포함)
MERGED_FULL_GLOB = os.path.join(OUT_DIR, "merged_full_round*_*.jsonl")

# (C) student_gen / gpt_rewritten shard
STUDENT_GEN_GLOB = os.path.join(OUT_DIR, "round*_student_gen_*.jsonl")
GPT_REWRITE_GLOB = os.path.join(OUT_DIR, "round*_gpt_rewritten_*.jsonl")

# ----------------------------
# JSONL 로더
# ----------------------------
def load_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def parse_round_from_name(name: str) -> int:
    m = re.search(r"round(\d+)", name)
    return int(m.group(1)) if m else -1

# ============================================================
# ✅ 포맷 자동 감지: src/joined + tgt/daily_diary 추출
# ============================================================
def extract_joined_and_diary(row: Any) -> Tuple[str, str]:
    joined = ""
    diary = ""

    if not isinstance(row, dict):
        return joined, diary

    # 1) joined/src 추출
    if isinstance(row.get("input"), dict):
        joined = row["input"].get("joined", "") or ""

    if not joined:
        # train shard 후보들
        for k in ["src", "source", "prompt", "input_text", "text", "x", "joined"]:
            v = row.get(k, "")
            if isinstance(v, str) and v.strip():
                joined = v
                break

    # 2) diary/tgt 추출
    if isinstance(row.get("output"), dict):
        diary = row["output"].get("daily_diary", "") or ""

    if not diary:
        for k in ["tgt", "target", "daily_diary", "label", "y", "completion", "response", "output_text", "text"]:
            v = row.get(k, "")
            if isinstance(v, str) and v.strip():
                diary = v
                break

    return joined.strip(), diary.strip()

def extract_stage(row: Any) -> str:
    if isinstance(row, dict):
        return row.get("stage", "UNKNOWN")
    return "UNKNOWN"

def extract_novelty(row: Any):
    if isinstance(row, dict):
        meta = row.get("meta", {})
        if isinstance(meta, dict):
            return meta.get("novelty", None)
    return None

def extract_one_lines(row: Any) -> List[str]:
    if isinstance(row, dict):
        inp = row.get("input", {})
        if isinstance(inp, dict):
            ol = inp.get("one_lines", [])
            if isinstance(ol, list):
                return [str(x) for x in ol]
    return []

# ============================================================
# 구조 필터(라운드5에서 쓰려는 기준)
# ============================================================
_CLICHE_PATTERNS = [
    r"오늘 하루.*소중",
    r"소중하게 다가왔",
    r"마음(이 )?놓였",
    r"안도(하|한|하는)",
    r"세심하게 살펴야겠",
    r"생각(이 )?들었",
    r"평범한 하루",
    r"마음(이 )?따뜻",
    r"작은 변화",
    r"눈에 들어온 날",
]

def sentence_count_heuristic(text: str) -> int:
    if not text:
        return 0
    parts = [p.strip() for p in re.split(r"[.!?\n]+", text) if p.strip()]
    return len(parts)

def has_unfinished_tail(text: str) -> bool:
    if not text:
        return True
    t = text.strip()
    if re.search(r"[.!?。！？]\s*$", t):
        return False
    if re.search(r"(다|요|죠|음)\s*$", t):
        return False
    return True

def count_cliches(text: str) -> int:
    if not text:
        return 0
    c = 0
    for p in _CLICHE_PATTERNS:
        if re.search(p, text):
            c += 1
    return c

def exact_duplicate_sentence_ratio(text: str) -> float:
    if not text:
        return 0.0
    sents = [s.strip() for s in re.split(r"[.!?\n]+", text) if s.strip()]
    if len(sents) <= 1:
        return 0.0
    norm = [re.sub(r"\s+", "", s) for s in sents]
    uniq = len(set(norm))
    return float(1.0 - (uniq / max(1, len(norm))))

def ngram_repeat_ratio(text: str, n: int = 4) -> float:
    if not text:
        return 0.0
    toks = re.findall(r"[가-힣]{1,}|[0-9]+|[A-Za-z]+", text)
    if len(toks) < n * 2:
        return 0.0
    ngrams = [" ".join(toks[i:i+n]) for i in range(len(toks)-n+1)]
    total = len(ngrams)
    uniq = len(set(ngrams))
    return float(1.0 - (uniq / max(1, total)))

def has_weird_korean(text: str) -> bool:
    if not text:
        return True
    if re.search(r"[ㄱ-ㅎㅏ-ㅣ]{2,}", text):
        return True
    if re.search(r"(.)\1\1\1", text):
        return True
    return False

def failure_reasons(text: str) -> List[str]:
    reasons = []
    if not text:
        return ["empty"]
    if has_weird_korean(text):
        reasons.append("weird_korean")
    if has_unfinished_tail(text):
        reasons.append("unfinished_tail")
    if exact_duplicate_sentence_ratio(text) >= 0.25:
        reasons.append("dup_sentence")
    if ngram_repeat_ratio(text, n=4) >= 0.16:
        reasons.append("ngram4_repeat")
    if count_cliches(text) >= 2:
        reasons.append("cliche>=2")
    if not reasons:
        reasons.append("ok")
    return reasons

# ============================================================
# (1) 라운드별 요약 통계 함수
# ============================================================
def summarize_records(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    total = 0
    ok = 0
    reason_cnt = Counter()

    sent_cnts = []
    cliche_cnts = []
    dup_ratios = []
    ngram4s = []
    unfinished = 0
    end_char_cnt = Counter()

    for r in records:
        _, tgt = extract_joined_and_diary(r)
        tgt = (tgt or "").strip()
        total += 1

        rs = failure_reasons(tgt)
        for x in rs:
            reason_cnt[x] += 1
        if "ok" in rs:
            ok += 1

        sc = sentence_count_heuristic(tgt)
        sent_cnts.append(sc)
        c = count_cliches(tgt)
        cliche_cnts.append(c)
        d = exact_duplicate_sentence_ratio(tgt)
        dup_ratios.append(d)
        ng = ngram_repeat_ratio(tgt, 4)
        ngram4s.append(ng)

        if has_unfinished_tail(tgt):
            unfinished += 1

        tail = tgt[-6:] if len(tgt) >= 6 else tgt
        end_char_cnt[tail] += 1

    def pct_ge(vals, thr):
        return sum(1 for v in vals if v >= thr) / max(1, len(vals))

    out = {
        "total": total,
        "ok": ok,
        "pass_rate": ok / max(1, total),
        "reasons": reason_cnt,
        "unfinished_rate": unfinished / max(1, total),
        "sent_avg": sum(sent_cnts) / max(1, len(sent_cnts)),
        "sent_ge12": pct_ge(sent_cnts, 12),
        "sent_le4": sum(1 for v in sent_cnts if v <= 4) / max(1, len(sent_cnts)),
        "cliche_ge2": sum(1 for v in cliche_cnts if v >= 2) / max(1, len(cliche_cnts)),
        "dup_ge025": sum(1 for v in dup_ratios if v >= 0.25) / max(1, len(dup_ratios)),
        "ngram4_ge016": sum(1 for v in ngram4s if v >= 0.16) / max(1, len(ngram4s)),
        "tail_top": end_char_cnt.most_common(10),
    }
    return out

def print_summary(tag: str, summ: Dict[str, Any]):
    print(f"\n[{tag}] total={summ['total']} ok={summ['ok']} pass_rate={summ['pass_rate']:.3f} unfinished={summ['unfinished_rate']:.2%}")
    print("  top_fail_reasons:", summ["reasons"].most_common(8))
    print(f"  sent_avg={summ['sent_avg']:.2f}  >=12={summ['sent_ge12']:.2%}  <=4={summ['sent_le4']:.2%}")
    print(f"  cliche>=2={summ['cliche_ge2']:.2%}  dup>=0.25={summ['dup_ge025']:.2%}  ngram4>=0.16={summ['ngram4_ge016']:.2%}")
    print("  tail_top10:", summ["tail_top"])

# ============================================================
# (2) 반복 문장 TOP-N
# ============================================================
def extract_dup_sentences(text: str) -> List[str]:
    sents = [s.strip() for s in re.split(r"[.!?\n]+", text) if s.strip()]
    norm = [re.sub(r"\s+", "", s) for s in sents]
    cnt = Counter(norm)
    norm_to_orig = {}
    for s, n in zip(sents, norm):
        if n not in norm_to_orig:
            norm_to_orig[n] = s
    dups = [(norm_to_orig[k], v) for k, v in cnt.items() if v >= 2]
    dups.sort(key=lambda x: x[1], reverse=True)
    return [f"{s}  (x{c})" for s, c in dups[:10]]

def collect_top_dup_sentences(records: List[Dict[str, Any]], max_samples: int = 3000) -> List[Tuple[str,int]]:
    cnt = Counter()
    seen = 0
    for r in records:
        _, tgt = extract_joined_and_diary(r)
        tgt = (tgt or "").strip()
        if not tgt:
            continue
        if exact_duplicate_sentence_ratio(tgt) < 0.25:
            continue
        for item in extract_dup_sentences(tgt):
            key = re.sub(r"\s+\(x\d+\)$", "", item)
            cnt[key] += 1
        seen += 1
        if seen >= max_samples:
            break
    return cnt.most_common(30)

# ============================================================
# (3) 입력 노이즈
# ============================================================
_NOISE_PATTERNS = {
    "eseo_eseo": r"에서에서",
    "itda_hadaga": r"있다하다가",
    "eulreul_raw": r"을\(를\)",
    "double_space": r"\s{2,}",
    "dangling_comma": r",\s*[.?!]?$",
}

def input_noise_stats(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    total = 0
    pat_cnt = Counter()
    for r in records:
        joined, _ = extract_joined_and_diary(r)
        total += 1
        for k, p in _NOISE_PATTERNS.items():
            if re.search(p, joined):
                pat_cnt[k] += 1
    rates = {k: v / max(1, total) for k, v in pat_cnt.items()}
    return {"total": total, "counts": pat_cnt, "rates": rates}

def print_noise(tag: str, st: Dict[str, Any]):
    print(f"\n[INPUT NOISE {tag}] total={st['total']}")
    for k, v in st["rates"].items():
        print(f"  {k}: {v:.2%} (n={st['counts'][k]})")

# ============================================================
# (4) novelty 상위 샘플
# ============================================================
def print_top_novelty(records: List[Dict[str, Any]], topk: int = 5):
    cand = []
    for r in records:
        nov = extract_novelty(r)
        if nov is None:
            continue
        cand.append((nov, r))
    cand.sort(key=lambda x: x[0], reverse=True)
    print(f"\n[TOP NOVELTY] showing {min(topk, len(cand))}")
    for i, (nov, r) in enumerate(cand[:topk], 1):
        print(f"\n--- #{i} novelty={nov:.3f} id={r.get('id')} stage={extract_stage(r)}")
        one_lines = extract_one_lines(r)
        if one_lines:
            print("[one_lines]")
            for x in one_lines:
                print(" -", x)
        _, tgt = extract_joined_and_diary(r)
        print("[tgt]")
        print((tgt or "")[:600])

# ============================================================
# 실행
# ============================================================

def load_grouped_by_round(paths: List[str]) -> Dict[int, List[str]]:
    m = defaultdict(list)
    for p in paths:
        r = parse_round_from_name(os.path.basename(p))
        m[r].append(p)
    return dict(m)

def load_all(paths: List[str]) -> List[Dict[str, Any]]:
    out = []
    for p in paths:
        out.extend(list(load_jsonl(p)))
    return out

# 1) TRAIN
train_paths = sorted(glob.glob(TRAIN_GLOB))
if not train_paths:
    print("[WARN] No train shards found:", TRAIN_GLOB)
else:
    round_to_train = load_grouped_by_round(train_paths)
    print("\n[TRAIN FOUND ROUNDS]", sorted(round_to_train.keys()))
    for r in sorted(round_to_train.keys()):
        records = load_all(round_to_train[r])
        summ = summarize_records(records)
        print_summary(f"TRAIN round{r}", summ)

        top_dup = collect_top_dup_sentences(records)
        if top_dup:
            print("\n  [TOP DUP SENTENCES]")
            for s, c in top_dup[:10]:
                print(f"   - ({c}) {s}")

# 2) MERGED_FULL
merged_paths = sorted(glob.glob(MERGED_FULL_GLOB))
if not merged_paths:
    print("\n[INFO] merged_full shards not found:", MERGED_FULL_GLOB)
else:
    round_to_merged = load_grouped_by_round(merged_paths)
    print("\n[MERGED_FULL FOUND ROUNDS]", sorted(round_to_merged.keys()))
    for r in sorted(round_to_merged.keys()):
        merged = load_all(round_to_merged[r])

        summ_all = summarize_records(merged)
        print_summary(f"MERGED_FULL round{r} ALL", summ_all)

        # stage별
        by_stage = defaultdict(list)
        for rec in merged:
            by_stage[extract_stage(rec)].append(rec)

        for stg in sorted(by_stage.keys()):
            if ("student_gen" not in stg) and ("gpt_rewrite" not in stg):
                continue
            s = summarize_records(by_stage[stg])
            print_summary(f"MERGED_FULL round{r} {stg}", s)

        # 입력 노이즈
        noise = input_noise_stats(merged)
        print_noise(f"round{r}", noise)

        # novelty 상위
        print_top_novelty(merged, topk=5)

# 3) Separate stage shards (옵션)
student_paths = sorted(glob.glob(STUDENT_GEN_GLOB))
gpt_paths = sorted(glob.glob(GPT_REWRITE_GLOB))
if student_paths or gpt_paths:
    print("\n[INFO] Separate stage shards detected. (Optional deeper check)")

    if student_paths:
        rmap = load_grouped_by_round(student_paths)
        for r in sorted(rmap.keys()):
            recs = load_all(rmap[r])
            s = summarize_records(recs)
            print_summary(f"STUDENT_GEN round{r}", s)
            noise = input_noise_stats(recs)
            print_noise(f"STUDENT_GEN round{r}", noise)

    if gpt_paths:
        rmap = load_grouped_by_round(gpt_paths)
        for r in sorted(rmap.keys()):
            recs = load_all(rmap[r])
            s = summarize_records(recs)
            print_summary(f"GPT_REWRITE round{r}", s)


[TRAIN FOUND ROUNDS] [2, 3, 4]

[TRAIN round2] total=4692 ok=1813 pass_rate=0.386 unfinished=54.37%
  top_fail_reasons: [('cliche>=2', 2668), ('unfinished_tail', 2551), ('ok', 1813), ('ngram4_repeat', 1078), ('dup_sentence', 86)]
  sent_avg=9.54  >=12=34.04%  <=4=17.67%
  cliche>=2=56.86%  dup>=0.25=1.83%  ngram4>=0.16=22.98%
  tail_top10: [('상적이었다.', 186), ('이 들었다.', 155), (' 느껴졌다.', 136), (' 하루였다.', 121), (' 오늘 하루', 82), ('다. 아이가', 77), ('고 있었다.', 63), ('정과 행동을', 63), (' 다가왔다.', 56), ('수 있었다.', 55)]

  [TOP DUP SENTENCES]
   - (29) 오늘 하루 아이와 함께한 소소한 순간들이 소중하게 다가왔다
   - (14) 오늘 하루 아이와 함께한 시간이 소중하게 느껴졌다
   - (12) 앞으로도 아이와 함께한 시간이 소중하게 다가왔다
   - (6) 앞으로도 아이와 함께한 시간이 소중하게 느껴졌다
   - (6) 오늘 하루도 아이와 함께한 시간이 소중하게 느껴졌다
   - (5) 오늘 하루 아이와 함께한 소소한 순간들이 소중하게 느껴졌다
   - (5) 오늘 하루 아이의 다양한 감정과 행동을 지켜보며 조금 더 세심하게 살펴야겠다는 생각이 들었다
   - (4) 오늘 하루 아이의 다양한 감정과 행동을 보며 조금 더 세심하게 살펴야겠다는 생각이 들었다
   - (4) 아이가 다양한 감정을 표현하는 모습을 보며 조금 더 세심하게 살펴야겠다는 생각이 들었다
   - (4) 아이와 함께한 시간이 소중하게 느껴졌다

[TRAIN round3] total=4532

In [5]:
# ============================================================
# [CELL] Round 5 운영 (GPT 정화 중심, EXAONE 제거 버전)
# - Student 생성(10k)
# - 생성 결과 postprocess + hard filter
# - hard filter 통과분: keep_pass -> 학습에 포함
# - hard filter 불통과분: GPT rewrite -> 학습에 포함
# - anchors + (keep_pass + gpt_rewrite)로 다음 student 학습
# ============================================================

import os, time, random, re, math, json, glob
from typing import List, Dict, Any, Tuple

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

# ----------------------------
# 라운드/설정
# ----------------------------
ROUND_IDX = 5
STUDENT_GEN_TARGET = 10000

# ✅ Round5는 비율이 아니라 "불합격 전량 GPT 정화"가 핵심
# 다만 비용/속도 때문에 상한을 둠(필요 시 조절)
GPT_REWRITE_MAX_RATIO = 0.60     # 불합격이 너무 많을 때 최대 60%까지만 교정
GPT_REWRITE_MIN_RATIO = 0.20     # 불합격이 적어도 최소 20%는 교정(분포 유지)

OUT_DIR = "generated/aiary_v3_hybrid_20260220_014739"

# ✅ Round5 시작 시: CURRENT는 round4 모델이어야 함
CURRENT_STUDENT_PATH = f"{OUT_DIR}/models/kobart_student_round4"
NEXT_STUDENT_SAVE_DIR = f"{OUT_DIR}/models/kobart_student_round{ROUND_IDX}"
os.makedirs(NEXT_STUDENT_SAVE_DIR, exist_ok=True)

SEED = 777
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------------------
# Teacher(GPT)
# ----------------------------
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)
TEACHER_MODEL = "gpt-4.1-mini"

def call_teacher(system: str, user: str, temperature: float, max_output_tokens: int) -> str:
    backoff = 1.5
    last_err = None
    for attempt in range(1, 9):
        try:
            resp = client.responses.create(
                model=TEACHER_MODEL,
                input=[{"role":"system","content":system},{"role":"user","content":user}],
                temperature=temperature,
                max_output_tokens=max_output_tokens,
            )
            text = getattr(resp, "output_text", None)
            return str(text if text is not None else resp).strip()
        except Exception as e:
            last_err = e
            time.sleep(min(20, backoff ** attempt) + random.random())
    raise RuntimeError(f"Teacher call failed after retries: {last_err}")

# ============================================================
# (0) 입력 joined 노이즈 정규화
# ============================================================
def normalize_joined(s: str) -> str:
    if not s:
        return ""
    s = s.strip()
    s = s.replace("에서에서", "에서")
    s = s.replace("있다하다가", "있다가")
    s = s.replace("해본다하더니", "해보더니")
    s = re.sub(r"\s*을\(를\)\s*", " ", s)
    s = re.sub(r"[ \t]{2,}", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

# ============================================================
# (A) 구조 필터(네 check.py 기준 강화판)
# ============================================================
_CLICHE_PATTERNS = [
    r"오늘 하루.*소중",
    r"소중하게 다가왔",
    r"마음(이 )?놓였",
    r"조금 안도",
    r"세심하게 살펴야겠",
    r"생각(이 )?들었",
    r"평범한 하루",
    r"마음(이 )?따뜻",
    r"작은 변화",
    r"눈에 들어온 날",
    r"교훈을 남겼",
    r"앞으로는 .*해야겠",
]

def sentence_count_heuristic(text: str) -> int:
    if not text:
        return 0
    parts = [p.strip() for p in re.split(r"[.!?\n]+", text) if p.strip()]
    return len(parts)

def has_unfinished_tail(text: str) -> bool:
    if not text:
        return True
    t = text.strip()
    if re.search(r"[.!?。！？]\s*$", t):
        return False
    if re.search(r"(다|요|죠|음)\s*$", t):
        return False
    return True

def count_cliches(text: str) -> int:
    if not text:
        return 0
    c = 0
    for p in _CLICHE_PATTERNS:
        if re.search(p, text):
            c += 1
    return c

def exact_duplicate_sentence_ratio(text: str) -> float:
    if not text:
        return 0.0
    sents = [s.strip() for s in re.split(r"[.!?\n]+", text) if s.strip()]
    if len(sents) <= 1:
        return 0.0
    norm = [re.sub(r"\s+", "", s) for s in sents]
    uniq = len(set(norm))
    return float(1.0 - (uniq / max(1, len(norm))))

def ngram_repeat_ratio(text: str, n: int = 4) -> float:
    if not text:
        return 0.0
    toks = re.findall(r"[가-힣]{1,}|[0-9]+|[A-Za-z]+", text)
    if len(toks) < n * 2:
        return 0.0
    ngrams = [" ".join(toks[i:i+n]) for i in range(len(toks)-n+1)]
    total = len(ngrams)
    uniq = len(set(ngrams))
    return float(1.0 - (uniq / max(1, total)))

def has_weird_korean(text: str) -> bool:
    if not text:
        return True
    if re.search(r"[ㄱ-ㅎㅏ-ㅣ]{2,}", text):
        return True
    if re.search(r"(.)\1\1\1", text):
        return True
    return False

# ✅ 하드 필터: student_keep는 이것만 통과해야 학습에 들어감
def pass_hard_filter(text: str) -> bool:
    if not text:
        return False
    if has_weird_korean(text):
        return False
    if has_unfinished_tail(text):
        return False
    if count_cliches(text) >= 2:
        return False
    if exact_duplicate_sentence_ratio(text) >= 0.20:
        return False
    if ngram_repeat_ratio(text, 4) >= 0.12:
        return False
    sc = sentence_count_heuristic(text)
    if sc <= 3 or sc >= 12:
        return False
    return True

# ============================================================
# (B) postprocess: 끝 미완성/상투문구 연타 컷
# ============================================================
_CUT_CLICHE = [
    "오늘 하루 아이와 함께한 소소한 순간들이",
    "조금 더 세심하게 살펴야겠다는 생각이 들었다",
    "조금 안도하는 마음이 들었다",
]

def trim_to_last_complete_sentence(text: str) -> str:
    if not text:
        return ""
    t = text.strip()

    # 이미 종결부호면 ok
    if re.search(r"[.!?。！？]\s*$", t):
        return t

    # 마지막 종결부호 기준 컷
    last = max(t.rfind("."), t.rfind("!"), t.rfind("?"), t.rfind("。"), t.rfind("！"), t.rfind("？"))
    if last != -1 and last >= 20:
        t = t[: last + 1].strip()

    # 그래도 종결부호가 없으면 문장 하나 마침표 추가(너무 짧으면 그대로)
    if len(t) >= 30 and not re.search(r"[.!?。！？]\s*$", t):
        t = t + "."
    return t.strip()

def drop_trailing_cliche_repeats(text: str, max_occ: int = 1) -> str:
    if not text:
        return ""
    t = text
    for p in _CUT_CLICHE:
        occ = [m.start() for m in re.finditer(re.escape(p), t)]
        if len(occ) > max_occ:
            cut_at = occ[max_occ]
            t = t[:cut_at].strip()
    return t.strip()

def postprocess_diary(text: str) -> str:
    t = (text or "").strip()
    t = trim_to_last_complete_sentence(t)
    t = drop_trailing_cliche_repeats(t, max_occ=1)
    t = trim_to_last_complete_sentence(t)
    return t.strip()

# ============================================================
# (C) coverage / novelty / repetition (기존 유지, 가볍게만)
# ============================================================
_STOPWORDS = set(["오늘","아침","오전","점심","오후","저녁","밤","아이","아기","엄마","아빠","보호자"])
_CONJ = ["그리고", "그래서", "하지만", "그러다", "그러고", "또", "또는", "또한"]

def _ko_words(s: str) -> List[str]:
    return re.findall(r"[가-힣]{2,}", s)

def _keywords_from_line(line: str, max_k: int = 8) -> List[str]:
    words = [w for w in _ko_words(line) if w not in _STOPWORDS]
    out, seen = [], set()
    for w in words:
        if w in seen:
            continue
        seen.add(w)
        out.append(w)
        if len(out) >= max_k:
            break
    return out

def coverage_score(one_lines: List[str], diary: str) -> float:
    if not one_lines:
        return 0.0
    hit = 0
    for line in one_lines:
        kws = _keywords_from_line(line, max_k=8)
        if not kws:
            hit += 1 if any(tok in diary for tok in _ko_words(line)[:3]) else 0
            continue
        if any(k in diary for k in kws):
            hit += 1
    return hit / max(1, len(one_lines))

def novelty_score(one_lines: List[str], diary: str) -> float:
    in_words = set()
    for l in one_lines:
        in_words.update(_keywords_from_line(l, max_k=999))
    out_words = [w for w in _ko_words(diary) if w not in _STOPWORDS]
    if not out_words:
        return 0.0
    out_set = set(out_words)
    novel = [w for w in out_set if w not in in_words]
    return len(novel) / max(1, len(out_set))

def repetition_score(diary: str) -> float:
    if not diary:
        return 1.0
    conj_cnt = sum(diary.count(c) for c in _CONJ)
    sents = [s.strip() for s in re.split(r"[.!?\n]+", diary) if s.strip()]
    starts = [s[:4] for s in sents if len(s) >= 4]

    start_rep = 0.0
    if starts:
        counts = {}
        for st in starts:
            counts[st] = counts.get(st, 0) + 1
        maxc = max(counts.values())
        start_rep = max(0, (maxc - 1) / max(1, len(starts) - 1))

    words = [w for w in _ko_words(diary) if w not in _STOPWORDS]
    uniq_ratio = (len(set(words)) / max(1, len(words))) if words else 1.0
    word_rep = 1.0 - uniq_ratio
    conj_norm = min(1.0, conj_cnt / 10.0)
    return float(min(1.0, 0.45 * word_rep + 0.35 * start_rep + 0.20 * conj_norm))

# ============================================================
# (D) GPT 프롬프트(강화: 상투문구/교훈/미래다짐 금지 + 입력만 반영)
# ============================================================
def build_gpt_rewrite_prompt(one_lines: List[str], length_tag: str) -> Tuple[str, str, float, int]:
    if length_tag == "SHORT":
        guideline, max_tokens, temp = "3~5문장", 260, 0.45
    elif length_tag == "MEDIUM":
        guideline, max_tokens, temp = "6~8문장", 420, 0.50
    else:
        guideline, max_tokens, temp = "9~11문장", 560, 0.55

    system = (
        "너는 한국어 육아 하루일기 편집자다. "
        "입력 한 줄 기록에 있는 정보만 사용한다(새 장소/사물/행동/사건 추가 금지). "
        "문장 반복 금지, 상투문구/교훈/미래 다짐/시적 표현 금지. "
        "감정 문장은 1~2문장만 허용하고, 관찰 기반으로 짧게 쓴다. "
        "마지막 문장은 반드시 종결부호로 끝낸다."
    )
    user = (
        f"아래 한 줄 기록을 바탕으로 하루일기를 재작성하라.\n"
        f"- 문장 수: {guideline}\n"
        f"- 조건: 각 한 줄 내용을 최소 1회 이상 구체적으로 반영\n"
        f"- 금지: '오늘 하루 소중', '세심하게', '안도', '교훈', '앞으로도' 같은 마무리 템플릿\n\n"
        "[한 줄 기록]\n" + "\n".join([f"- {x}" for x in one_lines])
    )
    return system, user, temp, max_tokens

# ============================================================
# (E) student(KoBART) 로드 + 생성(반복 억제 + 후처리)
# ============================================================
student_tok = AutoTokenizer.from_pretrained(CURRENT_STUDENT_PATH, use_fast=True)
student_model = AutoModelForSeq2SeqLM.from_pretrained(CURRENT_STUDENT_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"
student_model = student_model.to(device)
student_model.eval()

def gen_student_daily(one_lines: List[str], length_tag: str) -> str:
    # 네 기존 safe_join_one_lines 사용
    src = normalize_joined(safe_join_one_lines(one_lines))

    if length_tag == "SHORT":
        max_new, min_new = 140, 60
    elif length_tag == "MEDIUM":
        max_new, min_new = 220, 110
    else:
        max_new, min_new = 300, 160

    enc = student_tok(src, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(device) for k, v in enc.items()}
    enc.pop("token_type_ids", None)

    with torch.no_grad():
        out = student_model.generate(
            **enc,
            max_new_tokens=max_new,
            min_new_tokens=min_new,
            num_beams=4,
            do_sample=False,
            early_stopping=True,
            no_repeat_ngram_size=4,
            repetition_penalty=1.12,
            eos_token_id=student_tok.eos_token_id,
            pad_token_id=student_tok.pad_token_id,
        )

    raw = normalize_space(student_tok.decode(out[0], skip_special_tokens=True))
    return postprocess_diary(raw)

# ============================================================
# (F) student 초안 생성 (네 기존 generator 함수들 사용)
# ============================================================
def generate_student_batch(target_n: int) -> List[Dict[str, Any]]:
    created = 0
    attempts = 0
    records = []
    while created < target_n and attempts < MAX_ATTEMPTS:
        attempts += 1

        day_situations = sample_day_situations(by_age)
        if not validate_day_diversity(day_situations):
            continue

        one_lines, one_lines_trace = [], []
        for s in day_situations:
            ol, tr = render_one_line_with_trace(s, state_surface)
            one_lines.append(ol)
            one_lines_trace.append(tr)

        one_lines, one_lines_trace = perturb_one_lines(one_lines, one_lines_trace)
        if not validate_one_lines(one_lines):
            continue

        length_tag = weighted_choice(DAILY_LENGTH_DIST)
        diary = gen_student_daily(one_lines, length_tag)

        cov = coverage_score(one_lines, diary)
        nov = novelty_score(one_lines, diary)
        rep = repetition_score(diary)

        age_group = day_situations[0].age_group if day_situations else ""
        rec = {
            "id": f"r{ROUND_IDX}_student_{created:06d}",
            "stage": f"round{ROUND_IDX}_student_gen",
            "meta": {
                "round_idx": ROUND_IDX,
                "age_group": age_group,
                "k_photos": len(one_lines),
                "length_tag": length_tag,
                "student_model": CURRENT_STUDENT_PATH,
                "seed": SEED,
                "coverage": cov,
                "novelty": nov,
                "repetition": rep,
                "hard_pass": bool(pass_hard_filter(diary)),
            },
            "input": {
                "one_lines": one_lines,
                "joined": normalize_joined(safe_join_one_lines(one_lines)),
                "one_lines_trace": one_lines_trace,
            },
            "output": {"daily_diary": diary},
        }
        records.append(rec)
        created += 1

        if created % 200 == 0:
            print(f"[STUDENT GEN] {created}/{target_n} attempts={attempts}")

    if created < target_n:
        print("[WARN] student generation ended early:", created, "created")
    return records

print("[ROUND5] generating student batch...")
student_recs = generate_student_batch(STUDENT_GEN_TARGET)
write_jsonl_shards(f"round{ROUND_IDX}_student_gen", student_recs)

# ============================================================
# (G) split: hard_pass keep vs gpt_targets(불합격 전량, 단 상한/하한)
# ============================================================
keep_pass = [r for r in student_recs if r["meta"].get("hard_pass", False)]
fail_pool = [r for r in student_recs if not r["meta"].get("hard_pass", False)]

# GPT 교정량 결정
min_k = int(len(student_recs) * GPT_REWRITE_MIN_RATIO)
max_k = int(len(student_recs) * GPT_REWRITE_MAX_RATIO)
k_gpt = min(max_k, max(min_k, len(fail_pool)))

# fail_pool이 너무 많으면 위험군 우선(반복/미종결/상투문구 우선)
def _fail_severity(r):
    t = (r.get("output", {}) or {}).get("daily_diary", "")
    sev = 0.0
    sev += 2.0 if has_unfinished_tail(t) else 0.0
    sev += 1.5 if count_cliches(t) >= 2 else 0.0
    sev += 1.2 if ngram_repeat_ratio(t, 4) >= 0.12 else 0.0
    sev += 1.0 if exact_duplicate_sentence_ratio(t) >= 0.20 else 0.0
    sev += float(r.get("meta", {}).get("novelty", 0.0))  # novelty 높으면 추가 사실 위험
    return sev

fail_pool_sorted = sorted(fail_pool, key=_fail_severity, reverse=True)
gpt_targets = fail_pool_sorted[:k_gpt]
drop_fail = fail_pool_sorted[k_gpt:]  # 학습에 안 넣음

print("[SPLIT]")
print(" keep_pass:", len(keep_pass))
print(" gpt_targets:", len(gpt_targets))
print(" drop_fail(not trained):", len(drop_fail))

# ============================================================
# (H) GPT 교정(전부 학습 포함)
# ============================================================
gpt_rewritten = []
for i, r in enumerate(gpt_targets, 1):
    one_lines = r["input"]["one_lines"]
    length_tag = r["meta"]["length_tag"]

    system, user, temp, max_tokens = build_gpt_rewrite_prompt(one_lines, length_tag)
    rew = normalize_space(call_teacher(system, user, temp, max_tokens))
    rew = postprocess_diary(rew)

    # 교정 실패/금지어면 student 원문 사용(그래도 postprocess는 적용)
    if (not rew) or contains_banned(rew):
        rew = postprocess_diary(r["output"]["daily_diary"])

    rr = dict(r)
    rr["stage"] = f"round{ROUND_IDX}_gpt_rewrite"
    rr["meta"] = dict(r["meta"])
    rr["meta"]["rewriter"] = "gpt_final"
    rr["meta"]["teacher_model"] = TEACHER_MODEL
    rr["meta"]["rewritten"] = True
    rr["meta"]["hard_pass_after"] = bool(pass_hard_filter(rew))
    rr["output"] = {"daily_diary": rew}
    gpt_rewritten.append(rr)

    if i % 100 == 0:
        print(f"[GPT REWRITE] {i}/{len(gpt_targets)}")

write_jsonl_shards(f"round{ROUND_IDX}_gpt_rewritten", gpt_rewritten)

# ============================================================
# (I) 저장용 merged_full(전체 보관)
# ============================================================
merged_full = keep_pass + gpt_rewritten + drop_fail
random.shuffle(merged_full)
write_jsonl_shards(f"merged_full_round{ROUND_IDX}", merged_full)
print("[DONE] merged_full saved:", f"merged_full_round{ROUND_IDX}_****.jsonl")

# ============================================================
# (J) 학습용 데이터 구성
# - keep_pass: 전량
# - gpt_rewritten: 전량
# - drop_fail: 학습 제외
# ============================================================
train_for_finetune = keep_pass + gpt_rewritten
random.shuffle(train_for_finetune)
write_jsonl_shards(f"train_round{ROUND_IDX}_for_finetune", train_for_finetune)

print("[TRAIN SPLIT]")
print(" keep_pass:", len(keep_pass))
print(" gpt:", len(gpt_rewritten))
print(" total:", len(train_for_finetune))

# ============================================================
# (K) anchors + train_for_finetune 로 다음 student 학습
# ============================================================
anchors_all = load_jsonl_glob(os.path.join(OUT_DIR, "anchors_*.jsonl"))
train_records = anchors_all + train_for_finetune
random.shuffle(train_records)

def to_hf_dataset(records: List[Dict[str, Any]]) -> Dataset:
    rows = [{"src": r["input"]["joined"], "tgt": r["output"]["daily_diary"]} for r in records]
    return Dataset.from_list(rows)

ds = to_hf_dataset(train_records)
split = ds.train_test_split(test_size=0.02, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

tok = AutoTokenizer.from_pretrained("gogamza/kobart-base-v2", use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(CURRENT_STUDENT_PATH)

MAX_SRC_LEN = 512
MAX_TGT_LEN = 256

def preprocess(batch):
    model_inputs = tok(batch["src"], max_length=MAX_SRC_LEN, truncation=True)
    labels = tok(text_target=batch["tgt"], max_length=MAX_TGT_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tok, model=model)

training_args = TrainingArguments(
    output_dir=f"{OUT_DIR}/runs/kobart_round{ROUND_IDX}",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=4e-5,
    num_train_epochs=2,
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=300,
    save_steps=300,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
)

print("[ROUND5] training next student...")
trainer.train()
trainer.save_model(NEXT_STUDENT_SAVE_DIR)
tok.save_pretrained(NEXT_STUDENT_SAVE_DIR)

extra = {
    "round_idx": ROUND_IDX,
    "student_gen_target": STUDENT_GEN_TARGET,
    "gpt_rewrite_min_ratio": GPT_REWRITE_MIN_RATIO,
    "gpt_rewrite_max_ratio": GPT_REWRITE_MAX_RATIO,
    "current_student_path": CURRENT_STUDENT_PATH,
    "next_student_path": NEXT_STUDENT_SAVE_DIR,
    "teacher_model": TEACHER_MODEL,
    "keep_pass_count": len(keep_pass),
    "gpt_rewrite_count": len(gpt_rewritten),
    "drop_fail_count": len(drop_fail),
}
save_run_config(extra=extra)

print("[DONE] next student saved:", NEXT_STUDENT_SAVE_DIR)
print("[NEXT] 다음 라운드 실행 방법")
print("       ROUND_IDX += 1")
print("       CURRENT_STUDENT_PATH = NEXT_STUDENT_SAVE_DIR")
print("[OUT_DIR]", OUT_DIR)

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


[ROUND5] generating student batch...
[STUDENT GEN] 200/10000 attempts=200
[STUDENT GEN] 400/10000 attempts=400
[STUDENT GEN] 600/10000 attempts=601
[STUDENT GEN] 800/10000 attempts=801
[STUDENT GEN] 1000/10000 attempts=1001
[STUDENT GEN] 1200/10000 attempts=1201
[STUDENT GEN] 1400/10000 attempts=1401
[STUDENT GEN] 1600/10000 attempts=1601
[STUDENT GEN] 1800/10000 attempts=1801
[STUDENT GEN] 2000/10000 attempts=2001
[STUDENT GEN] 2200/10000 attempts=2201
[STUDENT GEN] 2400/10000 attempts=2401
[STUDENT GEN] 2600/10000 attempts=2602
[STUDENT GEN] 2800/10000 attempts=2802
[STUDENT GEN] 3000/10000 attempts=3003
[STUDENT GEN] 3200/10000 attempts=3204
[STUDENT GEN] 3400/10000 attempts=3404
[STUDENT GEN] 3600/10000 attempts=3604
[STUDENT GEN] 3800/10000 attempts=3804
[STUDENT GEN] 4000/10000 attempts=4004
[STUDENT GEN] 4200/10000 attempts=4204
[STUDENT GEN] 4400/10000 attempts=4404
[STUDENT GEN] 4600/10000 attempts=4604
[STUDENT GEN] 4800/10000 attempts=4804
[STUDENT GEN] 5000/10000 attempts=5

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Map: 100%|██████████| 165/165 [00:00<00:00, 17675.34 examples/s]


[ROUND5] training next student...


/Users/jeongseong-gyeong/Documents/aiary/aiary/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss
300,1.233400,1.215441


RuntimeError: MPS backend out of memory (MPS allocated: 12.04 GiB, other allocations: 14.97 GiB, max allowed: 27.20 GiB). Tried to allocate 197.75 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [2]:
# ============================================================
# [CELL] Round 5 - 학습만 분리 (generation / GPT rewrite / split 완료 후 실행)
# 전제:
# - OUT_DIR 아래에 다음 파일들이 이미 존재해야 함
#   1) anchors_*.jsonl
#   2) train_round5_for_finetune_****.jsonl  (write_jsonl_shards로 저장된 shard들)
# - CURRENT_STUDENT_PATH 는 round4 student
# 결과:
# - NEXT_STUDENT_SAVE_DIR 에 round5 student 저장
# ============================================================

import os, json, glob, random
from typing import List, Dict, Any
import gc

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

# ----------------------------
# 설정
# ----------------------------
ROUND_IDX = 5
SEED = 777
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_DIR = "generated/aiary_v3_hybrid_20260220_014739"

CURRENT_STUDENT_PATH = f"{OUT_DIR}/models/kobart_student_round4"
NEXT_STUDENT_SAVE_DIR = f"{OUT_DIR}/models/kobart_student_round{ROUND_IDX}"
os.makedirs(NEXT_STUDENT_SAVE_DIR, exist_ok=True)

# 학습 데이터(shards)
ANCHORS_GLOB = os.path.join(OUT_DIR, "anchors_*.jsonl")
TRAIN_GLOB   = os.path.join(OUT_DIR, f"train_round{ROUND_IDX}_for_finetune_*.jsonl")

# ----------------------------
# 유틸: jsonl 로드
# ----------------------------
def load_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def load_jsonl_glob(pattern: str) -> List[Dict[str, Any]]:
    paths = sorted(glob.glob(pattern))
    if not paths:
        raise FileNotFoundError(f"No files matched: {pattern}")
    all_rows = []
    for p in paths:
        all_rows.extend(load_jsonl(p))
    return all_rows

# ----------------------------
# (1) 학습 레코드 로드: anchors + train_for_finetune
# ----------------------------
anchors_all = load_jsonl_glob(ANCHORS_GLOB)
train_for_finetune = load_jsonl_glob(TRAIN_GLOB)

train_records = anchors_all + train_for_finetune
random.shuffle(train_records)

print("[LOAD]")
print(" anchors:", len(anchors_all))
print(" train_for_finetune:", len(train_for_finetune))
print(" total:", len(train_records))

# ----------------------------
# (2) HF Dataset 변환
# ----------------------------
def to_hf_dataset(records: List[Dict[str, Any]]) -> Dataset:
    rows = []
    for r in records:
        inp = (r.get("input") or {})
        out = (r.get("output") or {})
        src = inp.get("joined", "")
        tgt = out.get("daily_diary", "")
        if not src or not tgt:
            continue
        rows.append({"src": src, "tgt": tgt})
    return Dataset.from_list(rows)

ds = to_hf_dataset(train_records)
if len(ds) == 0:
    raise RuntimeError("학습 데이터가 비어있음: src/tgt 누락 레코드가 많을 수 있음")

split = ds.train_test_split(test_size=0.02, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

print("[DATASET]")
print(" train:", len(train_ds))
print(" eval:", len(eval_ds))

# ----------------------------
# (3) 토크나이저/모델 (학습)
# - tok: gogamza/kobart-base-v2 (너 기존 유지)
# - model: round4 student에서 이어서 학습
# ----------------------------
tok = AutoTokenizer.from_pretrained("gogamza/kobart-base-v2", use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(CURRENT_STUDENT_PATH)

MAX_SRC_LEN = 512
MAX_TGT_LEN = 256

def preprocess(batch):
    model_inputs = tok(batch["src"], max_length=MAX_SRC_LEN, truncation=True)
    labels = tok(text_target=batch["tgt"], max_length=MAX_TGT_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tok, model=model)

# ----------------------------
# (4) Trainer 학습 + 저장
# ----------------------------
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
    
training_args = TrainingArguments(
    output_dir=f"{OUT_DIR}/runs/kobart_round{ROUND_IDX}",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=4e-5,
    num_train_epochs=2,
    warmup_ratio=0.05,
    weight_decay=0.01,

    logging_steps=50,

    eval_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    save_total_limit=2,

    fp16=False,
    bf16=False,

    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
)

print(f"[ROUND{ROUND_IDX}] training...")
trainer.train()

print("[SAVE] model/tokenizer ->", NEXT_STUDENT_SAVE_DIR)
trainer.save_model(NEXT_STUDENT_SAVE_DIR)
tok.save_pretrained(NEXT_STUDENT_SAVE_DIR)

print("[DONE] next student saved:", NEXT_STUDENT_SAVE_DIR)

[LOAD]
 anchors: 2000
 train_for_finetune: 6234
 total: 8234
[DATASET]
 train: 8069
 eval: 165


You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Map: 100%|██████████| 165/165 [00:00<00:00, 16119.91 examples/s]


[ROUND5] training...


Step,Training Loss,Validation Loss
1000,1.079100,1.118879


[SAVE] model/tokenizer -> generated/aiary_v3_hybrid_20260220_014739/models/kobart_student_round5
[DONE] next student saved: generated/aiary_v3_hybrid_20260220_014739/models/kobart_student_round5


In [4]:
# ============================================================
# Round4 vs Round5 비교용 평가 코드 (업데이트 버전)
# 변경점
# - 후처리 강화: "문장으로 끝나게" 보장
# - 중복 문장(정규화 기준) 발생 시: "중복이 처음 발생한 지점 이후" 전부 잘라냄
# - 문장 수 상한 컷(기본 11문장) 포함
# - 모델은 순차 로드(맥 MPS OOM 방지)
# ============================================================

import os, re, gc, json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ----------------------------
# 경로
# ----------------------------
OUT_DIR = "generated/aiary_v3_hybrid_20260220_014739"
R4_PATH = f"{OUT_DIR}/models/kobart_student_round4"
R5_PATH = f"{OUT_DIR}/models/kobart_student_round5"

# ----------------------------
# 디바이스
# ----------------------------
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
print("[DEVICE]", DEVICE)

def cleanup_torch():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

# ----------------------------
# 기본 유틸
# ----------------------------
def normalize_space(s: str) -> str:
    if not s:
        return ""
    s = s.replace("\u200b", " ")
    s = re.sub(r"[ \t]{2,}", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def normalize_joined(s: str) -> str:
    if not s:
        return ""
    s = s.strip()
    s = s.replace("에서에서", "에서")
    s = s.replace("있다하다가", "있다가")
    s = s.replace("해본다하더니", "해보더니")
    s = re.sub(r"\s*을\(를\)\s*", " ", s)
    s = re.sub(r"[ \t]{2,}", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def safe_join_one_lines(one_lines):
    return "\n".join([x.strip() for x in one_lines if x and x.strip()])

# ----------------------------
# 문장 분해/후처리 핵심
# ----------------------------
_SENT_END = r"[.!?。！？]"
_SENT_SPLIT_RE = re.compile(rf"(?<={_SENT_END})\s+|\n+")

def split_sentences(text: str):
    t = normalize_space(text or "")
    if not t:
        return []
    parts = [p.strip() for p in _SENT_SPLIT_RE.split(t) if p.strip()]
    return parts

def ensure_ends_with_sentence(text: str) -> str:
    if not text:
        return ""
    t = text.strip()
    # 이미 종결부호면 ok
    if re.search(rf"{_SENT_END}\s*$", t):
        return t
    # 마지막 종결부호 기준 컷
    last = max(t.rfind("."), t.rfind("!"), t.rfind("?"), t.rfind("。"), t.rfind("！"), t.rfind("？"))
    if last != -1 and last >= 20:
        t = t[: last + 1].strip()
        if re.search(rf"{_SENT_END}\s*$", t):
            return t
    # 그래도 없으면 길이 충분할 때 마침표 추가
    if len(t) >= 30:
        return (t + ".").strip()
    return t.strip()

def cut_to_max_sentences(text: str, max_sents: int = 11) -> str:
    sents = split_sentences(text)
    if len(sents) <= max_sents:
        return ensure_ends_with_sentence(" ".join(sents).strip())
    t = " ".join(sents[:max_sents]).strip()
    return ensure_ends_with_sentence(t)

def _norm_sent_for_dup(s: str) -> str:
    # 문장 중복 판정용 정규화: 공백/구두점/이모지 제거, 한글/영문/숫자만 남김
    s = re.sub(r"\s+", "", s)
    s = re.sub(r"[^0-9A-Za-z가-힣]", "", s)
    return s

def cut_after_duplicate_sentence(text: str, min_chars: int = 12) -> str:
    """
    중복 문장이 발생하면 "중복이 처음 발생한 시점"에서 컷.
    예) A B C B D -> A B C
    """
    sents = split_sentences(text)
    if len(sents) <= 1:
        return ensure_ends_with_sentence(" ".join(sents).strip())

    seen = set()
    kept = []
    for s in sents:
        key = _norm_sent_for_dup(s)
        # 너무 짧은 문장은 중복판정에서 제외(노이즈 방지)
        if len(key) >= min_chars:
            if key in seen:
                break
            seen.add(key)
        kept.append(s)

    t = " ".join(kept).strip()
    return ensure_ends_with_sentence(t)

def postprocess_diary(text: str, max_sents: int = 11) -> str:
    t = normalize_space(text or "")
    t = ensure_ends_with_sentence(t)
    t = cut_after_duplicate_sentence(t)
    t = cut_to_max_sentences(t, max_sents=max_sents)
    t = ensure_ends_with_sentence(t)
    return t.strip()

# ----------------------------
# 하드필터/지표(축약)
# ----------------------------
_CLICHE_PATTERNS = [
    r"오늘 하루.*소중",
    r"소중하게 다가왔",
    r"마음(이 )?놓였",
    r"조금 안도",
    r"세심하게 살펴야겠",
    r"생각(이 )?들었",
    r"평범한 하루",
    r"마음(이 )?따뜻",
    r"작은 변화",
    r"눈에 들어온 날",
    r"교훈을 남겼",
    r"앞으로는 .*해야겠",
]

def sentence_count_heuristic(text: str) -> int:
    return len(split_sentences(text))

def has_unfinished_tail(text: str) -> bool:
    if not text:
        return True
    return not bool(re.search(rf"{_SENT_END}\s*$", text.strip()) or re.search(r"(다|요|죠|음)\s*$", text.strip()))

def count_cliches(text: str) -> int:
    if not text:
        return 0
    c = 0
    for p in _CLICHE_PATTERNS:
        if re.search(p, text):
            c += 1
    return c

def exact_duplicate_sentence_ratio(text: str) -> float:
    sents = split_sentences(text)
    if len(sents) <= 1:
        return 0.0
    norm = [_norm_sent_for_dup(s) for s in sents]
    norm = [x for x in norm if x]
    if len(norm) <= 1:
        return 0.0
    uniq = len(set(norm))
    return float(1.0 - (uniq / max(1, len(norm))))

def ngram_repeat_ratio(text: str, n: int = 4) -> float:
    if not text:
        return 0.0
    toks = re.findall(r"[가-힣]{1,}|[0-9]+|[A-Za-z]+", text)
    if len(toks) < n * 2:
        return 0.0
    ngrams = [" ".join(toks[i:i+n]) for i in range(len(toks)-n+1)]
    total = len(ngrams)
    uniq = len(set(ngrams))
    return float(1.0 - (uniq / max(1, total)))

def has_weird_korean(text: str) -> bool:
    if not text:
        return True
    if re.search(r"[ㄱ-ㅎㅏ-ㅣ]{2,}", text):
        return True
    if re.search(r"(.)\1\1\1", text):
        return True
    return False

def pass_hard_filter(text: str) -> bool:
    if not text:
        return False
    if has_weird_korean(text):
        return False
    if has_unfinished_tail(text):
        return False
    if count_cliches(text) >= 2:
        return False
    if exact_duplicate_sentence_ratio(text) >= 0.20:
        return False
    if ngram_repeat_ratio(text, 4) >= 0.12:
        return False
    sc = sentence_count_heuristic(text)
    if sc <= 3 or sc >= 12:
        return False
    return True

_STOPWORDS = set(["오늘","아침","오전","점심","오후","저녁","밤","아이","아기","엄마","아빠","보호자"])
_CONJ = ["그리고", "그래서", "하지만", "그러다", "그러고", "또", "또는", "또한"]

def _ko_words(s: str):
    return re.findall(r"[가-힣]{2,}", s or "")

def _keywords_from_line(line: str, max_k: int = 8):
    words = [w for w in _ko_words(line) if w not in _STOPWORDS]
    out, seen = [], set()
    for w in words:
        if w in seen:
            continue
        seen.add(w)
        out.append(w)
        if len(out) >= max_k:
            break
    return out

def coverage_score(one_lines, diary: str) -> float:
    if not one_lines:
        return 0.0
    hit = 0
    for line in one_lines:
        kws = _keywords_from_line(line, max_k=8)
        if not kws:
            hit += 1 if any(tok in diary for tok in _ko_words(line)[:3]) else 0
            continue
        if any(k in diary for k in kws):
            hit += 1
    return hit / max(1, len(one_lines))

def novelty_score(one_lines, diary: str) -> float:
    in_words = set()
    for l in one_lines:
        in_words.update(_keywords_from_line(l, max_k=999))
    out_words = [w for w in _ko_words(diary) if w not in _STOPWORDS]
    if not out_words:
        return 0.0
    out_set = set(out_words)
    novel = [w for w in out_set if w not in in_words]
    return len(novel) / max(1, len(out_set))

def repetition_score(diary: str) -> float:
    if not diary:
        return 1.0
    conj_cnt = sum((diary or "").count(c) for c in _CONJ)
    sents = split_sentences(diary)
    starts = [s[:4] for s in sents if len(s) >= 4]

    start_rep = 0.0
    if starts:
        counts = {}
        for st in starts:
            counts[st] = counts.get(st, 0) + 1
        maxc = max(counts.values())
        start_rep = max(0, (maxc - 1) / max(1, len(starts) - 1))

    words = [w for w in _ko_words(diary) if w not in _STOPWORDS]
    uniq_ratio = (len(set(words)) / max(1, len(words))) if words else 1.0
    word_rep = 1.0 - uniq_ratio
    conj_norm = min(1.0, conj_cnt / 10.0)
    return float(min(1.0, 0.45 * word_rep + 0.35 * start_rep + 0.20 * conj_norm))

# ----------------------------
# 생성 함수
# ----------------------------
def gen_daily(model, tok, one_lines, length_tag="MEDIUM"):
    src = normalize_joined(safe_join_one_lines(one_lines))

    if length_tag == "SHORT":
        max_new, min_new = 120, 0
    elif length_tag == "MEDIUM":
        max_new, min_new = 180, 0
    else:
        max_new, min_new = 240, 0

    enc = tok(src, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    enc.pop("token_type_ids", None)

    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new,
            min_new_tokens=min_new,
            num_beams=2,
            do_sample=False,
            early_stopping=True,
            no_repeat_ngram_size=4,
            repetition_penalty=1.10,
            length_penalty=0.85,
            eos_token_id=tok.eos_token_id,
            pad_token_id=tok.pad_token_id,
        )

    raw = tok.decode(out[0], skip_special_tokens=True)
    return postprocess_diary(raw, max_sents=11)

def run_one_model(model_path: str, samples):
    print("\n[LOAD MODEL]", model_path)
    tok = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
    model.eval()

    outs = []
    for s in samples:
        one_lines = s["one_lines"]
        length_tag = s.get("length_tag", "MEDIUM")
        y = gen_daily(model, tok, one_lines, length_tag=length_tag)

        metrics = {
            "len_tag": length_tag,
            "sent_cnt": sentence_count_heuristic(y),
            "unfinished": has_unfinished_tail(y),
            "cliches": count_cliches(y),
            "dup_sent": round(exact_duplicate_sentence_ratio(y), 3),
            "ng4_rep": round(ngram_repeat_ratio(y, 4), 3),
            "cov": round(coverage_score(one_lines, y), 3),
            "nov": round(novelty_score(one_lines, y), 3),
            "rep": round(repetition_score(y), 3),
            "hard_pass": pass_hard_filter(y),
        }
        outs.append({"text": y, "metrics": metrics})

    del model
    del tok
    cleanup_torch()
    return outs

def pretty_print_pair(one_lines, r4, r5):
    print("\n" + "="*90)
    print("[INPUT one_lines]")
    for x in one_lines:
        print("-", x)
    print("\n[ROUND4 OUTPUT]")
    print(r4["text"])
    print("  metrics:", r4["metrics"])
    print("\n[ROUND5 OUTPUT]")
    print(r5["text"])
    print("  metrics:", r5["metrics"])

# ----------------------------
# 샘플
# ----------------------------
SAMPLES = [
    {
        "name": "sample1",
        "length_tag": "MEDIUM",
        "one_lines": [
            "따뜻한 차 안에서 과자 먹방 시작",
            "고기 냄새 가득한 따뜻한 가족 외식시간",
            "장난감 총과 친구, 오늘도 신나게 놀았어요",
            "따끈한 고기 냄새에 기대 가득한 밤",
            "불꽃 속에서 아빠 손 잡고 따뜻함을 느꼈어요",
        ],
    }
]

# ----------------------------
# 실행: R4 -> R5 (순차 로드)
# ----------------------------
cleanup_torch()
r4_outs = run_one_model(R4_PATH, SAMPLES)
r5_outs = run_one_model(R5_PATH, SAMPLES)

for i, s in enumerate(SAMPLES):
    pretty_print_pair(s["one_lines"], r4_outs[i], r5_outs[i])

print("\n[DONE] 비교 끝")

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


[DEVICE] mps

[LOAD MODEL] generated/aiary_v3_hybrid_20260220_014739/models/kobart_student_round4


You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.



[LOAD MODEL] generated/aiary_v3_hybrid_20260220_014739/models/kobart_student_round5

[INPUT one_lines]
- 따뜻한 차 안에서 과자 먹방 시작
- 고기 냄새 가득한 따뜻한 가족 외식시간
- 장난감 총과 친구, 오늘도 신나게 놀았어요
- 따끈한 고기 냄새에 기대 가득한 밤
- 불꽃 속에서 아빠 손 잡고 따뜻함을 느꼈어요

[ROUND4 OUTPUT]
오늘 아침, 아이가 따뜻한 차 안에서 과자 먹방을 하며 즐거운 시간을 보냈다. 아빠가 옆에서 손을 잡고 함께 있어 마음이 편안해졌다. 저녁에는 아이가 장난감 총과 친구와 함께 신나게 놀고 있었는데, 그 모습이 참 귀여워서 살짝 웃음이 났다. 오후에는 아빠가 아이 손을 잡고 함께 시간을 보냈는데, 아이가 아빠 손을 잡고 따뜻함을 느끼는 모습이 참 따뜻하게 느껴졌다. 오늘 하루 아이와 함께한 소소한 순간들이 참 소중하게 다가왔다. 아이가 다양한 감정을 표현하는 모습을 보며 조금 더 세심하게 살펴야겠다는 생각이 들었다. 오후에는 아이가 좋아하는 장난감들을 가지고 놀며 즐거운 시간을 보냈고, 그 모습을 보며 나도 덩달아 기분이 좋아졌다. 오늘 하루 아이의 다양한 감정과 행동을 지켜보며 조금 더 가까워진 느낌이 들었다. 아이가 좋아하는 장난감을 가지고 놀면서도 즐거워하는 모습을 보니 마음이 놓였다.
  metrics: {'len_tag': 'MEDIUM', 'sent_cnt': 9, 'unfinished': False, 'cliches': 5, 'dup_sent': 0.0, 'ng4_rep': 0.0, 'cov': 0.8, 'nov': 0.889, 'rep': 0.166, 'hard_pass': False}

[ROUND5 OUTPUT]
오늘 아침, 아이가 따뜻한 차 안에서 과자 먹방을 시작했다. 아이가 좋아하는 과자를 먹는 모습을 보니 마음이 편안해졌다. 오후에는 장난감 총과 친구와 함께 신나게 놀았는데, 아이가 신나게 뛰어노는 모습에 살짝 웃

In [5]:
# ============================================================
# Round4 vs Round5 비교용 평가 코드 (업데이트)
# 추가 기능
# - "4단어 이상"이 동일한 순서로 반복 등장하는 문장은 통째로 삭제
#   * 여기서 '단어' = 한글/영문/숫자 토큰(정규식 기반)
#   * 예: "A B C D ... A B C D" 형태(연속/비연속 모두)면 해당 문장 삭제
#   * 문장 내에서 4-그램 중복률이 높아도 삭제(보수적으로)
# - 기존 기능 유지:
#   * 문장 종결 보장
#   * 중복 문장 발생 시 그 지점 이후 컷
#   * 문장 수 상한 컷(기본 11)
# ============================================================

import os, re, gc
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ----------------------------
# 경로
# ----------------------------
OUT_DIR = "generated/aiary_v3_hybrid_20260220_014739"
R4_PATH = f"{OUT_DIR}/models/kobart_student_round4"
R5_PATH = f"{OUT_DIR}/models/kobart_student_round5"

# ----------------------------
# 디바이스
# ----------------------------
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
print("[DEVICE]", DEVICE)

def cleanup_torch():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

# ----------------------------
# 기본 유틸
# ----------------------------
def normalize_space(s: str) -> str:
    if not s:
        return ""
    s = s.replace("\u200b", " ")
    s = re.sub(r"[ \t]{2,}", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def normalize_joined(s: str) -> str:
    if not s:
        return ""
    s = s.strip()
    s = s.replace("에서에서", "에서")
    s = s.replace("있다하다가", "있다가")
    s = s.replace("해본다하더니", "해보더니")
    s = re.sub(r"\s*을\(를\)\s*", " ", s)
    s = re.sub(r"[ \t]{2,}", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def safe_join_one_lines(one_lines):
    return "\n".join([x.strip() for x in one_lines if x and x.strip()])

# ----------------------------
# 문장 분해/후처리 핵심
# ----------------------------
_SENT_END = r"[.!?。！？]"
_SENT_SPLIT_RE = re.compile(rf"(?<={_SENT_END})\s+|\n+")

def split_sentences(text: str):
    t = normalize_space(text or "")
    if not t:
        return []
    parts = [p.strip() for p in _SENT_SPLIT_RE.split(t) if p.strip()]
    return parts

def ensure_ends_with_sentence(text: str) -> str:
    if not text:
        return ""
    t = text.strip()
    if re.search(rf"{_SENT_END}\s*$", t):
        return t
    last = max(t.rfind("."), t.rfind("!"), t.rfind("?"), t.rfind("。"), t.rfind("！"), t.rfind("？"))
    if last != -1 and last >= 20:
        t = t[: last + 1].strip()
        if re.search(rf"{_SENT_END}\s*$", t):
            return t
    if len(t) >= 30:
        return (t + ".").strip()
    return t.strip()

def cut_to_max_sentences(text: str, max_sents: int = 11) -> str:
    sents = split_sentences(text)
    if len(sents) <= max_sents:
        return ensure_ends_with_sentence(" ".join(sents).strip())
    t = " ".join(sents[:max_sents]).strip()
    return ensure_ends_with_sentence(t)

def _norm_sent_for_dup(s: str) -> str:
    s = re.sub(r"\s+", "", s)
    s = re.sub(r"[^0-9A-Za-z가-힣]", "", s)
    return s

def cut_after_duplicate_sentence(text: str, min_chars: int = 12) -> str:
    sents = split_sentences(text)
    if len(sents) <= 1:
        return ensure_ends_with_sentence(" ".join(sents).strip())

    seen = set()
    kept = []
    for s in sents:
        key = _norm_sent_for_dup(s)
        if len(key) >= min_chars:
            if key in seen:
                break
            seen.add(key)
        kept.append(s)

    t = " ".join(kept).strip()
    return ensure_ends_with_sentence(t)

# ----------------------------
# NEW: 4단어 이상 동일 순서 반복 문장 삭제
# ----------------------------
_WORD_RE = re.compile(r"[가-힣]{1,}|[A-Za-z]{1,}|[0-9]+")

def _word_tokens(s: str):
    return _WORD_RE.findall(s or "")

def has_repeated_4gram(sentence: str, min_repeats: int = 2) -> bool:
    """
    문장 내 4-그램(연속 토큰 4개)이 동일 순서로 min_repeats 이상 등장하면 True
    - 연속/비연속 모두 잡힘(같은 4그램이 2번 이상 나오면)
    """
    toks = _word_tokens(sentence)
    if len(toks) < 8:
        return False
    grams = []
    for i in range(len(toks) - 4 + 1):
        grams.append(tuple(toks[i:i+4]))
    # 빈도 체크
    cnt = {}
    for g in grams:
        cnt[g] = cnt.get(g, 0) + 1
        if cnt[g] >= min_repeats:
            return True
    return False

def drop_sentences_with_repeated_4gram(text: str) -> str:
    sents = split_sentences(text)
    if not sents:
        return ""
    kept = []
    for s in sents:
        if has_repeated_4gram(s, min_repeats=2):
            continue  # ✅ 해당 문장 전체 삭제
        kept.append(s)
    return ensure_ends_with_sentence(" ".join(kept).strip())

def postprocess_diary(text: str, max_sents: int = 11) -> str:
    t = normalize_space(text or "")
    t = ensure_ends_with_sentence(t)

    # ✅ 4그램 반복 문장 삭제
    t = drop_sentences_with_repeated_4gram(t)

    # ✅ 중복 문장 이후 컷
    t = cut_after_duplicate_sentence(t)

    # ✅ 문장 수 상한
    t = cut_to_max_sentences(t, max_sents=max_sents)

    t = ensure_ends_with_sentence(t)
    return t.strip()

# ----------------------------
# 하드필터/지표(축약)
# ----------------------------
_CLICHE_PATTERNS = [
    r"오늘 하루.*소중",
    r"소중하게 다가왔",
    r"마음(이 )?놓였",
    r"조금 안도",
    r"세심하게 살펴야겠",
    r"생각(이 )?들었",
    r"평범한 하루",
    r"마음(이 )?따뜻",
    r"작은 변화",
    r"눈에 들어온 날",
    r"교훈을 남겼",
    r"앞으로는 .*해야겠",
]

def sentence_count_heuristic(text: str) -> int:
    return len(split_sentences(text))

def has_unfinished_tail(text: str) -> bool:
    if not text:
        return True
    return not bool(re.search(rf"{_SENT_END}\s*$", text.strip()) or re.search(r"(다|요|죠|음)\s*$", text.strip()))

def count_cliches(text: str) -> int:
    if not text:
        return 0
    c = 0
    for p in _CLICHE_PATTERNS:
        if re.search(p, text):
            c += 1
    return c

def exact_duplicate_sentence_ratio(text: str) -> float:
    sents = split_sentences(text)
    if len(sents) <= 1:
        return 0.0
    norm = [_norm_sent_for_dup(s) for s in sents]
    norm = [x for x in norm if x]
    if len(norm) <= 1:
        return 0.0
    uniq = len(set(norm))
    return float(1.0 - (uniq / max(1, len(norm))))

def ngram_repeat_ratio(text: str, n: int = 4) -> float:
    if not text:
        return 0.0
    toks = re.findall(r"[가-힣]{1,}|[0-9]+|[A-Za-z]+", text)
    if len(toks) < n * 2:
        return 0.0
    ngrams = [" ".join(toks[i:i+n]) for i in range(len(toks)-n+1)]
    total = len(ngrams)
    uniq = len(set(ngrams))
    return float(1.0 - (uniq / max(1, total)))

def has_weird_korean(text: str) -> bool:
    if not text:
        return True
    if re.search(r"[ㄱ-ㅎㅏ-ㅣ]{2,}", text):
        return True
    if re.search(r"(.)\1\1\1", text):
        return True
    return False

def pass_hard_filter(text: str) -> bool:
    if not text:
        return False
    if has_weird_korean(text):
        return False
    if has_unfinished_tail(text):
        return False
    if count_cliches(text) >= 2:
        return False
    if exact_duplicate_sentence_ratio(text) >= 0.20:
        return False
    if ngram_repeat_ratio(text, 4) >= 0.12:
        return False
    sc = sentence_count_heuristic(text)
    if sc <= 3 or sc >= 12:
        return False
    return True

_STOPWORDS = set(["오늘","아침","오전","점심","오후","저녁","밤","아이","아기","엄마","아빠","보호자"])
_CONJ = ["그리고", "그래서", "하지만", "그러다", "그러고", "또", "또는", "또한"]

def _ko_words(s: str):
    return re.findall(r"[가-힣]{2,}", s or "")

def _keywords_from_line(line: str, max_k: int = 8):
    words = [w for w in _ko_words(line) if w not in _STOPWORDS]
    out, seen = [], set()
    for w in words:
        if w in seen:
            continue
        seen.add(w)
        out.append(w)
        if len(out) >= max_k:
            break
    return out

def coverage_score(one_lines, diary: str) -> float:
    if not one_lines:
        return 0.0
    hit = 0
    for line in one_lines:
        kws = _keywords_from_line(line, max_k=8)
        if not kws:
            hit += 1 if any(tok in diary for tok in _ko_words(line)[:3]) else 0
            continue
        if any(k in diary for k in kws):
            hit += 1
    return hit / max(1, len(one_lines))

def novelty_score(one_lines, diary: str) -> float:
    in_words = set()
    for l in one_lines:
        in_words.update(_keywords_from_line(l, max_k=999))
    out_words = [w for w in _ko_words(diary) if w not in _STOPWORDS]
    if not out_words:
        return 0.0
    out_set = set(out_words)
    novel = [w for w in out_set if w not in in_words]
    return len(novel) / max(1, len(out_set))

def repetition_score(diary: str) -> float:
    if not diary:
        return 1.0
    conj_cnt = sum((diary or "").count(c) for c in _CONJ)
    sents = split_sentences(diary)
    starts = [s[:4] for s in sents if len(s) >= 4]

    start_rep = 0.0
    if starts:
        counts = {}
        for st in starts:
            counts[st] = counts.get(st, 0) + 1
        maxc = max(counts.values())
        start_rep = max(0, (maxc - 1) / max(1, len(starts) - 1))

    words = [w for w in _ko_words(diary) if w not in _STOPWORDS]
    uniq_ratio = (len(set(words)) / max(1, len(words))) if words else 1.0
    word_rep = 1.0 - uniq_ratio
    conj_norm = min(1.0, conj_cnt / 10.0)
    return float(min(1.0, 0.45 * word_rep + 0.35 * start_rep + 0.20 * conj_norm))

# ----------------------------
# 생성 함수
# ----------------------------
def gen_daily(model, tok, one_lines, length_tag="MEDIUM"):
    src = normalize_joined(safe_join_one_lines(one_lines))

    if length_tag == "SHORT":
        max_new, min_new = 120, 0
    elif length_tag == "MEDIUM":
        max_new, min_new = 180, 0
    else:
        max_new, min_new = 240, 0

    enc = tok(src, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    enc.pop("token_type_ids", None)

    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new,
            min_new_tokens=min_new,
            num_beams=2,
            do_sample=False,
            early_stopping=True,
            no_repeat_ngram_size=4,
            repetition_penalty=1.10,
            length_penalty=0.85,
            eos_token_id=tok.eos_token_id,
            pad_token_id=tok.pad_token_id,
        )

    raw = tok.decode(out[0], skip_special_tokens=True)
    return postprocess_diary(raw, max_sents=11)

def run_one_model(model_path: str, samples):
    print("\n[LOAD MODEL]", model_path)
    tok = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
    model.eval()

    outs = []
    for s in samples:
        one_lines = s["one_lines"]
        length_tag = s.get("length_tag", "MEDIUM")
        y = gen_daily(model, tok, one_lines, length_tag=length_tag)

        metrics = {
            "len_tag": length_tag,
            "sent_cnt": sentence_count_heuristic(y),
            "unfinished": has_unfinished_tail(y),
            "cliches": count_cliches(y),
            "dup_sent": round(exact_duplicate_sentence_ratio(y), 3),
            "ng4_rep": round(ngram_repeat_ratio(y, 4), 3),
            "cov": round(coverage_score(one_lines, y), 3),
            "nov": round(novelty_score(one_lines, y), 3),
            "rep": round(repetition_score(y), 3),
            "hard_pass": pass_hard_filter(y),
        }
        outs.append({"text": y, "metrics": metrics})

    del model
    del tok
    cleanup_torch()
    return outs

def pretty_print_pair(one_lines, r4, r5):
    print("\n" + "="*90)
    print("[INPUT one_lines]")
    for x in one_lines:
        print("-", x)
    print("\n[ROUND4 OUTPUT]")
    print(r4["text"])
    print("  metrics:", r4["metrics"])
    print("\n[ROUND5 OUTPUT]")
    print(r5["text"])
    print("  metrics:", r5["metrics"])

# ----------------------------
# 샘플
# ----------------------------
SAMPLES = [
    {
        "name": "sample1",
        "length_tag": "MEDIUM",
        "one_lines": [
            "따뜻한 차 안에서 과자 먹방 시작",
            "고기 냄새 가득한 따뜻한 가족 외식시간",
            "장난감 총과 친구, 오늘도 신나게 놀았어요",
            "따끈한 고기 냄새에 기대 가득한 밤",
            "불꽃 속에서 아빠 손 잡고 따뜻함을 느꼈어요",
        ],
    }
]

# ----------------------------
# 실행: R4 -> R5
# ----------------------------
cleanup_torch()
r4_outs = run_one_model(R4_PATH, SAMPLES)
r5_outs = run_one_model(R5_PATH, SAMPLES)

for i, s in enumerate(SAMPLES):
    pretty_print_pair(s["one_lines"], r4_outs[i], r5_outs[i])

print("\n[DONE] 비교 끝")

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


[DEVICE] mps

[LOAD MODEL] generated/aiary_v3_hybrid_20260220_014739/models/kobart_student_round4


You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.



[LOAD MODEL] generated/aiary_v3_hybrid_20260220_014739/models/kobart_student_round5

[INPUT one_lines]
- 따뜻한 차 안에서 과자 먹방 시작
- 고기 냄새 가득한 따뜻한 가족 외식시간
- 장난감 총과 친구, 오늘도 신나게 놀았어요
- 따끈한 고기 냄새에 기대 가득한 밤
- 불꽃 속에서 아빠 손 잡고 따뜻함을 느꼈어요

[ROUND4 OUTPUT]
오늘 아침, 아이가 따뜻한 차 안에서 과자 먹방을 하며 즐거운 시간을 보냈다. 아빠가 옆에서 손을 잡고 함께 있어 마음이 편안해졌다. 저녁에는 아이가 장난감 총과 친구와 함께 신나게 놀고 있었는데, 그 모습이 참 귀여워서 살짝 웃음이 났다. 오후에는 아빠가 아이 손을 잡고 함께 시간을 보냈는데, 아이가 아빠 손을 잡고 따뜻함을 느끼는 모습이 참 따뜻하게 느껴졌다. 오늘 하루 아이와 함께한 소소한 순간들이 참 소중하게 다가왔다. 아이가 다양한 감정을 표현하는 모습을 보며 조금 더 세심하게 살펴야겠다는 생각이 들었다. 오후에는 아이가 좋아하는 장난감들을 가지고 놀며 즐거운 시간을 보냈고, 그 모습을 보며 나도 덩달아 기분이 좋아졌다. 오늘 하루 아이의 다양한 감정과 행동을 지켜보며 조금 더 가까워진 느낌이 들었다. 아이가 좋아하는 장난감을 가지고 놀면서도 즐거워하는 모습을 보니 마음이 놓였다.
  metrics: {'len_tag': 'MEDIUM', 'sent_cnt': 9, 'unfinished': False, 'cliches': 5, 'dup_sent': 0.0, 'ng4_rep': 0.0, 'cov': 0.8, 'nov': 0.889, 'rep': 0.166, 'hard_pass': False}

[ROUND5 OUTPUT]
오늘 아침, 아이가 따뜻한 차 안에서 과자 먹방을 시작했다. 아이가 좋아하는 과자를 먹는 모습을 보니 마음이 편안해졌다. 오후에는 장난감 총과 친구와 함께 신나게 놀았는데, 아이가 신나게 뛰어노는 모습에 살짝 웃

In [6]:
# ============================================================
# Mom-tone one_lines(5쌍)로 Round4 vs Round5 비교 코드
# - 사용자 제공(엄마 톤) one_lines 5쌍을 고정 입력으로 사용
# - Round4 / Round5 모델을 순차 로드하여(MPS OOM 방지) 각 입력에 대해 생성 비교
# - 후처리 포함:
#   * 문장 종결 보장
#   * 중복 문장 발생 시 그 지점 이후 컷
#   * 문장 수 상한 컷(기본 11)
#   * "4단어 이상" 동일 순서 반복(4-그램 2회 이상) 문장은 통째로 삭제
# ============================================================

import os, re, gc
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ----------------------------
# 경로
# ----------------------------
OUT_DIR = "generated/aiary_v3_hybrid_20260220_014739"
R4_PATH = f"{OUT_DIR}/models/kobart_student_round4"
R5_PATH = f"{OUT_DIR}/models/kobart_student_round5"

# ----------------------------
# 디바이스
# ----------------------------
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
print("[DEVICE]", DEVICE)

def cleanup_torch():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

# ----------------------------
# 기본 유틸
# ----------------------------
def normalize_space(s: str) -> str:
    if not s:
        return ""
    s = s.replace("\u200b", " ")
    s = re.sub(r"[ \t]{2,}", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def normalize_joined(s: str) -> str:
    if not s:
        return ""
    s = s.strip()
    s = s.replace("에서에서", "에서")
    s = s.replace("있다하다가", "있다가")
    s = s.replace("해본다하더니", "해보더니")
    s = re.sub(r"\s*을\(를\)\s*", " ", s)
    s = re.sub(r"[ \t]{2,}", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def safe_join_one_lines(one_lines):
    return "\n".join([x.strip() for x in one_lines if x and x.strip()])

# ----------------------------
# 문장 분해/후처리 핵심
# ----------------------------
_SENT_END = r"[.!?。！？]"
_SENT_SPLIT_RE = re.compile(rf"(?<={_SENT_END})\s+|\n+")

def split_sentences(text: str):
    t = normalize_space(text or "")
    if not t:
        return []
    parts = [p.strip() for p in _SENT_SPLIT_RE.split(t) if p.strip()]
    return parts

def ensure_ends_with_sentence(text: str) -> str:
    if not text:
        return ""
    t = text.strip()
    if re.search(rf"{_SENT_END}\s*$", t):
        return t
    last = max(t.rfind("."), t.rfind("!"), t.rfind("?"), t.rfind("。"), t.rfind("！"), t.rfind("？"))
    if last != -1 and last >= 20:
        t = t[: last + 1].strip()
        if re.search(rf"{_SENT_END}\s*$", t):
            return t
    if len(t) >= 30:
        return (t + ".").strip()
    return t.strip()

def cut_to_max_sentences(text: str, max_sents: int = 11) -> str:
    sents = split_sentences(text)
    if len(sents) <= max_sents:
        return ensure_ends_with_sentence(" ".join(sents).strip())
    t = " ".join(sents[:max_sents]).strip()
    return ensure_ends_with_sentence(t)

def _norm_sent_for_dup(s: str) -> str:
    s = re.sub(r"\s+", "", s)
    s = re.sub(r"[^0-9A-Za-z가-힣]", "", s)
    return s

def cut_after_duplicate_sentence(text: str, min_chars: int = 12) -> str:
    sents = split_sentences(text)
    if len(sents) <= 1:
        return ensure_ends_with_sentence(" ".join(sents).strip())

    seen = set()
    kept = []
    for s in sents:
        key = _norm_sent_for_dup(s)
        if len(key) >= min_chars:
            if key in seen:
                break
            seen.add(key)
        kept.append(s)

    t = " ".join(kept).strip()
    return ensure_ends_with_sentence(t)

# ----------------------------
# 4단어 이상 동일 순서 반복 문장 삭제
# ----------------------------
_WORD_RE = re.compile(r"[가-힣]{1,}|[A-Za-z]{1,}|[0-9]+")

def _word_tokens(s: str):
    return _WORD_RE.findall(s or "")

def has_repeated_4gram(sentence: str, min_repeats: int = 2) -> bool:
    toks = _word_tokens(sentence)
    if len(toks) < 8:
        return False
    cnt = {}
    for i in range(len(toks) - 4 + 1):
        g = tuple(toks[i:i+4])
        cnt[g] = cnt.get(g, 0) + 1
        if cnt[g] >= min_repeats:
            return True
    return False

def drop_sentences_with_repeated_4gram(text: str) -> str:
    sents = split_sentences(text)
    if not sents:
        return ""
    kept = []
    for s in sents:
        if has_repeated_4gram(s, min_repeats=2):
            continue
        kept.append(s)
    return ensure_ends_with_sentence(" ".join(kept).strip())

def postprocess_diary(text: str, max_sents: int = 11) -> str:
    t = normalize_space(text or "")
    t = ensure_ends_with_sentence(t)
    t = drop_sentences_with_repeated_4gram(t)
    t = cut_after_duplicate_sentence(t)
    t = cut_to_max_sentences(t, max_sents=max_sents)
    t = ensure_ends_with_sentence(t)
    return t.strip()

# ----------------------------
# 평가 지표(간단)
# ----------------------------
_CLICHE_PATTERNS = [
    r"오늘 하루.*소중",
    r"소중하게 다가왔",
    r"마음(이 )?놓였",
    r"조금 안도",
    r"세심하게 살펴야겠",
    r"생각(이 )?들었",
    r"평범한 하루",
    r"마음(이 )?따뜻",
    r"작은 변화",
    r"눈에 들어온 날",
    r"교훈을 남겼",
    r"앞으로는 .*해야겠",
]

def sentence_count_heuristic(text: str) -> int:
    return len(split_sentences(text))

def has_unfinished_tail(text: str) -> bool:
    if not text:
        return True
    return not bool(re.search(rf"{_SENT_END}\s*$", text.strip()) or re.search(r"(다|요|죠|음)\s*$", text.strip()))

def count_cliches(text: str) -> int:
    if not text:
        return 0
    c = 0
    for p in _CLICHE_PATTERNS:
        if re.search(p, text):
            c += 1
    return c

def ngram_repeat_ratio(text: str, n: int = 4) -> float:
    if not text:
        return 0.0
    toks = re.findall(r"[가-힣]{1,}|[0-9]+|[A-Za-z]+", text)
    if len(toks) < n * 2:
        return 0.0
    ngrams = [" ".join(toks[i:i+n]) for i in range(len(toks)-n+1)]
    total = len(ngrams)
    uniq = len(set(ngrams))
    return float(1.0 - (uniq / max(1, total)))

def exact_duplicate_sentence_ratio(text: str) -> float:
    sents = split_sentences(text)
    if len(sents) <= 1:
        return 0.0
    norm = [_norm_sent_for_dup(s) for s in sents]
    norm = [x for x in norm if x]
    if len(norm) <= 1:
        return 0.0
    uniq = len(set(norm))
    return float(1.0 - (uniq / max(1, len(norm))))

# ----------------------------
# 입력: 엄마 톤 one_lines 5쌍 (고정)
# ----------------------------
SAMPLES = [
    {
        "name": "pair1",
        "length_tag": "MEDIUM",
        "one_lines": [
            "차 안에서 카시트에 앉아 과자 하나 꼭 쥐고 먹는 모습이 귀여워서 한참 바라봤다",
            "유모차에 앉아 모자 푹 눌러쓰고 바깥 구경하느라 눈이 반짝반짝했다",
            "미끄럼틀 앞에서 두 손 뻗고 올라가 보겠다고 낑낑대는 모습이 사랑스러웠다",
            "식탁에 앉아 작은 숟가락으로 밥 떠먹겠다고 집중하는 모습이 기특했다",
            "욕조 거품 속에서 장난감 가지고 까르르 웃는 소리에 나까지 기분이 좋아졌다",
        ],
    },
    {
        "name": "pair2",
        "length_tag": "MEDIUM",
        "one_lines": [
            "거실 바닥에 블록을 잔뜩 늘어놓고 탑 쌓겠다며 집중하는 모습이 귀여웠다",
            "소파에 앉아 그림책 페이지를 한 장씩 넘기며 조용히 보는 시간이 참 평화로웠다",
            "주방에서 과일 한 조각 집어 들고 맛있게 먹는 얼굴이 너무 사랑스러웠다",
            "공원에서 비눗방울 따라 뛰어다니며 웃는 모습에 나도 덩달아 웃음이 났다",
            "침대 위에서 이불 끌어안고 스르르 잠들 준비하는 모습이 포근했다",
        ],
    },
    {
        "name": "pair3",
        "length_tag": "MEDIUM",
        "one_lines": [
            "어린이집 앞에서 가방 메고 손 흔드는 모습이 대견하면서도 조금 짠했다",
            "교실에서 색연필 쥐고 종이에 끄적이며 집중하는 모습이 기특했다",
            "점심시간에 식판 앞에서 국을 들여다보며 조심스럽게 먹는 모습이 귀여웠다",
            "놀이터 모래밭에서 삽으로 모래 퍼 담으며 신나게 노는 시간이 즐거워 보였다",
            "집에 와서 세면대 앞에서 손 씻는 모습이 제법 의젓해 보였다",
        ],
    },
    {
        "name": "pair4",
        "length_tag": "MEDIUM",
        "one_lines": [
            "마트 카트에 앉아 과자 가리키며 고르겠다고 눈을 반짝이던 순간이 귀여웠다",
            "장난감 코너에서 인형을 꼭 안고 한참 바라보는 모습이 사랑스러웠다",
            "집에 와서 식탁 위 과일을 한입 베어 물고 환하게 웃어 보였다",
            "거실 바닥에서 장난감 자동차 밀며 혼자서도 신나게 노는 모습이 기특했다",
            "잠옷 입고 조명 아래에서 하이파이브하며 하루 마무리하는 시간이 따뜻했다",
        ],
    },
    {
        "name": "pair5",
        "length_tag": "MEDIUM",
        "one_lines": [
            "키즈카페 트램펄린 위에서 점프하려고 무릎 굽히는 모습이 너무 신나 보였다",
            "볼풀장 안에서 공을 두 손으로 꼭 쥐고 까르르 웃는 모습이 사랑스러웠다",
            "작은 미끄럼틀 내려오며 두 팔 벌리고 웃는 모습에 나도 같이 웃었다",
            "음료 컵을 양손으로 잡고 조심스럽게 한 모금 마시는 모습이 귀여웠다",
            "차 안에서 피곤한 얼굴로 꾸벅꾸벅 졸기 시작해 오늘 하루가 길었구나 싶었다",
        ],
    },
]

# ----------------------------
# 생성 함수
# ----------------------------
def gen_daily(model, tok, one_lines, length_tag="MEDIUM"):
    src = normalize_joined(safe_join_one_lines(one_lines))

    if length_tag == "SHORT":
        max_new, min_new = 120, 0
    elif length_tag == "MEDIUM":
        max_new, min_new = 180, 0
    else:
        max_new, min_new = 240, 0

    enc = tok(src, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    enc.pop("token_type_ids", None)

    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new,
            min_new_tokens=min_new,
            num_beams=2,
            do_sample=False,
            early_stopping=True,
            no_repeat_ngram_size=4,
            repetition_penalty=1.10,
            length_penalty=0.85,
            eos_token_id=tok.eos_token_id,
            pad_token_id=tok.pad_token_id,
        )

    raw = tok.decode(out[0], skip_special_tokens=True)
    return postprocess_diary(raw, max_sents=11)

def run_one_model(model_path: str, samples):
    print("\n[LOAD MODEL]", model_path)
    tok = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
    model.eval()

    outs = []
    for s in samples:
        one_lines = s["one_lines"]
        length_tag = s.get("length_tag", "MEDIUM")
        y = gen_daily(model, tok, one_lines, length_tag=length_tag)

        metrics = {
            "len_tag": length_tag,
            "sent_cnt": sentence_count_heuristic(y),
            "unfinished": has_unfinished_tail(y),
            "cliches": count_cliches(y),
            "dup_sent": round(exact_duplicate_sentence_ratio(y), 3),
            "ng4_rep": round(ngram_repeat_ratio(y, 4), 3),
        }
        outs.append({"text": y, "metrics": metrics})

    del model
    del tok
    cleanup_torch()
    return outs

def pretty_print_pair(idx, name, one_lines, r4, r5):
    print("\n" + "="*90)
    print(f"[{name} / PAIR {idx}]")
    print("[INPUT one_lines]")
    for x in one_lines:
        print("-", x)

    print("\n[ROUND4 OUTPUT]")
    print(r4["text"])
    print("  metrics:", r4["metrics"])

    print("\n[ROUND5 OUTPUT]")
    print(r5["text"])
    print("  metrics:", r5["metrics"])

# ----------------------------
# 실행: R4 -> R5 (순차 로드)
# ----------------------------
cleanup_torch()
r4_outs = run_one_model(R4_PATH, SAMPLES)
r5_outs = run_one_model(R5_PATH, SAMPLES)

for i, s in enumerate(SAMPLES, start=1):
    pretty_print_pair(i, s["name"], s["one_lines"], r4_outs[i-1], r5_outs[i-1])

print("\n[DONE] 엄마 톤 one_lines 5쌍 비교 끝")

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


[DEVICE] mps

[LOAD MODEL] generated/aiary_v3_hybrid_20260220_014739/models/kobart_student_round4


You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.



[LOAD MODEL] generated/aiary_v3_hybrid_20260220_014739/models/kobart_student_round5

[pair1 / PAIR 1]
[INPUT one_lines]
- 차 안에서 카시트에 앉아 과자 하나 꼭 쥐고 먹는 모습이 귀여워서 한참 바라봤다
- 유모차에 앉아 모자 푹 눌러쓰고 바깥 구경하느라 눈이 반짝반짝했다
- 미끄럼틀 앞에서 두 손 뻗고 올라가 보겠다고 낑낑대는 모습이 사랑스러웠다
- 식탁에 앉아 작은 숟가락으로 밥 떠먹겠다고 집중하는 모습이 기특했다
- 욕조 거품 속에서 장난감 가지고 까르르 웃는 소리에 나까지 기분이 좋아졌다

[ROUND4 OUTPUT]
오늘 아침 안에서 아이가 카시트에 앉아 과자 하나 꼭 쥐고 먹는 모습을 한참 바라보는 모습이 귀여웠다. 유모차에서는 아이가 모자를 푹 눌러쓰고 바깥 구경하는 동안 눈이 반짝반짝 반짝이는 모습이 인상적이었다. 미끄럼틀 앞에서는 두 손 뻗고 올라가 보겠다고 낑낑거리는 모습이 귀여웠고, 식탁에 앉아 작은 숟가락으로 밥을 떠먹겠다고 집중하는 모습이 인상적이었어요. 욕조 거품 속에서 장난감 가지고 까르르 웃는 소리에 나까지 기분이 좋아져서 마음이 놓였답니다. 오늘 하루 아이와 함께한 소소한 순간들이 참 소중하게 다가왔습니다. 유모차와 유모차에 앉아 모자 푹 눌러쓰고 밖 구경하는 동안에도 눈이 반짝 반짝 빛나는 모습이 인상적이었고, 미끄럼힐 앞에서는 두 손을 뻗고 올라가보겠다고 낑끙거리는 모습이 귀여워서 살짝 웃음이 났습니다.
  metrics: {'len_tag': 'MEDIUM', 'sent_cnt': 6, 'unfinished': False, 'cliches': 3, 'dup_sent': 0.0, 'ng4_rep': 0.0}

[ROUND5 OUTPUT]
오늘 아침 안에서 아이가 카시트에 앉아 과자 하나 꼭 쥐고 먹는 모습을 한참 동안 바라보았다. 유모차에서는 아이가 모자를 푹 눌러쓰고 바깥 구경하는 동안 눈이 반짝이는 모습이 인상적이었다. 미끄럼틀